# 02 — Corrected SUES-200 Location-Disjoint Benchmark

**Objective:** Build and validate a location-disjoint SUES-200 evaluation protocol and report results by nominal height.

**Project:** GPS-Denied UAV Visual Positioning System

**Provenance rule:** Never describe local reimplementation results as official MobileGeo results.


In [ ]:
from pathlib import Path
import json
import hashlib
import random
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/mobilegeo_project")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Seed:", SEED)


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

DATASET_ROOT = Path(
    "/content/reused_sues200_dataset/SUES-200-512x512"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Manifest root:", MANIFEST_ROOT)

print("\nDataset exists:", DATASET_ROOT.exists())

if DATASET_ROOT.exists():
    print("\nDataset folders:")

    for path in sorted(DATASET_ROOT.iterdir()):
        print("-", path.name)

print("\n✅ Phase 2 setup ready")

Mounted at /content/drive
Project root: /content/drive/MyDrive/mobilegeo_project
Dataset root: /content/reused_sues200_dataset/SUES-200-512x512
Output root: /content/drive/MyDrive/mobilegeo_project/results/sues200_corrected
Manifest root: /content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected

Dataset exists: False

✅ Phase 2 setup ready


In [4]:
from pathlib import Path
import subprocess

MY_DRIVE = Path("/content/drive/MyDrive")

common_paths = [
    MY_DRIVE / "SUES-200-512x512",
    MY_DRIVE / "SUES-200.zip",
    MY_DRIVE / "SUES-200-512x512.zip",
    MY_DRIVE / "mobilegeo_final" / "SUES-200-512x512.zip",
    MY_DRIVE / "mobilegeo_final" / "data" / "SUES-200-512x512",
    MY_DRIVE / "mobilegeo_project" / "data" / "SUES-200-512x512",
]

print("Checking common locations...\n")

found = []

for path in common_paths:
    if path.exists():
        found.append(path)
        print("✅ Found:", path)

if not found:
    print("Checking nearby Drive folders only...\n")

    result = subprocess.run(
        [
            "find",
            "/content/drive/MyDrive",
            "-maxdepth",
            "4",
            "(",
            "-iname",
            "*sues*.zip",
            "-o",
            "-iname",
            "SUES-200-512x512",
            ")",
        ],
        capture_output=True,
        text=True,
    )

    matches = [
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip()
    ]

    if matches:
        for match in matches:
            print("✅ Found:", match)
    else:
        print("❌ SUES-200 ZIP or folder was not found in Google Drive.")
        print("Upload the SUES-200 ZIP into My Drive.")

Checking common locations...

Checking nearby Drive folders only...

✅ Found: /content/drive/MyDrive/SUES-200-512x512-V2.zip
✅ Found: /content/drive/MyDrive/Copy of SUES-200-512x512-V2.zip
✅ Found: /content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512
✅ Found: /content/drive/MyDrive/drone/Copy of SUES-200-512x512-V2.zip
✅ Found: /content/drive/MyDrive/drone/Copy of SUES-200-512x512-V2/SUES-200-512x512


In [8]:
# ============================================================
# REPAIR INCOMPLETE SUES-200 DATASET
# ============================================================

from pathlib import Path
import zipfile
import shutil
import os

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

# Existing extracted dataset locations.
candidate_roots = [
    Path(
        "/content/drive/MyDrive/"
        "SUES-200-512x512_extracted/"
        "SUES-200-512x512"
    ),
    Path(
        "/content/drive/MyDrive/drone/"
        "Copy of SUES-200-512x512-V2/"
        "SUES-200-512x512"
    ),
]

# Available ZIP files.
zip_candidates = [
    Path(
        "/content/drive/MyDrive/"
        "SUES-200-512x512-V2.zip"
    ),
    Path(
        "/content/drive/MyDrive/"
        "Copy of SUES-200-512x512-V2.zip"
    ),
    Path(
        "/content/drive/MyDrive/drone/"
        "Copy of SUES-200-512x512-V2.zip"
    ),
]

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


def find_view_folder(root, keyword):
    """Find a folder containing the requested view name."""

    if not root.is_dir():
        return None

    # Check the dataset root first.
    direct_candidates = [
        path
        for path in root.iterdir()
        if path.is_dir()
        and keyword in path.name.lower()
    ]

    if direct_candidates:
        return direct_candidates[0]

    # Search only a few folder levels.
    root_depth = len(root.parts)

    for current_root, directories, files in os.walk(root):

        current_path = Path(current_root)
        depth = len(current_path.parts) - root_depth

        if depth > 3:
            directories[:] = []
            continue

        for directory in directories:
            if keyword in directory.lower():
                return current_path / directory

    return None


# ------------------------------------------------------------
# 1. Check both extracted copies
# ------------------------------------------------------------

DATASET_ROOT = None
DRONE_ROOT = None
SATELLITE_ROOT = None

print("Checking extracted dataset copies...\n")

for root in candidate_roots:

    print("Dataset candidate:")
    print(root)
    print("Exists:", root.is_dir())

    if not root.is_dir():
        print()
        continue

    drone_folder = find_view_folder(
        root,
        "drone",
    )

    satellite_folder = find_view_folder(
        root,
        "satellite",
    )

    print("Drone folder:", drone_folder)
    print("Satellite folder:", satellite_folder)
    print()

    if (
        drone_folder is not None
        and satellite_folder is not None
    ):
        DATASET_ROOT = root
        DRONE_ROOT = drone_folder
        SATELLITE_ROOT = satellite_folder
        break

# ------------------------------------------------------------
# 2. Restore satellite folder from ZIP when missing
# ------------------------------------------------------------

if SATELLITE_ROOT is None:

    print(
        "No complete extracted copy found.\n"
        "Restoring the satellite folder from ZIP..."
    )

    DATASET_ROOT = candidate_roots[0]
    DATASET_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    DRONE_ROOT = find_view_folder(
        DATASET_ROOT,
        "drone",
    )

    selected_zip = next(
        (
            path
            for path in zip_candidates
            if path.is_file()
        ),
        None,
    )

    if selected_zip is None:
        raise FileNotFoundError(
            "No SUES-200 ZIP file was found."
        )

    print("\nUsing ZIP:")
    print(selected_zip)

    temporary_extract_root = Path(
        "/content/sues200_satellite_restore"
    )

    if temporary_extract_root.exists():
        shutil.rmtree(
            temporary_extract_root
        )

    temporary_extract_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        selected_zip,
        "r",
    ) as archive:

        all_members = archive.namelist()

        satellite_members = [
            member
            for member in all_members
            if any(
                "satellite" in part.lower()
                for part in Path(member).parts
            )
        ]

        print(
            "Satellite ZIP entries found:",
            len(satellite_members),
        )

        if not satellite_members:
            raise FileNotFoundError(
                "The ZIP does not contain a folder "
                "with 'satellite' in its name."
            )

        for member in satellite_members:
            archive.extract(
                member,
                temporary_extract_root,
            )

    # Find the restored satellite directory.
    restored_candidates = []

    for path in temporary_extract_root.rglob("*"):

        if (
            path.is_dir()
            and "satellite" in path.name.lower()
        ):
            image_count = sum(
                1
                for file_path in path.rglob("*")
                if file_path.is_file()
                and file_path.suffix.lower()
                in image_extensions
            )

            if image_count > 0:
                restored_candidates.append(
                    (
                        image_count,
                        path,
                    )
                )

    if not restored_candidates:
        raise FileNotFoundError(
            "Satellite images could not be found "
            "after ZIP extraction."
        )

    restored_candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    satellite_image_count, restored_folder = (
        restored_candidates[0]
    )

    destination_folder = (
        DATASET_ROOT
        / restored_folder.name
    )

    print("\nRestored satellite source:")
    print(restored_folder)

    print("\nCopying satellite folder to:")
    print(destination_folder)

    shutil.copytree(
        restored_folder,
        destination_folder,
        dirs_exist_ok=True,
    )

    SATELLITE_ROOT = destination_folder

# ------------------------------------------------------------
# 3. Validate image counts
# ------------------------------------------------------------

if DRONE_ROOT is None:
    DRONE_ROOT = find_view_folder(
        DATASET_ROOT,
        "drone",
    )

if SATELLITE_ROOT is None:
    SATELLITE_ROOT = find_view_folder(
        DATASET_ROOT,
        "satellite",
    )

if DRONE_ROOT is None:
    raise FileNotFoundError(
        "Drone folder is still missing."
    )

if SATELLITE_ROOT is None:
    raise FileNotFoundError(
        "Satellite folder is still missing."
    )

drone_image_count = sum(
    1
    for path in DRONE_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
)

satellite_image_count = sum(
    1
    for path in SATELLITE_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
)

# ------------------------------------------------------------
# 4. Save correct dataset paths
# ------------------------------------------------------------

config_file = (
    PROJECT_ROOT
    / "configs"
    / "sues200_resolved_paths.txt"
)

config_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)

config_file.write_text(
    f"DATASET_ROOT={DATASET_ROOT}\n"
    f"DRONE_ROOT={DRONE_ROOT}\n"
    f"SATELLITE_ROOT={SATELLITE_ROOT}\n",
    encoding="utf-8",
)

print("\n" + "=" * 65)
print("✅ SUES-200 DATASET REPAIR COMPLETE")
print("=" * 65)

print("\nDataset root:")
print(DATASET_ROOT)

print("\nDrone root:")
print(DRONE_ROOT)

print("\nDrone images:")
print(drone_image_count)

print("\nSatellite root:")
print(SATELLITE_ROOT)

print("\nSatellite images:")
print(satellite_image_count)

print("\nSaved configuration:")
print(config_file)

Checking extracted dataset copies...

Dataset candidate:
/content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512
Exists: True
Drone folder: /content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512/drone_view_512
Satellite folder: None

Dataset candidate:
/content/drive/MyDrive/drone/Copy of SUES-200-512x512-V2/SUES-200-512x512
Exists: True
Drone folder: /content/drive/MyDrive/drone/Copy of SUES-200-512x512-V2/SUES-200-512x512/drone_view_512
Satellite folder: None

No complete extracted copy found.
Restoring the satellite folder from ZIP...

Using ZIP:
/content/drive/MyDrive/SUES-200-512x512-V2.zip
Satellite ZIP entries found: 401

Restored satellite source:
/content/sues200_satellite_restore/SUES-200-512x512/satellite-view

Copying satellite folder to:
/content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512/satellite-view

✅ SUES-200 DATASET REPAIR COMPLETE

Dataset root:
/content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512

Drone root:
/co

In [9]:
# ============================================================
# PHASE 2 — CELL 2
# Inspect SUES-200 folder and label structure
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CONFIG_FILE = (
    PROJECT_ROOT
    / "configs"
    / "sues200_resolved_paths.txt"
)

if not CONFIG_FILE.is_file():
    raise FileNotFoundError(CONFIG_FILE)

# Read saved paths.
resolved_paths = {}

for line in CONFIG_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if "=" in line:
        key, value = line.split("=", 1)
        resolved_paths[key.strip()] = Path(
            value.strip()
        )

DATASET_ROOT = resolved_paths["DATASET_ROOT"]
DRONE_ROOT = resolved_paths["DRONE_ROOT"]
SATELLITE_ROOT = resolved_paths["SATELLITE_ROOT"]

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# ------------------------------------------------------------
# Scan image paths
# ------------------------------------------------------------

def scan_images(root, view_name):

    records = []

    for image_path in sorted(
        root.rglob("*")
    ):
        if (
            not image_path.is_file()
            or image_path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        relative_path = image_path.relative_to(
            root
        )

        parts = relative_path.parts

        records.append({
            "view": view_name,
            "absolute_path": str(image_path),
            "relative_path": str(relative_path),
            "filename": image_path.name,
            "path_depth": len(parts),
            "level_1": (
                parts[0]
                if len(parts) >= 2
                else ""
            ),
            "level_2": (
                parts[1]
                if len(parts) >= 3
                else ""
            ),
            "level_3": (
                parts[2]
                if len(parts) >= 4
                else ""
            ),
            "parent_folder": image_path.parent.name,
        })

    return records


drone_records = scan_images(
    DRONE_ROOT,
    "drone",
)

satellite_records = scan_images(
    SATELLITE_ROOT,
    "satellite",
)

all_records = (
    drone_records
    + satellite_records
)

inventory_df = pd.DataFrame(
    all_records
)

inventory_csv = (
    OUTPUT_ROOT
    / "dataset_structure_inventory.csv"
)

inventory_df.to_csv(
    inventory_csv,
    index=False,
)

# ------------------------------------------------------------
# Print structure summary
# ------------------------------------------------------------

print("=" * 70)
print("SUES-200 DATASET STRUCTURE AUDIT")
print("=" * 70)

print("\nDataset root:")
print(DATASET_ROOT)

print("\nImage counts:")
print("Drone:", len(drone_records))
print("Satellite:", len(satellite_records))

print("\nDrone path-depth counts:")
print(
    inventory_df[
        inventory_df["view"] == "drone"
    ]["path_depth"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSatellite path-depth counts:")
print(
    inventory_df[
        inventory_df["view"] == "satellite"
    ]["path_depth"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nDrone level-1 folders:")
drone_level_1 = (
    inventory_df[
        inventory_df["view"] == "drone"
    ]["level_1"]
    .value_counts()
)

print(
    drone_level_1.head(30).to_string()
)

print("\nSatellite level-1 folders:")
satellite_level_1 = (
    inventory_df[
        inventory_df["view"] == "satellite"
    ]["level_1"]
    .value_counts()
)

print(
    satellite_level_1.head(30).to_string()
)

print("\nFirst 20 drone relative paths:")

for path in (
    inventory_df[
        inventory_df["view"] == "drone"
    ]["relative_path"]
    .head(20)
):
    print("-", path)

print("\nFirst 20 satellite relative paths:")

for path in (
    inventory_df[
        inventory_df["view"] == "satellite"
    ]["relative_path"]
    .head(20)
):
    print("-", path)

print("\nUnique drone parent folders:")
print(
    inventory_df[
        inventory_df["view"] == "drone"
    ]["parent_folder"].nunique()
)

print("\nUnique satellite parent folders:")
print(
    inventory_df[
        inventory_df["view"] == "satellite"
    ]["parent_folder"].nunique()
)

print("\n✅ Structure inventory saved:")
print(inventory_csv)

SUES-200 DATASET STRUCTURE AUDIT

Dataset root:
/content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512

Image counts:
Drone: 36773
Satellite: 200

Drone path-depth counts:
path_depth
3    36773

Satellite path-depth counts:
path_depth
2    200

Drone level-1 folders:
level_1
0001    200
0002    200
0003    200
0004    200
0005    200
0006    200
0007    200
0008    200
0009    200
0010    200
0011    200
0012    200
0013    200
0014    200
0015    200
0016    200
0017    200
0018    200
0019    200
0020    200
0021    200
0022    200
0023    200
0024    200
0025    200
0026    200
0027    200
0028    200
0029    200
0030    200

Satellite level-1 folders:
level_1
0001    1
0002    1
0003    1
0004    1
0005    1
0006    1
0007    1
0008    1
0009    1
0010    1
0011    1
0012    1
0013    1
0014    1
0015    1
0016    1
0017    1
0018    1
0019    1
0020    1
0021    1
0022    1
0023    1
0024    1
0025    1
0026    1
0027    1
0028    1
0029    1
0030    1

First 20 drone 

In [11]:
# ============================================================
# REPAIR MISSING OR INCOMPLETE DRONE LOCATIONS
# ============================================================

from pathlib import Path
import zipfile
import shutil
import tempfile

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CONFIG_FILE = (
    PROJECT_ROOT
    / "configs"
    / "sues200_resolved_paths.txt"
)

# Read saved dataset paths.
resolved_paths = {}

for line in CONFIG_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if "=" in line:
        key, value = line.split("=", 1)
        resolved_paths[key.strip()] = Path(
            value.strip()
        )

DATASET_ROOT = resolved_paths["DATASET_ROOT"]
DRONE_ROOT = resolved_paths["DRONE_ROOT"]
SATELLITE_ROOT = resolved_paths["SATELLITE_ROOT"]

ZIP_PATH = Path(
    "/content/drive/MyDrive/SUES-200-512x512-V2.zip"
)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

if not ZIP_PATH.is_file():
    raise FileNotFoundError(ZIP_PATH)

# ------------------------------------------------------------
# 1. Expected locations from satellite folders
# ------------------------------------------------------------

expected_locations = sorted(
    path.name
    for path in SATELLITE_ROOT.iterdir()
    if path.is_dir()
)

print("Expected locations:", len(expected_locations))

# ------------------------------------------------------------
# 2. Find missing or incomplete drone locations
# ------------------------------------------------------------

location_counts_before = {}
locations_to_repair = []

for location_id in expected_locations:

    location_folder = (
        DRONE_ROOT
        / location_id
    )

    image_count = sum(
        1
        for path in location_folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ) if location_folder.is_dir() else 0

    location_counts_before[
        location_id
    ] = image_count

    # SUES-200 should contain 200 drone images per location.
    if image_count != 200:
        locations_to_repair.append(
            location_id
        )

print("\nLocations requiring repair:")
print(len(locations_to_repair))

for location_id in locations_to_repair:
    print(
        location_id,
        "current images:",
        location_counts_before[
            location_id
        ],
    )

if not locations_to_repair:
    print("\n✅ All drone locations are already complete.")

else:

    # --------------------------------------------------------
    # 3. Find the ZIP path prefix for drone_view_512
    # --------------------------------------------------------

    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as archive:

        zip_members = archive.namelist()

        matching_members = []

        for member in zip_members:

            member_parts = Path(member).parts
            lower_parts = [
                part.lower()
                for part in member_parts
            ]

            if "drone_view_512" not in lower_parts:
                continue

            drone_index = lower_parts.index(
                "drone_view_512"
            )

            if len(member_parts) <= drone_index + 1:
                continue

            location_id = member_parts[
                drone_index + 1
            ]

            if location_id in locations_to_repair:
                matching_members.append(
                    member
                )

        print(
            "\nZIP entries selected:",
            len(matching_members),
        )

        if not matching_members:
            raise RuntimeError(
                "No matching drone files were found in the ZIP."
            )

        # ----------------------------------------------------
        # 4. Extract selected locations temporarily
        # ----------------------------------------------------

        temporary_root = Path(
            tempfile.mkdtemp(
                prefix="sues200_drone_repair_",
                dir="/content",
            )
        )

        print("\nTemporary extraction folder:")
        print(temporary_root)

        for member in matching_members:
            archive.extract(
                member,
                temporary_root,
            )

    # --------------------------------------------------------
    # 5. Locate and copy each repaired location
    # --------------------------------------------------------

    restored_drone_roots = [
        path
        for path in temporary_root.rglob(
            "drone_view_512"
        )
        if path.is_dir()
    ]

    if not restored_drone_roots:
        raise RuntimeError(
            "Extracted drone_view_512 folder was not found."
        )

    restored_drone_root = restored_drone_roots[0]

    print("\nRestored drone root:")
    print(restored_drone_root)

    for location_id in locations_to_repair:

        source_folder = (
            restored_drone_root
            / location_id
        )

        destination_folder = (
            DRONE_ROOT
            / location_id
        )

        if not source_folder.is_dir():
            print(
                "⚠️ Missing in ZIP:",
                location_id,
            )
            continue

        # Replace incomplete folder completely.
        if destination_folder.exists():
            shutil.rmtree(
                destination_folder
            )

        shutil.copytree(
            source_folder,
            destination_folder,
        )

        print(
            "✅ Restored:",
            location_id,
        )

    shutil.rmtree(
        temporary_root,
        ignore_errors=True,
    )

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

invalid_locations = []
total_drone_images = 0

print("\n" + "=" * 65)
print("FINAL DRONE DATASET VALIDATION")
print("=" * 65)

for location_id in expected_locations:

    location_folder = (
        DRONE_ROOT
        / location_id
    )

    image_count = sum(
        1
        for path in location_folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ) if location_folder.is_dir() else 0

    total_drone_images += image_count

    if image_count != 200:
        invalid_locations.append(
            {
                "location_id": location_id,
                "image_count": image_count,
            }
        )

print("\nTotal locations:")
print(len(expected_locations))

print("\nTotal drone images:")
print(total_drone_images)

print("\nIncomplete locations:")
print(len(invalid_locations))

if invalid_locations:
    for item in invalid_locations:
        print(
            item["location_id"],
            item["image_count"],
        )

    raise RuntimeError(
        "Some drone locations are still incomplete."
    )

print("\n✅ All 200 locations contain 200 drone images")
print("✅ Expected total drone images: 40,000")

Expected locations: 200

Locations requiring repair:
17
0184 current images: 173
0185 current images: 0
0186 current images: 0
0187 current images: 0
0188 current images: 0
0189 current images: 0
0190 current images: 0
0191 current images: 0
0192 current images: 0
0193 current images: 0
0194 current images: 0
0195 current images: 0
0196 current images: 0
0197 current images: 0
0198 current images: 0
0199 current images: 0
0200 current images: 0

ZIP entries selected: 3485

Temporary extraction folder:
/content/sues200_drone_repair_cax6rfkj

Restored drone root:
/content/sues200_drone_repair_cax6rfkj/SUES-200-512x512/drone_view_512
✅ Restored: 0184
✅ Restored: 0185
✅ Restored: 0186
✅ Restored: 0187
✅ Restored: 0188
✅ Restored: 0189
✅ Restored: 0190
✅ Restored: 0191
✅ Restored: 0192
✅ Restored: 0193
✅ Restored: 0194
✅ Restored: 0195
✅ Restored: 0196
✅ Restored: 0197
✅ Restored: 0198
✅ Restored: 0199
✅ Restored: 0200

FINAL DRONE DATASET VALIDATION

Total locations:
200

Total drone image

In [12]:
# ============================================================
# PHASE 2 — CELL 3
# Build location-disjoint SUES-200 manifests
# ============================================================

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import random
import json

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CONFIG_FILE = (
    PROJECT_ROOT
    / "configs"
    / "sues200_resolved_paths.txt"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Read resolved dataset paths.
resolved_paths = {}

for line in CONFIG_FILE.read_text(
    encoding="utf-8"
).splitlines():

    if "=" in line:
        key, value = line.split("=", 1)

        resolved_paths[key.strip()] = Path(
            value.strip()
        )

DATASET_ROOT = resolved_paths["DATASET_ROOT"]
DRONE_ROOT = resolved_paths["DRONE_ROOT"]
SATELLITE_ROOT = resolved_paths["SATELLITE_ROOT"]

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# ------------------------------------------------------------
# 2. Detect locations
# ------------------------------------------------------------

drone_locations = sorted(
    path.name
    for path in DRONE_ROOT.iterdir()
    if path.is_dir()
)

satellite_locations = sorted(
    path.name
    for path in SATELLITE_ROOT.iterdir()
    if path.is_dir()
)

drone_location_set = set(
    drone_locations
)

satellite_location_set = set(
    satellite_locations
)

missing_satellite_locations = sorted(
    drone_location_set
    - satellite_location_set
)

missing_drone_locations = sorted(
    satellite_location_set
    - drone_location_set
)

if missing_satellite_locations:
    raise RuntimeError(
        "Locations missing satellite images:\n"
        + str(missing_satellite_locations)
    )

if missing_drone_locations:
    raise RuntimeError(
        "Locations missing drone images:\n"
        + str(missing_drone_locations)
    )

location_ids = sorted(
    drone_location_set
    & satellite_location_set
)

print("Matched location IDs:", len(location_ids))

if len(location_ids) != 200:
    print(
        "Warning: expected 200 locations, but found",
        len(location_ids),
    )

# ------------------------------------------------------------
# 3. Deterministic location-disjoint split
# ------------------------------------------------------------

SEED = 42

shuffled_locations = location_ids.copy()

random.Random(SEED).shuffle(
    shuffled_locations
)

location_count = len(
    shuffled_locations
)

if location_count == 200:
    train_count = 120
    validation_count = 40
    test_count = 40
else:
    train_count = int(
        round(location_count * 0.60)
    )

    validation_count = int(
        round(location_count * 0.20)
    )

    test_count = (
        location_count
        - train_count
        - validation_count
    )

train_locations = set(
    shuffled_locations[
        :train_count
    ]
)

validation_locations = set(
    shuffled_locations[
        train_count:
        train_count + validation_count
    ]
)

test_locations = set(
    shuffled_locations[
        train_count + validation_count:
    ]
)

# Validate zero overlap.
train_validation_overlap = (
    train_locations
    & validation_locations
)

train_test_overlap = (
    train_locations
    & test_locations
)

validation_test_overlap = (
    validation_locations
    & test_locations
)

if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):
    raise RuntimeError(
        "Location leakage detected."
    )


def get_split(location_id):

    if location_id in train_locations:
        return "train"

    if location_id in validation_locations:
        return "validation"

    if location_id in test_locations:
        return "test"

    raise KeyError(
        f"Location has no split: {location_id}"
    )

# ------------------------------------------------------------
# 4. Location split manifest
# ------------------------------------------------------------

location_records = []

for location_id in location_ids:

    location_records.append({
        "location_id": location_id,
        "split": get_split(
            location_id
        ),
        "split_seed": SEED,
        "split_protocol": (
            "PROJECT_DEFINED_"
            "LOCATION_DISJOINT_60_20_20"
        ),
    })

location_split_df = pd.DataFrame(
    location_records
).sort_values(
    [
        "split",
        "location_id",
    ]
)

location_split_csv = (
    MANIFEST_ROOT
    / "sues200_location_split.csv"
)

location_split_df.to_csv(
    location_split_csv,
    index=False,
)

# ------------------------------------------------------------
# 5. Satellite manifest
# ------------------------------------------------------------

satellite_records = []

satellite_path_by_location = {}

for location_id in location_ids:

    location_folder = (
        SATELLITE_ROOT
        / location_id
    )

    satellite_images = sorted(
        path
        for path in location_folder.rglob("*")
        if path.is_file()
        and path.suffix.lower()
        in IMAGE_EXTENSIONS
    )

    if len(satellite_images) != 1:
        raise RuntimeError(
            f"Expected exactly one satellite image for "
            f"location {location_id}, found "
            f"{len(satellite_images)}"
        )

    image_path = satellite_images[0]

    tile_id = (
        f"sat_{location_id}"
    )

    satellite_path_by_location[
        location_id
    ] = image_path

    satellite_records.append({
        "map_id": "SUES200",
        "tile_id": tile_id,
        "location_id": location_id,
        "class_id": location_id,
        "split": get_split(
            location_id
        ),
        "source_view": "satellite",
        "absolute_path": str(
            image_path
        ),
        "relative_path": str(
            image_path.relative_to(
                DATASET_ROOT
            )
        ),
        "filename": image_path.name,
        "positive_location_id": (
            location_id
        ),
        "crs": "UNKNOWN",
        "latitude": np.nan,
        "longitude": np.nan,
        "geographic_polygon": "",
        "gsd_m_per_pixel": np.nan,
        "source_date": "",
        "georeferenced": False,
        "split_protocol": (
            "PROJECT_DEFINED_"
            "LOCATION_DISJOINT_60_20_20"
        ),
    })

satellite_manifest_df = pd.DataFrame(
    satellite_records
).sort_values(
    [
        "split",
        "location_id",
    ]
)

satellite_manifest_csv = (
    MANIFEST_ROOT
    / "sues200_satellite_manifest.csv"
)

satellite_manifest_df.to_csv(
    satellite_manifest_csv,
    index=False,
)

# ------------------------------------------------------------
# 6. Drone manifest
# ------------------------------------------------------------

drone_records = []

for location_id in location_ids:

    location_folder = (
        DRONE_ROOT
        / location_id
    )

    for altitude_folder in sorted(
        location_folder.iterdir()
    ):

        if not altitude_folder.is_dir():
            continue

        try:
            nominal_height = int(
                altitude_folder.name
            )
        except ValueError:
            print(
                "Skipping unexpected folder:",
                altitude_folder,
            )
            continue

        drone_images = sorted(
            (
                path
                for path in altitude_folder.rglob("*")
                if path.is_file()
                and path.suffix.lower()
                in IMAGE_EXTENSIONS
            ),
            key=lambda path: (
                int(path.stem)
                if path.stem.isdigit()
                else path.stem
            ),
        )

        for sequence_index, image_path in enumerate(
            drone_images
        ):

            frame_id = (
                f"drone_{location_id}_"
                f"{nominal_height}_"
                f"{sequence_index:04d}"
            )

            positive_tile_id = (
                f"sat_{location_id}"
            )

            drone_records.append({
                "frame_id": frame_id,
                "location_id": location_id,
                "class_id": location_id,
                "split": get_split(
                    location_id
                ),
                "source_view": "drone",
                "nominal_height": (
                    nominal_height
                ),
                "altitude_reference": (
                    "NOMINAL_DATASET_FOLDER_LABEL_"
                    "NOT_VERIFIED_AS_AGL_OR_MSL"
                ),
                "sequence_index": (
                    sequence_index
                ),
                "absolute_path": str(
                    image_path
                ),
                "relative_path": str(
                    image_path.relative_to(
                        DATASET_ROOT
                    )
                ),
                "filename": image_path.name,
                "positive_tile_id": (
                    positive_tile_id
                ),
                "positive_location_id": (
                    location_id
                ),
                "georeferenced": False,
                "split_protocol": (
                    "PROJECT_DEFINED_"
                    "LOCATION_DISJOINT_60_20_20"
                ),
            })

drone_manifest_df = pd.DataFrame(
    drone_records
).sort_values(
    [
        "split",
        "location_id",
        "nominal_height",
        "sequence_index",
    ]
)

drone_manifest_csv = (
    MANIFEST_ROOT
    / "sues200_drone_manifest.csv"
)

drone_manifest_df.to_csv(
    drone_manifest_csv,
    index=False,
)

# ------------------------------------------------------------
# 7. Validate positive satellite mapping
# ------------------------------------------------------------

valid_tile_ids = set(
    satellite_manifest_df[
        "tile_id"
    ]
)

missing_positive_tiles = sorted(
    set(
        drone_manifest_df[
            "positive_tile_id"
        ]
    )
    - valid_tile_ids
)

if missing_positive_tiles:
    raise RuntimeError(
        "Drone images reference missing "
        "satellite tiles:\n"
        + str(
            missing_positive_tiles
        )
    )

# ------------------------------------------------------------
# 8. Count by split and nominal height
# ------------------------------------------------------------

drone_split_height_summary = (
    drone_manifest_df
    .groupby(
        [
            "split",
            "nominal_height",
        ]
    )
    .agg(
        image_count=(
            "frame_id",
            "count",
        ),
        location_count=(
            "location_id",
            "nunique",
        ),
    )
    .reset_index()
)

drone_split_height_csv = (
    OUTPUT_ROOT
    / "drone_split_height_summary.csv"
)

drone_split_height_summary.to_csv(
    drone_split_height_csv,
    index=False,
)

satellite_split_summary = (
    satellite_manifest_df
    .groupby("split")
    .agg(
        satellite_count=(
            "tile_id",
            "count",
        ),
        location_count=(
            "location_id",
            "nunique",
        ),
    )
    .reset_index()
)

satellite_split_summary_csv = (
    OUTPUT_ROOT
    / "satellite_split_summary.csv"
)

satellite_split_summary.to_csv(
    satellite_split_summary_csv,
    index=False,
)

# ------------------------------------------------------------
# 9. Save split audit
# ------------------------------------------------------------

split_audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "dataset": "SUES-200-512x512",

    "protocol_name": (
        "PROJECT_DEFINED_"
        "LOCATION_DISJOINT_60_20_20"
    ),

    "official_sues200_protocol_claimed": False,

    "split_seed": SEED,

    "total_locations": len(
        location_ids
    ),

    "train_locations": len(
        train_locations
    ),

    "validation_locations": len(
        validation_locations
    ),

    "test_locations": len(
        test_locations
    ),

    "total_drone_images": len(
        drone_manifest_df
    ),

    "total_satellite_images": len(
        satellite_manifest_df
    ),

    "train_validation_overlap": len(
        train_validation_overlap
    ),

    "train_test_overlap": len(
        train_test_overlap
    ),

    "validation_test_overlap": len(
        validation_test_overlap
    ),

    "missing_satellite_locations": (
        missing_satellite_locations
    ),

    "missing_drone_locations": (
        missing_drone_locations
    ),

    "missing_positive_tiles": (
        missing_positive_tiles
    ),

    "altitude_reference": (
        "NOMINAL_DATASET_FOLDER_LABEL_"
        "NOT_VERIFIED_AS_AGL_OR_MSL"
    ),

    "location_disjoint_validation_passed": True,
}

split_audit_json = (
    OUTPUT_ROOT
    / "sues200_location_disjoint_audit.json"
)

split_audit_json.write_text(
    json.dumps(
        split_audit,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 10. Final output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("✅ SUES-200 LOCATION-DISJOINT MANIFESTS CREATED")
print("=" * 72)

print("\nLocation split:")
print(
    location_split_df[
        "split"
    ].value_counts().to_string()
)

print("\nDrone image counts by split and height:")
print(
    drone_split_height_summary.to_string(
        index=False
    )
)

print("\nSatellite counts by split:")
print(
    satellite_split_summary.to_string(
        index=False
    )
)

print("\nLocation overlap validation:")
print(
    "Train ↔ Validation:",
    len(train_validation_overlap),
)
print(
    "Train ↔ Test:",
    len(train_test_overlap),
)
print(
    "Validation ↔ Test:",
    len(validation_test_overlap),
)

print("\nMissing positive satellite tiles:")
print(len(missing_positive_tiles))

print("\nDrone manifest:")
print(drone_manifest_csv)

print("\nSatellite manifest:")
print(satellite_manifest_csv)

print("\nLocation split manifest:")
print(location_split_csv)

print("\nAudit JSON:")
print(split_audit_json)

Matched location IDs: 200

✅ SUES-200 LOCATION-DISJOINT MANIFESTS CREATED

Location split:
split
train         120
test           40
validation     40

Drone image counts by split and height:
     split  nominal_height  image_count  location_count
      test             150         2000              40
      test             200         2000              40
      test             250         2000              40
      test             300         2000              40
     train             150         6000             120
     train             200         6000             120
     train             250         6000             120
     train             300         6000             120
validation             150         2000              40
validation             200         2000              40
validation             250         2000              40
validation             300         2000              40

Satellite counts by split:
     split  satellite_count  location_count
      te

In [13]:
# ============================================================
# PHASE 2 — CELL 4
# Prepare test query and satellite gallery manifests
# ============================================================

from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

DRONE_MANIFEST = (
    MANIFEST_ROOT
    / "sues200_drone_manifest.csv"
)

SATELLITE_MANIFEST = (
    MANIFEST_ROOT
    / "sues200_satellite_manifest.csv"
)

# ------------------------------------------------------------
# 1. Load manifests
# ------------------------------------------------------------

drone_df = pd.read_csv(
    DRONE_MANIFEST,
    dtype={
        "location_id": str,
        "class_id": str,
    },
)

satellite_df = pd.read_csv(
    SATELLITE_MANIFEST,
    dtype={
        "location_id": str,
        "class_id": str,
    },
)

# Preserve four-digit location IDs.
drone_df["location_id"] = (
    drone_df["location_id"]
    .str.zfill(4)
)

satellite_df["location_id"] = (
    satellite_df["location_id"]
    .str.zfill(4)
)

# ------------------------------------------------------------
# 2. Select test-only data
# ------------------------------------------------------------

test_queries_df = (
    drone_df[
        drone_df["split"] == "test"
    ]
    .copy()
    .sort_values(
        [
            "nominal_height",
            "location_id",
            "sequence_index",
        ]
    )
    .reset_index(drop=True)
)

test_gallery_df = (
    satellite_df[
        satellite_df["split"] == "test"
    ]
    .copy()
    .sort_values(
        "location_id"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Validate test protocol
# ------------------------------------------------------------

query_locations = set(
    test_queries_df["location_id"]
)

gallery_locations = set(
    test_gallery_df["location_id"]
)

missing_gallery_locations = sorted(
    query_locations
    - gallery_locations
)

extra_gallery_locations = sorted(
    gallery_locations
    - query_locations
)

duplicate_gallery_locations = (
    test_gallery_df[
        test_gallery_df.duplicated(
            "location_id",
            keep=False,
        )
    ]["location_id"]
    .unique()
    .tolist()
)

if missing_gallery_locations:
    raise RuntimeError(
        "Test queries have no gallery image for: "
        + str(missing_gallery_locations)
    )

if extra_gallery_locations:
    raise RuntimeError(
        "Gallery contains unexpected test locations: "
        + str(extra_gallery_locations)
    )

if duplicate_gallery_locations:
    raise RuntimeError(
        "Duplicate gallery locations found: "
        + str(duplicate_gallery_locations)
    )

# Every positive tile must exist.
gallery_tile_ids = set(
    test_gallery_df["tile_id"]
)

missing_positive_tiles = sorted(
    set(
        test_queries_df["positive_tile_id"]
    )
    - gallery_tile_ids
)

if missing_positive_tiles:
    raise RuntimeError(
        "Missing positive gallery tiles: "
        + str(missing_positive_tiles)
    )

# ------------------------------------------------------------
# 4. Save evaluation manifests
# ------------------------------------------------------------

test_query_csv = (
    MANIFEST_ROOT
    / "sues200_test_queries.csv"
)

test_gallery_csv = (
    MANIFEST_ROOT
    / "sues200_test_gallery.csv"
)

test_queries_df.to_csv(
    test_query_csv,
    index=False,
)

test_gallery_df.to_csv(
    test_gallery_csv,
    index=False,
)

# Create one query manifest per nominal height.
height_manifest_paths = {}

for height in [150, 200, 250, 300]:

    height_df = (
        test_queries_df[
            test_queries_df[
                "nominal_height"
            ] == height
        ]
        .copy()
        .reset_index(drop=True)
    )

    height_csv = (
        MANIFEST_ROOT
        / f"sues200_test_queries_{height}.csv"
    )

    height_df.to_csv(
        height_csv,
        index=False,
    )

    height_manifest_paths[str(height)] = str(
        height_csv
    )

# ------------------------------------------------------------
# 5. Save protocol metadata
# ------------------------------------------------------------

protocol = {
    "dataset": "SUES-200-512x512",
    "protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),
    "evaluation_split": "test",
    "query_view": "drone",
    "gallery_view": "satellite",
    "test_location_count": int(
        test_queries_df[
            "location_id"
        ].nunique()
    ),
    "test_query_count": int(
        len(test_queries_df)
    ),
    "test_gallery_count": int(
        len(test_gallery_df)
    ),
    "queries_per_height": {
        str(height): int(
            (
                test_queries_df[
                    "nominal_height"
                ] == height
            ).sum()
        )
        for height in [
            150,
            200,
            250,
            300,
        ]
    },
    "location_overlap_with_train": 0,
    "location_overlap_with_validation": 0,
    "altitude_reference": (
        "NOMINAL_DATASET_FOLDER_LABEL_"
        "NOT_VERIFIED_AS_AGL_OR_MSL"
    ),
    "query_manifest": str(
        test_query_csv
    ),
    "gallery_manifest": str(
        test_gallery_csv
    ),
    "height_manifests": (
        height_manifest_paths
    ),
}

protocol_json = (
    OUTPUT_ROOT
    / "sues200_test_protocol.json"
)

protocol_json.write_text(
    json.dumps(
        protocol,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 6. Final output
# ------------------------------------------------------------

print("=" * 70)
print("✅ SUES-200 TEST EVALUATION MANIFESTS READY")
print("=" * 70)

print("\nTest locations:")
print(
    test_queries_df[
        "location_id"
    ].nunique()
)

print("\nTotal test queries:")
print(len(test_queries_df))

print("\nTest satellite gallery:")
print(len(test_gallery_df))

print("\nQueries by nominal height:")
print(
    test_queries_df[
        "nominal_height"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nMissing gallery locations:")
print(len(missing_gallery_locations))

print("\nMissing positive tiles:")
print(len(missing_positive_tiles))

print("\nTest query manifest:")
print(test_query_csv)

print("\nTest gallery manifest:")
print(test_gallery_csv)

print("\nProtocol JSON:")
print(protocol_json)

✅ SUES-200 TEST EVALUATION MANIFESTS READY

Test locations:
40

Total test queries:
8000

Test satellite gallery:
40

Queries by nominal height:
nominal_height
150    2000
200    2000
250    2000
300    2000

Missing gallery locations:
0

Missing positive tiles:
0

Test query manifest:
/content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_test_queries.csv

Test gallery manifest:
/content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_test_gallery.csv

Protocol JSON:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/sues200_test_protocol.json


In [14]:
# ============================================================
# PHASE 2 — CELL 5
# Locate and inspect the local baseline checkpoint
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import torch

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Check known checkpoint locations
# ------------------------------------------------------------

checkpoint_candidates = [
    PROJECT_ROOT
    / "baseline_v1"
    / "checkpoint"
    / "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth",

    Path(
        "/content/drive/MyDrive/"
        "AdvancedEdgeGeoLPN_trained.pth"
    ),

    Path(
        "/content/drive/MyDrive/mobilegeo_final/"
        "AdvancedEdgeGeoLPN_trained.pth"
    ),
]

CHECKPOINT_PATH = None

print("Checking checkpoint locations:\n")

for candidate in checkpoint_candidates:

    exists = candidate.is_file()

    print(
        "✅" if exists else "❌",
        candidate,
    )

    if exists and CHECKPOINT_PATH is None:
        CHECKPOINT_PATH = candidate

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "AdvancedEdgeGeoLPN checkpoint was not found."
    )

print("\nSelected checkpoint:")
print(CHECKPOINT_PATH)

# ------------------------------------------------------------
# 2. Calculate checkpoint SHA-256
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


checkpoint_sha256 = sha256_file(
    CHECKPOINT_PATH
)

checkpoint_size_mb = (
    CHECKPOINT_PATH.stat().st_size
    / (1024 * 1024)
)

# ------------------------------------------------------------
# 3. Load checkpoint safely on CPU
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

except Exception as first_error:

    print(
        "\nSafe weights-only loading failed:"
    )
    print(repr(first_error))

    print(
        "\nTrying standard checkpoint loading..."
    )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

print("\nCheckpoint object type:")
print(type(checkpoint))

# ------------------------------------------------------------
# 4. Resolve state dictionary
# ------------------------------------------------------------

checkpoint_container_keys = []

if isinstance(checkpoint, dict):

    checkpoint_container_keys = list(
        checkpoint.keys()
    )

    if (
        "state_dict" in checkpoint
        and isinstance(
            checkpoint["state_dict"],
            dict,
        )
    ):
        state_dict = checkpoint[
            "state_dict"
        ]

        state_dict_source = (
            "checkpoint['state_dict']"
        )

    elif (
        "model_state_dict" in checkpoint
        and isinstance(
            checkpoint["model_state_dict"],
            dict,
        )
    ):
        state_dict = checkpoint[
            "model_state_dict"
        ]

        state_dict_source = (
            "checkpoint['model_state_dict']"
        )

    elif (
        "model" in checkpoint
        and isinstance(
            checkpoint["model"],
            dict,
        )
    ):
        state_dict = checkpoint[
            "model"
        ]

        state_dict_source = (
            "checkpoint['model']"
        )

    elif all(
        torch.is_tensor(value)
        for value in checkpoint.values()
    ):
        state_dict = checkpoint

        state_dict_source = (
            "checkpoint_root"
        )

    else:
        raise RuntimeError(
            "The checkpoint is a dictionary, but no "
            "recognizable model state dictionary was found."
        )

else:
    raise RuntimeError(
        "Unsupported checkpoint format: "
        f"{type(checkpoint)}"
    )

# Remove DataParallel prefix only for inspection.
clean_state_dict = {}

for key, value in state_dict.items():

    clean_key = (
        key[7:]
        if key.startswith("module.")
        else key
    )

    clean_state_dict[
        clean_key
    ] = value

# ------------------------------------------------------------
# 5. Inspect parameter information
# ------------------------------------------------------------

tensor_keys = [
    key
    for key, value in clean_state_dict.items()
    if torch.is_tensor(value)
]

total_tensor_values = sum(
    value.numel()
    for value in clean_state_dict.values()
    if torch.is_tensor(value)
)

parameter_prefix_counts = {}

for key in tensor_keys:

    prefix = key.split(".")[0]

    parameter_prefix_counts[prefix] = (
        parameter_prefix_counts.get(
            prefix,
            0,
        )
        + 1
    )

first_keys = tensor_keys[:40]

# ------------------------------------------------------------
# 6. Save resolved checkpoint path
# ------------------------------------------------------------

checkpoint_path_file = (
    PROJECT_ROOT
    / "configs"
    / "local_baseline_checkpoint_path.txt"
)

checkpoint_path_file.write_text(
    str(CHECKPOINT_PATH),
    encoding="utf-8",
)

checkpoint_audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model_name": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),

    "provenance_label": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_size_mb": (
        checkpoint_size_mb
    ),

    "checkpoint_sha256": (
        checkpoint_sha256
    ),

    "checkpoint_object_type": str(
        type(checkpoint)
    ),

    "checkpoint_container_keys": [
        str(key)
        for key in checkpoint_container_keys
    ],

    "state_dict_source": (
        state_dict_source
    ),

    "state_dict_tensor_count": len(
        tensor_keys
    ),

    "total_tensor_values": int(
        total_tensor_values
    ),

    "parameter_prefix_counts": (
        parameter_prefix_counts
    ),

    "first_parameter_keys": (
        first_keys
    ),

    "official_mobilegeo_checkpoint": False,

    "raw_image_model_loading_tested": False,
}

checkpoint_audit_json = (
    OUTPUT_ROOT
    / "local_checkpoint_audit.json"
)

checkpoint_audit_json.write_text(
    json.dumps(
        checkpoint_audit,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 7. Final output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ LOCAL CHECKPOINT AUDIT COMPLETE")
print("=" * 70)

print("\nModel name:")
print(
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
)

print("\nProvenance:")
print("LOCAL_REIMPLEMENTATION")

print("\nCheckpoint size:")
print(f"{checkpoint_size_mb:.2f} MB")

print("\nCheckpoint SHA-256:")
print(checkpoint_sha256)

print("\nState dictionary source:")
print(state_dict_source)

print("\nTensor entries:")
print(len(tensor_keys))

print("\nTotal tensor values:")
print(f"{total_tensor_values:,}")

print("\nParameter prefix counts:")

for prefix, count in sorted(
    parameter_prefix_counts.items()
):
    print(
        f"- {prefix}: {count}"
    )

print("\nFirst 40 parameter keys:")

for key in first_keys:
    print("-", key)

print("\nSaved checkpoint path:")
print(checkpoint_path_file)

print("\nSaved audit JSON:")
print(checkpoint_audit_json)

Checking checkpoint locations:

✅ /content/drive/MyDrive/mobilegeo_project/baseline_v1/checkpoint/AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth
✅ /content/drive/MyDrive/AdvancedEdgeGeoLPN_trained.pth
❌ /content/drive/MyDrive/mobilegeo_final/AdvancedEdgeGeoLPN_trained.pth

Selected checkpoint:
/content/drive/MyDrive/mobilegeo_project/baseline_v1/checkpoint/AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth

Checkpoint object type:
<class 'collections.OrderedDict'>

✅ LOCAL CHECKPOINT AUDIT COMPLETE

Model name:
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION

Provenance:
LOCAL_REIMPLEMENTATION

Checkpoint size:
19.05 MB

Checkpoint SHA-256:
adc64177582638690b841c1be390b7d05dfb27f4d2001b3d52fb74455020b807

State dictionary source:
checkpoint_root

Tensor entries:
316

Total tensor values:
4,964,526

Parameter prefix counts:
- backbone: 308
- fcs: 8

First 40 parameter keys:
- backbone.0.0.weight
- backbone.0.1.weight
- backbone.0.1.bias
- backbone.0.1.running_mean
- backbone.0.1.running_var
- backb

In [15]:
# ============================================================
# PHASE 2 — CELL 6
# Inspect exact checkpoint tensor shapes
# ============================================================

from pathlib import Path
import torch

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CHECKPOINT_PATH = Path(
    PROJECT_ROOT
    / "configs"
    / "local_baseline_checkpoint_path.txt"
).read_text(
    encoding="utf-8"
).strip()

CHECKPOINT_PATH = Path(CHECKPOINT_PATH)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)

try:
    state_dict = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )
except TypeError:
    state_dict = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

# Remove DataParallel prefix if present.
clean_state_dict = {
    (
        key[7:]
        if key.startswith("module.")
        else key
    ): value
    for key, value in state_dict.items()
}

tensor_items = [
    (key, value)
    for key, value in clean_state_dict.items()
    if torch.is_tensor(value)
]

# ------------------------------------------------------------
# 1. Print all descriptor-head tensors
# ------------------------------------------------------------

print("=" * 70)
print("DESCRIPTOR HEAD TENSORS")
print("=" * 70)

fcs_items = [
    (key, value)
    for key, value in tensor_items
    if key.startswith("fcs.")
]

for key, tensor in fcs_items:
    print(
        f"{key:<30} shape={tuple(tensor.shape)}"
    )

# ------------------------------------------------------------
# 2. Print final backbone tensors
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LAST 40 BACKBONE TENSORS")
print("=" * 70)

backbone_items = [
    (key, value)
    for key, value in tensor_items
    if key.startswith("backbone.")
]

for key, tensor in backbone_items[-40:]:
    print(
        f"{key:<55} shape={tuple(tensor.shape)}"
    )

# ------------------------------------------------------------
# 3. Infer descriptor dimensions
# ------------------------------------------------------------

linear_weights = [
    tensor
    for key, tensor in fcs_items
    if key.endswith(".weight")
    and tensor.ndim == 2
]

if not linear_weights:
    raise RuntimeError(
        "No FC weight tensors were found."
    )

part_dimensions = [
    int(tensor.shape[0])
    for tensor in linear_weights
]

input_dimensions = [
    int(tensor.shape[1])
    for tensor in linear_weights
]

descriptor_dimension = sum(
    part_dimensions
)

print("\n" + "=" * 70)
print("INFERRED MODEL STRUCTURE")
print("=" * 70)

print("\nNumber of descriptor parts:")
print(len(linear_weights))

print("\nInput dimensions per part:")
print(input_dimensions)

print("\nOutput dimensions per part:")
print(part_dimensions)

print("\nCombined descriptor dimension:")
print(descriptor_dimension)

expected_descriptor_dimension = 2048

print("\nExpected descriptor dimension:")
print(expected_descriptor_dimension)

print("\nDescriptor dimension matches:")
print(
    descriptor_dimension
    == expected_descriptor_dimension
)

if descriptor_dimension != expected_descriptor_dimension:
    raise RuntimeError(
        "Unexpected descriptor dimension."
    )

print("\n✅ Checkpoint structure inspection complete")

DESCRIPTOR HEAD TENSORS
fcs.0.weight                   shape=(512, 960)
fcs.0.bias                     shape=(512,)
fcs.1.weight                   shape=(512, 960)
fcs.1.bias                     shape=(512,)
fcs.2.weight                   shape=(512, 960)
fcs.2.bias                     shape=(512,)
fcs.3.weight                   shape=(512, 960)
fcs.3.bias                     shape=(512,)

LAST 40 BACKBONE TENSORS
backbone.14.block.1.1.running_var                       shape=(960,)
backbone.14.block.1.1.num_batches_tracked               shape=()
backbone.14.block.2.fc1.weight                          shape=(240, 960, 1, 1)
backbone.14.block.2.fc1.bias                            shape=(240,)
backbone.14.block.2.fc2.weight                          shape=(960, 240, 1, 1)
backbone.14.block.2.fc2.bias                            shape=(960,)
backbone.14.block.3.0.weight                            shape=(160, 960, 1, 1)
backbone.14.block.3.1.weight                            shape=(160,)
back

In [16]:
# ============================================================
# PHASE 2 — CELL 7
# Reconstruct local model, load checkpoint and smoke-test
# ============================================================

from pathlib import Path
from PIL import Image
import json
import time

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

CHECKPOINT_PATH = Path(
    (
        PROJECT_ROOT
        / "configs"
        / "local_baseline_checkpoint_path.txt"
    ).read_text(
        encoding="utf-8"
    ).strip()
)

TEST_GALLERY_MANIFEST = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
    / "sues200_test_gallery.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        CHECKPOINT_PATH
    )

if not TEST_GALLERY_MANIFEST.is_file():
    raise FileNotFoundError(
        TEST_GALLERY_MANIFEST
    )

# ------------------------------------------------------------
# 2. Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 3. Reconstruct model
# ------------------------------------------------------------

class AdvancedEdgeGeoLPNLocal(nn.Module):
    """
    Local reimplementation reconstructed from checkpoint keys.

    Architecture:
    - MobileNetV3-Large feature backbone
    - Four horizontal pooled feature parts
    - Four independent 960 -> 512 linear heads
    - Concatenated 2048-dimensional descriptor
    """

    def __init__(
        self,
        number_of_parts=4,
        part_dimension=512,
    ):
        super().__init__()

        base_model = models.mobilenet_v3_large(
            weights=None
        )

        self.backbone = base_model.features

        self.number_of_parts = (
            number_of_parts
        )

        self.part_pool = (
            nn.AdaptiveAvgPool2d(
                (
                    number_of_parts,
                    1,
                )
            )
        )

        self.fcs = nn.ModuleList(
            [
                nn.Linear(
                    960,
                    part_dimension,
                )
                for _ in range(
                    number_of_parts
                )
            ]
        )

        self.descriptor_dimension = (
            number_of_parts
            * part_dimension
        )

    def forward(
        self,
        images,
        normalize=True,
    ):
        feature_map = self.backbone(
            images
        )

        pooled = self.part_pool(
            feature_map
        )

        part_descriptors = []

        for part_index, head in enumerate(
            self.fcs
        ):
            part_feature = pooled[
                :,
                :,
                part_index,
                0,
            ]

            part_descriptor = head(
                part_feature
            )

            if normalize:
                part_descriptor = F.normalize(
                    part_descriptor,
                    p=2,
                    dim=1,
                )

            part_descriptors.append(
                part_descriptor
            )

        descriptor = torch.cat(
            part_descriptors,
            dim=1,
        )

        if normalize:
            descriptor = F.normalize(
                descriptor,
                p=2,
                dim=1,
            )

        return descriptor

# ------------------------------------------------------------
# 4. Load checkpoint
# ------------------------------------------------------------

try:
    state_dict = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    state_dict = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

clean_state_dict = {
    (
        key[7:]
        if key.startswith("module.")
        else key
    ): value
    for key, value in state_dict.items()
}

model = AdvancedEdgeGeoLPNLocal()

load_result = model.load_state_dict(
    clean_state_dict,
    strict=True,
)

model = model.to(
    DEVICE
)

model.eval()

print("\n✅ Checkpoint loaded with strict=True")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# ------------------------------------------------------------
# 5. Provisional preprocessing
# ------------------------------------------------------------
# This uses standard ImageNet preprocessing.
# It must remain recorded as project preprocessing unless the
# original baseline notebook proves a different transform.

INPUT_HEIGHT = 224
INPUT_WIDTH = 224

preprocess = transforms.Compose(
    [
        transforms.Resize(
            (
                INPUT_HEIGHT,
                INPUT_WIDTH,
            )
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ]
)

# ------------------------------------------------------------
# 6. Load one real satellite image
# ------------------------------------------------------------

gallery_df = pd.read_csv(
    TEST_GALLERY_MANIFEST,
    dtype={
        "location_id": str,
        "class_id": str,
    },
)

sample_path = Path(
    gallery_df.iloc[0][
        "absolute_path"
    ]
)

if not sample_path.is_file():
    raise FileNotFoundError(
        sample_path
    )

sample_image = Image.open(
    sample_path
).convert("RGB")

sample_tensor = preprocess(
    sample_image
).unsqueeze(0).to(
    DEVICE
)

print("\nSample image:")
print(sample_path)

print("\nOriginal image size:")
print(sample_image.size)

print("\nModel input shape:")
print(tuple(sample_tensor.shape))

# ------------------------------------------------------------
# 7. Descriptor smoke test
# ------------------------------------------------------------

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.inference_mode():
    descriptor = model(
        sample_tensor,
        normalize=True,
    )

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

inference_ms = (
    time.perf_counter()
    - start_time
) * 1000.0

descriptor_norm = float(
    torch.linalg.vector_norm(
        descriptor,
        dim=1,
    )[0].item()
)

finite_values = bool(
    torch.isfinite(
        descriptor
    ).all().item()
)

expected_shape = (
    1,
    2048,
)

if tuple(descriptor.shape) != expected_shape:
    raise RuntimeError(
        f"Unexpected descriptor shape: "
        f"{tuple(descriptor.shape)}"
    )

if not finite_values:
    raise RuntimeError(
        "Descriptor contains NaN or infinite values."
    )

# ------------------------------------------------------------
# 8. Save model reconstruction record
# ------------------------------------------------------------

reconstruction_record = {
    "model_name": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),
    "provenance_label": (
        "LOCAL_REIMPLEMENTATION"
    ),
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_loading": (
        "STRICT_SUCCESS"
    ),
    "backbone": (
        "torchvision_mobilenet_v3_large_features"
    ),
    "backbone_output_channels": 960,
    "horizontal_parts": 4,
    "part_descriptor_dimension": 512,
    "combined_descriptor_dimension": 2048,
    "input_height": INPUT_HEIGHT,
    "input_width": INPUT_WIDTH,
    "preprocessing": (
        "PROVISIONAL_IMAGENET_NORMALIZATION"
    ),
    "sample_image": str(
        sample_path
    ),
    "sample_descriptor_shape": list(
        descriptor.shape
    ),
    "sample_descriptor_norm": (
        descriptor_norm
    ),
    "sample_descriptor_finite": (
        finite_values
    ),
    "sample_inference_ms": (
        inference_ms
    ),
    "device": str(
        DEVICE
    ),
}

record_path = (
    OUTPUT_ROOT
    / "local_model_reconstruction.json"
)

record_path.write_text(
    json.dumps(
        reconstruction_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 9. Final output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ LOCAL MODEL RECONSTRUCTION SUCCESSFUL")
print("=" * 70)

print("\nModel:")
print(
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
)

print("\nCheckpoint loading:")
print("STRICT_SUCCESS")

print("\nBackbone output channels:")
print(960)

print("\nHorizontal descriptor parts:")
print(4)

print("\nDescriptor shape:")
print(tuple(descriptor.shape))

print("\nDescriptor norm:")
print(f"{descriptor_norm:.6f}")

print("\nFinite descriptor values:")
print(finite_values)

print("\nSingle-image inference:")
print(f"{inference_ms:.2f} ms")

print("\nSaved reconstruction record:")
print(record_path)

Device: cuda

✅ Checkpoint loaded with strict=True
Missing keys: []
Unexpected keys: []

Sample image:
/content/drive/MyDrive/SUES-200-512x512_extracted/SUES-200-512x512/satellite-view/0002/0.png

Original image size:
(512, 512)

Model input shape:
(1, 3, 224, 224)

✅ LOCAL MODEL RECONSTRUCTION SUCCESSFUL

Model:
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION

Checkpoint loading:
STRICT_SUCCESS

Backbone output channels:
960

Horizontal descriptor parts:
4

Descriptor shape:
(1, 2048)

Descriptor norm:
1.000000

Finite descriptor values:
True

Single-image inference:
1037.65 ms

Saved reconstruction record:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/local_model_reconstruction.json


In [17]:
# ============================================================
# PHASE 2 — CELL 8
# Generate gallery and test-query descriptors
# ============================================================

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import torch
import time
import json
import hashlib

# ------------------------------------------------------------
# 1. Confirm model from previous cell
# ------------------------------------------------------------

if "model" not in globals():
    raise RuntimeError(
        "Run Phase 2 Cell 7 first to create and load the model."
    )

if "preprocess" not in globals():
    raise RuntimeError(
        "Run Phase 2 Cell 7 first to create preprocessing."
    )

model.eval()

# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

EMBEDDING_ROOT = (
    OUTPUT_ROOT
    / "embeddings"
)

EMBEDDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

QUERY_MANIFEST = (
    MANIFEST_ROOT
    / "sues200_test_queries.csv"
)

GALLERY_MANIFEST = (
    MANIFEST_ROOT
    / "sues200_test_gallery.csv"
)

if not QUERY_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Missing query manifest:\n{QUERY_MANIFEST}"
    )

if not GALLERY_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Missing gallery manifest:\n{GALLERY_MANIFEST}"
    )

# ------------------------------------------------------------
# 3. Runtime settings
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
NUM_WORKERS = 2

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)

# ------------------------------------------------------------
# 4. Load manifests
# ------------------------------------------------------------

query_df = pd.read_csv(
    QUERY_MANIFEST,
    dtype={
        "frame_id": str,
        "location_id": str,
        "class_id": str,
        "positive_tile_id": str,
    },
)

gallery_df = pd.read_csv(
    GALLERY_MANIFEST,
    dtype={
        "tile_id": str,
        "location_id": str,
        "class_id": str,
    },
)

query_df["location_id"] = (
    query_df["location_id"]
    .str.zfill(4)
)

gallery_df["location_id"] = (
    gallery_df["location_id"]
    .str.zfill(4)
)

print("\nQueries:", len(query_df))
print("Gallery:", len(gallery_df))

# ------------------------------------------------------------
# 5. Image dataset
# ------------------------------------------------------------

class ManifestImageDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = Path(
            row["absolute_path"]
        )

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                tensor = self.transform(image)

        except Exception as error:
            raise RuntimeError(
                f"Failed to load image:\n"
                f"{image_path}\n"
                f"{repr(error)}"
            )

        return tensor, index

# ------------------------------------------------------------
# 6. Descriptor extraction
# ------------------------------------------------------------

def extract_descriptors(
    dataframe,
    dataset_name,
):

    dataset = ManifestImageDataset(
        dataframe=dataframe,
        transform=preprocess,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        persistent_workers=(
            NUM_WORKERS > 0
        ),
    )

    # CUDA warm-up.
    if DEVICE.type == "cuda":

        warmup = torch.zeros(
            4,
            3,
            224,
            224,
            device=DEVICE,
        )

        with torch.inference_mode():

            for _ in range(3):
                _ = model(
                    warmup,
                    normalize=True,
                )

        torch.cuda.synchronize()

    descriptors = []
    output_indices = []

    total_start = time.perf_counter()
    inference_seconds = 0.0

    with torch.inference_mode():

        for batch_number, (
            images,
            indices,
        ) in enumerate(
            loader,
            start=1,
        ):

            images = images.to(
                DEVICE,
                non_blocking=True,
            )

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            inference_start = (
                time.perf_counter()
            )

            batch_descriptors = model(
                images,
                normalize=True,
            )

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            inference_seconds += (
                time.perf_counter()
                - inference_start
            )

            descriptors.append(
                batch_descriptors
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            output_indices.extend(
                indices.numpy().tolist()
            )

            processed = min(
                batch_number * BATCH_SIZE,
                len(dataset),
            )

            print(
                f"{dataset_name}: "
                f"{processed}/{len(dataset)}",
                end="\r",
            )

    total_seconds = (
        time.perf_counter()
        - total_start
    )

    descriptor_array = np.concatenate(
        descriptors,
        axis=0,
    )

    output_indices = np.asarray(
        output_indices,
        dtype=np.int64,
    )

    # Restore exact manifest order.
    order = np.argsort(
        output_indices
    )

    descriptor_array = (
        descriptor_array[order]
    )

    if len(descriptor_array) != len(dataframe):
        raise RuntimeError(
            f"{dataset_name} descriptor count mismatch."
        )

    if descriptor_array.shape[1] != 2048:
        raise RuntimeError(
            f"Unexpected descriptor dimension: "
            f"{descriptor_array.shape}"
        )

    if not np.isfinite(
        descriptor_array
    ).all():
        raise RuntimeError(
            f"{dataset_name} contains invalid descriptors."
        )

    norms = np.linalg.norm(
        descriptor_array,
        axis=1,
    )

    timing = {
        "dataset_name": dataset_name,
        "image_count": int(
            len(dataframe)
        ),
        "batch_size": BATCH_SIZE,
        "device": str(DEVICE),
        "descriptor_dimension": int(
            descriptor_array.shape[1]
        ),
        "total_end_to_end_seconds": float(
            total_seconds
        ),
        "inference_only_seconds": float(
            inference_seconds
        ),
        "end_to_end_ms_per_image": float(
            total_seconds
            * 1000
            / len(dataframe)
        ),
        "inference_ms_per_image": float(
            inference_seconds
            * 1000
            / len(dataframe)
        ),
        "descriptor_norm_mean": float(
            norms.mean()
        ),
        "descriptor_norm_std": float(
            norms.std()
        ),
    }

    print(
        f"\n✅ {dataset_name} descriptors complete"
    )

    print(
        "Shape:",
        descriptor_array.shape,
    )

    print(
        "Inference-only ms/image:",
        f"{timing['inference_ms_per_image']:.3f}",
    )

    print(
        "End-to-end ms/image:",
        f"{timing['end_to_end_ms_per_image']:.3f}",
    )

    return descriptor_array, timing

# ------------------------------------------------------------
# 7. Extract satellite gallery descriptors
# ------------------------------------------------------------

gallery_descriptors, gallery_timing = (
    extract_descriptors(
        dataframe=gallery_df,
        dataset_name="test_gallery",
    )
)

# ------------------------------------------------------------
# 8. Extract drone query descriptors
# ------------------------------------------------------------

query_descriptors, query_timing = (
    extract_descriptors(
        dataframe=query_df,
        dataset_name="test_queries",
    )
)

# ------------------------------------------------------------
# 9. Save gallery descriptor package
# ------------------------------------------------------------

gallery_output = (
    EMBEDDING_ROOT
    / "sues200_test_gallery_embeddings.npz"
)

np.savez_compressed(
    gallery_output,
    descriptors=gallery_descriptors,
    tile_ids=gallery_df[
        "tile_id"
    ].astype(str).to_numpy(),
    location_ids=gallery_df[
        "location_id"
    ].astype(str).to_numpy(),
    absolute_paths=gallery_df[
        "absolute_path"
    ].astype(str).to_numpy(),
)

# ------------------------------------------------------------
# 10. Save query descriptor package
# ------------------------------------------------------------

query_output = (
    EMBEDDING_ROOT
    / "sues200_test_query_embeddings.npz"
)

np.savez_compressed(
    query_output,
    descriptors=query_descriptors,
    frame_ids=query_df[
        "frame_id"
    ].astype(str).to_numpy(),
    location_ids=query_df[
        "location_id"
    ].astype(str).to_numpy(),
    nominal_heights=query_df[
        "nominal_height"
    ].to_numpy(dtype=np.int32),
    positive_tile_ids=query_df[
        "positive_tile_id"
    ].astype(str).to_numpy(),
    absolute_paths=query_df[
        "absolute_path"
    ].astype(str).to_numpy(),
)

# ------------------------------------------------------------
# 11. File hashes
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

# ------------------------------------------------------------
# 12. Save extraction report
# ------------------------------------------------------------

extraction_report = {
    "model_name": (
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
    ),
    "provenance_label": (
        "LOCAL_REIMPLEMENTATION"
    ),
    "benchmark_protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),
    "input_size": [
        224,
        224,
    ],
    "preprocessing": (
        "PROVISIONAL_IMAGENET_NORMALIZATION"
    ),
    "descriptor_dimension": 2048,
    "gallery": gallery_timing,
    "queries": query_timing,
    "gallery_embedding_file": str(
        gallery_output
    ),
    "gallery_embedding_sha256": (
        sha256_file(
            gallery_output
        )
    ),
    "query_embedding_file": str(
        query_output
    ),
    "query_embedding_sha256": (
        sha256_file(
            query_output
        )
    ),
}

report_path = (
    OUTPUT_ROOT
    / "descriptor_extraction_report.json"
)

report_path.write_text(
    json.dumps(
        extraction_report,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 13. Final output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("✅ SUES-200 DESCRIPTOR EXTRACTION COMPLETE")
print("=" * 72)

print("\nGallery descriptor shape:")
print(gallery_descriptors.shape)

print("\nQuery descriptor shape:")
print(query_descriptors.shape)

print("\nGallery file:")
print(gallery_output)

print("\nQuery file:")
print(query_output)

print("\nQuery inference-only speed:")
print(
    f"{query_timing['inference_ms_per_image']:.3f} ms/image"
)

print("\nQuery end-to-end speed:")
print(
    f"{query_timing['end_to_end_ms_per_image']:.3f} ms/image"
)

print("\nExtraction report:")
print(report_path)

Device: cuda
Batch size: 64
Workers: 2

Queries: 8000
Gallery: 40
test_gallery: 40/40
✅ test_gallery descriptors complete
Shape: (40, 2048)
Inference-only ms/image: 7.321
End-to-end ms/image: 107.287
test_queries: 8000/8000
✅ test_queries descriptors complete
Shape: (8000, 2048)
Inference-only ms/image: 0.856
End-to-end ms/image: 158.904

✅ SUES-200 DESCRIPTOR EXTRACTION COMPLETE

Gallery descriptor shape:
(40, 2048)

Query descriptor shape:
(8000, 2048)

Gallery file:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/embeddings/sues200_test_gallery_embeddings.npz

Query file:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/embeddings/sues200_test_query_embeddings.npz

Query inference-only speed:
0.856 ms/image

Query end-to-end speed:
158.904 ms/image

Extraction report:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/descriptor_extraction_report.json


In [20]:
# ============================================================
# PHASE 2 — CELL 9
# Corrected SUES-200 location-disjoint retrieval evaluation
#
# Calculates:
# - Overall Recall@1, Recall@5, Recall@10 and mAP
# - Metrics for heights 150, 200, 250 and 300
# - Complete rankings for all 8,000 queries
# - Per-location metrics
#
# Provenance:
# AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION
#
# Protocol:
# PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20
# ============================================================

from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

EMBEDDING_ROOT = (
    OUTPUT_ROOT
    / "embeddings"
)

GALLERY_FILE = (
    EMBEDDING_ROOT
    / "sues200_test_gallery_embeddings.npz"
)

QUERY_FILE = (
    EMBEDDING_ROOT
    / "sues200_test_query_embeddings.npz"
)

if not GALLERY_FILE.is_file():
    raise FileNotFoundError(GALLERY_FILE)

if not QUERY_FILE.is_file():
    raise FileNotFoundError(QUERY_FILE)

# ------------------------------------------------------------
# 2. Load descriptor packages
# ------------------------------------------------------------

gallery_data = np.load(
    GALLERY_FILE,
    allow_pickle=True,
)

query_data = np.load(
    QUERY_FILE,
    allow_pickle=True,
)

gallery_descriptors = (
    gallery_data["descriptors"]
    .astype(np.float32)
)

gallery_tile_ids = (
    gallery_data["tile_ids"]
    .astype(str)
)

gallery_location_ids = (
    gallery_data["location_ids"]
    .astype(str)
)

query_descriptors = (
    query_data["descriptors"]
    .astype(np.float32)
)

query_frame_ids = (
    query_data["frame_ids"]
    .astype(str)
)

query_location_ids = (
    query_data["location_ids"]
    .astype(str)
)

query_heights = (
    query_data["nominal_heights"]
    .astype(np.int32)
)

positive_tile_ids = (
    query_data["positive_tile_ids"]
    .astype(str)
)

print("Gallery descriptors:", gallery_descriptors.shape)
print("Query descriptors:", query_descriptors.shape)

# ------------------------------------------------------------
# 3. Validate descriptors and identities
# ------------------------------------------------------------

if gallery_descriptors.shape != (40, 2048):
    raise RuntimeError(
        f"Unexpected gallery shape: "
        f"{gallery_descriptors.shape}"
    )

if query_descriptors.shape != (8000, 2048):
    raise RuntimeError(
        f"Unexpected query shape: "
        f"{query_descriptors.shape}"
    )

if len(set(gallery_tile_ids)) != len(gallery_tile_ids):
    raise RuntimeError(
        "Duplicate gallery tile IDs detected."
    )

gallery_index_by_tile = {
    tile_id: index
    for index, tile_id in enumerate(
        gallery_tile_ids
    )
}

missing_positive_tiles = sorted(
    set(positive_tile_ids)
    - set(gallery_tile_ids)
)

if missing_positive_tiles:
    raise RuntimeError(
        "Positive gallery tiles are missing:\n"
        + str(missing_positive_tiles)
    )

if not np.isfinite(gallery_descriptors).all():
    raise RuntimeError(
        "Gallery descriptors contain invalid values."
    )

if not np.isfinite(query_descriptors).all():
    raise RuntimeError(
        "Query descriptors contain invalid values."
    )

# Normalize again defensively.
gallery_norms = np.linalg.norm(
    gallery_descriptors,
    axis=1,
    keepdims=True,
)

query_norms = np.linalg.norm(
    query_descriptors,
    axis=1,
    keepdims=True,
)

gallery_descriptors = (
    gallery_descriptors
    / np.maximum(gallery_norms, 1e-12)
)

query_descriptors = (
    query_descriptors
    / np.maximum(query_norms, 1e-12)
)

# ------------------------------------------------------------
# 4. Exact cosine retrieval
# ------------------------------------------------------------

search_start = perf_counter()

similarity_matrix = (
    query_descriptors
    @ gallery_descriptors.T
)

ranking_indices = np.argsort(
    -similarity_matrix,
    axis=1,
)

search_seconds = (
    perf_counter()
    - search_start
)

ranking_scores = np.take_along_axis(
    similarity_matrix,
    ranking_indices,
    axis=1,
)

# ------------------------------------------------------------
# 5. Calculate positive rank for every query
# ------------------------------------------------------------

positive_gallery_indices = np.asarray(
    [
        gallery_index_by_tile[
            tile_id
        ]
        for tile_id in positive_tile_ids
    ],
    dtype=np.int32,
)

positive_matches = (
    ranking_indices
    == positive_gallery_indices[:, None]
)

positive_ranks = (
    np.argmax(
        positive_matches,
        axis=1,
    )
    + 1
)

if not positive_matches.any(axis=1).all():
    raise RuntimeError(
        "At least one positive gallery tile "
        "was not present in its ranking."
    )

# One positive satellite image per query.
reciprocal_ranks = (
    1.0
    / positive_ranks.astype(np.float64)
)

top1_indices = ranking_indices[:, 0]
top2_indices = ranking_indices[:, 1]

top1_scores = ranking_scores[:, 0]
top2_scores = ranking_scores[:, 1]

top1_top2_margins = (
    top1_scores
    - top2_scores
)

# ------------------------------------------------------------
# 6. Metric helper
# ------------------------------------------------------------

def calculate_metrics(
    selected_indices,
    group_name,
    nominal_height,
):
    selected_indices = np.asarray(
        selected_indices,
        dtype=np.int64,
    )

    ranks = positive_ranks[
        selected_indices
    ]

    margins = top1_top2_margins[
        selected_indices
    ]

    return {
        "group": group_name,
        "nominal_height": nominal_height,
        "query_count": int(
            len(selected_indices)
        ),
        "location_count": int(
            len(
                np.unique(
                    query_location_ids[
                        selected_indices
                    ]
                )
            )
        ),
        "recall_at_1_percent": float(
            100.0
            * np.mean(ranks <= 1)
        ),
        "recall_at_5_percent": float(
            100.0
            * np.mean(ranks <= 5)
        ),
        "recall_at_10_percent": float(
            100.0
            * np.mean(ranks <= 10)
        ),
        "mean_average_precision_percent": float(
            100.0
            * np.mean(
                1.0
                / ranks.astype(np.float64)
            )
        ),
        "mean_positive_rank": float(
            np.mean(ranks)
        ),
        "median_positive_rank": float(
            np.median(ranks)
        ),
        "p95_positive_rank": float(
            np.percentile(
                ranks,
                95,
            )
        ),
        "mean_top1_top2_margin": float(
            np.mean(margins)
        ),
        "median_top1_top2_margin": float(
            np.median(margins)
        ),
    }

# ------------------------------------------------------------
# 7. Overall and per-height metrics
# ------------------------------------------------------------

metric_records = []

all_query_indices = np.arange(
    len(query_frame_ids)
)

metric_records.append(
    calculate_metrics(
        selected_indices=all_query_indices,
        group_name="overall",
        nominal_height="all",
    )
)

for height in [
    150,
    200,
    250,
    300,
]:
    height_indices = np.flatnonzero(
        query_heights == height
    )

    metric_records.append(
        calculate_metrics(
            selected_indices=height_indices,
            group_name=f"height_{height}",
            nominal_height=height,
        )
    )

metrics_df = pd.DataFrame(
    metric_records
)

metrics_csv = (
    OUTPUT_ROOT
    / "corrected_location_disjoint_metrics.csv"
)

metrics_df.to_csv(
    metrics_csv,
    index=False,
)

# ------------------------------------------------------------
# 8. Per-location metrics
# ------------------------------------------------------------

location_metric_records = []

for location_id in sorted(
    np.unique(query_location_ids)
):
    for height in [
        150,
        200,
        250,
        300,
    ]:
        selected = np.flatnonzero(
            (
                query_location_ids
                == location_id
            )
            & (
                query_heights
                == height
            )
        )

        if len(selected) == 0:
            continue

        location_ranks = (
            positive_ranks[selected]
        )

        location_metric_records.append({
            "location_id": location_id,
            "nominal_height": height,
            "query_count": int(
                len(selected)
            ),
            "recall_at_1_percent": float(
                100.0
                * np.mean(
                    location_ranks <= 1
                )
            ),
            "recall_at_5_percent": float(
                100.0
                * np.mean(
                    location_ranks <= 5
                )
            ),
            "recall_at_10_percent": float(
                100.0
                * np.mean(
                    location_ranks <= 10
                )
            ),
            "mean_average_precision_percent": float(
                100.0
                * np.mean(
                    1.0
                    / location_ranks.astype(
                        np.float64
                    )
                )
            ),
            "mean_positive_rank": float(
                np.mean(location_ranks)
            ),
        })

location_metrics_df = pd.DataFrame(
    location_metric_records
)

location_metrics_csv = (
    OUTPUT_ROOT
    / "corrected_per_location_metrics.csv"
)

location_metrics_df.to_csv(
    location_metrics_csv,
    index=False,
)

# ------------------------------------------------------------
# 9. Save complete rankings
# ------------------------------------------------------------

ranking_records = []

query_count, gallery_count = (
    ranking_indices.shape
)

for query_index in range(query_count):

    positive_tile_id = (
        positive_tile_ids[
            query_index
        ]
    )

    for rank_offset in range(
        gallery_count
    ):
        gallery_index = int(
            ranking_indices[
                query_index,
                rank_offset,
            ]
        )

        candidate_tile_id = (
            gallery_tile_ids[
                gallery_index
            ]
        )

        ranking_records.append({
            "method": (
                "AdvancedEdgeGeoLPN_"
                "LOCAL_REIMPLEMENTATION"
            ),
            "provenance_label": (
                "LOCAL_REIMPLEMENTATION"
            ),
            "benchmark_protocol": (
                "PROJECT_DEFINED_"
                "LOCATION_DISJOINT_60_20_20"
            ),
            "split": "test",
            "frame_id": (
                query_frame_ids[
                    query_index
                ]
            ),
            "query_location_id": (
                query_location_ids[
                    query_index
                ]
            ),
            "nominal_height": int(
                query_heights[
                    query_index
                ]
            ),
            "tile_id": (
                candidate_tile_id
            ),
            "gallery_location_id": (
                gallery_location_ids[
                    gallery_index
                ]
            ),
            "rank": (
                rank_offset + 1
            ),
            "score": float(
                ranking_scores[
                    query_index,
                    rank_offset,
                ]
            ),
            "is_positive": bool(
                candidate_tile_id
                == positive_tile_id
            ),
            "positive_tile_id": (
                positive_tile_id
            ),
            "positive_rank": int(
                positive_ranks[
                    query_index
                ]
            ),
            "top1_top2_margin": float(
                top1_top2_margins[
                    query_index
                ]
            ),
        })

rankings_df = pd.DataFrame(
    ranking_records
)

rankings_csv = (
    OUTPUT_ROOT
    / "corrected_location_disjoint_rankings.csv"
)

rankings_df.to_csv(
    rankings_csv,
    index=False,
)

# ------------------------------------------------------------
# 10. Save compact query-level results
# ------------------------------------------------------------

query_results_df = pd.DataFrame({
    "frame_id": query_frame_ids,
    "query_location_id": (
        query_location_ids
    ),
    "nominal_height": query_heights,
    "positive_tile_id": (
        positive_tile_ids
    ),
    "positive_rank": (
        positive_ranks
    ),
    "reciprocal_rank": (
        reciprocal_ranks
    ),
    "top1_tile_id": (
        gallery_tile_ids[
            top1_indices
        ]
    ),
    "top1_location_id": (
        gallery_location_ids[
            top1_indices
        ]
    ),
    "top1_score": top1_scores,
    "top2_tile_id": (
        gallery_tile_ids[
            top2_indices
        ]
    ),
    "top2_score": top2_scores,
    "top1_top2_margin": (
        top1_top2_margins
    ),
    "top1_correct": (
        positive_ranks == 1
    ),
})

query_results_csv = (
    OUTPUT_ROOT
    / "corrected_query_results.csv"
)

query_results_df.to_csv(
    query_results_csv,
    index=False,
)

# ------------------------------------------------------------
# 11. Search timing
# ------------------------------------------------------------

search_timing = {
    "query_count": int(query_count),
    "gallery_count": int(gallery_count),
    "similarity_comparisons": int(
        query_count
        * gallery_count
    ),
    "search_seconds": float(
        search_seconds
    ),
    "search_ms_per_query": float(
        search_seconds
        * 1000.0
        / query_count
    ),
    "search_scope": (
        "COSINE_MATRIX_MULTIPLICATION_AND_FULL_SORT"
    ),
    "descriptor_extraction_included": False,
}

# ------------------------------------------------------------
# 12. Save final metric JSON
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


final_results = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model_name": (
        "AdvancedEdgeGeoLPN_"
        "LOCAL_REIMPLEMENTATION"
    ),

    "provenance_label": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "benchmark_status": (
        "CORRECTED_PROJECT_DEFINED_"
        "LOCATION_DISJOINT_BENCHMARK"
    ),

    "official_mobilegeo_result": False,

    "official_sues200_protocol": False,

    "benchmark_protocol": (
        "PROJECT_DEFINED_"
        "LOCATION_DISJOINT_60_20_20"
    ),

    "train_location_count": 120,
    "validation_location_count": 40,
    "test_location_count": 40,

    "test_query_count": int(
        query_count
    ),

    "test_gallery_count": int(
        gallery_count
    ),

    "descriptor_dimension": 2048,

    "input_resolution": [
        224,
        224,
    ],

    "preprocessing": (
        "PROVISIONAL_IMAGENET_NORMALIZATION"
    ),

    "similarity": (
        "COSINE_SIMILARITY"
    ),

    "metrics": metric_records,

    "search_timing": search_timing,

    "files": {
        "metrics_csv": str(
            metrics_csv
        ),
        "query_results_csv": str(
            query_results_csv
        ),
        "rankings_csv": str(
            rankings_csv
        ),
        "per_location_metrics_csv": str(
            location_metrics_csv
        ),
    },

    "hashes": {
        "gallery_embeddings_sha256": (
            sha256_file(
                GALLERY_FILE
            )
        ),
        "query_embeddings_sha256": (
            sha256_file(
                QUERY_FILE
            )
        ),
        "metrics_csv_sha256": (
            sha256_file(
                metrics_csv
            )
        ),
        "rankings_csv_sha256": (
            sha256_file(
                rankings_csv
            )
        ),
    },

    "reporting_restrictions": [
        (
            "Do not describe this as an "
            "official MobileGeo result."
        ),
        (
            "Do not describe this split as "
            "the official SUES-200 protocol."
        ),
        (
            "The nominal height labels are not "
            "verified as AGL or MSL altitude."
        ),
        (
            "The result measures image retrieval, "
            "not verified geographic UAV pose."
        ),
    ],
}

metrics_json = (
    OUTPUT_ROOT
    / "corrected_location_disjoint_metrics.json"
)

metrics_json.write_text(
    json.dumps(
        final_results,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 13. Print results
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("✅ CORRECTED LOCATION-DISJOINT EVALUATION COMPLETE")
print("=" * 78)

display_columns = [
    "group",
    "nominal_height",
    "query_count",
    "recall_at_1_percent",
    "recall_at_5_percent",
    "recall_at_10_percent",
    "mean_average_precision_percent",
    "mean_positive_rank",
]

print(
    "\n",
    metrics_df[
        display_columns
    ].to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda value: f"{value:.2f}"
            ),
            "recall_at_5_percent": (
                lambda value: f"{value:.2f}"
            ),
            "recall_at_10_percent": (
                lambda value: f"{value:.2f}"
            ),
            "mean_average_precision_percent": (
                lambda value: f"{value:.2f}"
            ),
            "mean_positive_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nSearch time:")
print(f"{search_seconds:.6f} seconds")

print("\nSearch time per query:")
print(
    f"{search_timing['search_ms_per_query']:.6f} ms"
)

print("\nComplete ranking rows:")
print(len(rankings_df))

print("\nMetrics CSV:")
print(metrics_csv)

print("\nMetrics JSON:")
print(metrics_json)

print("\nQuery results:")
print(query_results_csv)

print("\nComplete rankings:")
print(rankings_csv)

print("\nPer-location metrics:")
print(location_metrics_csv)

Gallery descriptors: (40, 2048)
Query descriptors: (8000, 2048)

✅ CORRECTED LOCATION-DISJOINT EVALUATION COMPLETE

      group nominal_height  query_count recall_at_1_percent recall_at_5_percent recall_at_10_percent mean_average_precision_percent mean_positive_rank
   overall            all         8000               94.58               99.99               100.00                          96.93              1.082
height_150            150         2000               93.35               99.95               100.00                          96.19              1.103
height_200            200         2000               94.55              100.00               100.00                          96.89              1.083
height_250            250         2000               95.45              100.00               100.00                          97.33              1.077
height_300            300         2000               94.95              100.00               100.00                          97.29   

In [21]:
# ============================================================
# PHASE 2 — CELL 10
# Failure analysis and confidence-margin analysis
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

QUERY_RESULTS_CSV = (
    OUTPUT_ROOT
    / "corrected_query_results.csv"
)

PER_LOCATION_CSV = (
    OUTPUT_ROOT
    / "corrected_per_location_metrics.csv"
)

if not QUERY_RESULTS_CSV.is_file():
    raise FileNotFoundError(
        QUERY_RESULTS_CSV
    )

if not PER_LOCATION_CSV.is_file():
    raise FileNotFoundError(
        PER_LOCATION_CSV
    )

# ------------------------------------------------------------
# 1. Load results
# ------------------------------------------------------------

query_df = pd.read_csv(
    QUERY_RESULTS_CSV,
    dtype={
        "frame_id": str,
        "query_location_id": str,
        "positive_tile_id": str,
        "top1_tile_id": str,
        "top1_location_id": str,
        "top2_tile_id": str,
    },
)

location_df = pd.read_csv(
    PER_LOCATION_CSV,
    dtype={
        "location_id": str,
    },
)

query_df["query_location_id"] = (
    query_df["query_location_id"]
    .str.zfill(4)
)

query_df["top1_location_id"] = (
    query_df["top1_location_id"]
    .str.zfill(4)
)

location_df["location_id"] = (
    location_df["location_id"]
    .str.zfill(4)
)

# ------------------------------------------------------------
# 2. Separate correct and incorrect Top-1 results
# ------------------------------------------------------------

correct_df = (
    query_df[
        query_df["top1_correct"] == True
    ]
    .copy()
)

failure_df = (
    query_df[
        query_df["top1_correct"] == False
    ]
    .copy()
    .sort_values(
        [
            "positive_rank",
            "top1_top2_margin",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

failure_csv = (
    OUTPUT_ROOT
    / "top1_failure_cases.csv"
)

failure_df.to_csv(
    failure_csv,
    index=False,
)

# ------------------------------------------------------------
# 3. Failure summary by nominal height
# ------------------------------------------------------------

height_summary = (
    query_df
    .groupby(
        "nominal_height"
    )
    .agg(
        query_count=(
            "frame_id",
            "count",
        ),
        top1_correct_count=(
            "top1_correct",
            "sum",
        ),
        mean_positive_rank=(
            "positive_rank",
            "mean",
        ),
        mean_margin=(
            "top1_top2_margin",
            "mean",
        ),
        median_margin=(
            "top1_top2_margin",
            "median",
        ),
    )
    .reset_index()
)

height_summary[
    "top1_failure_count"
] = (
    height_summary["query_count"]
    - height_summary[
        "top1_correct_count"
    ]
)

height_summary[
    "recall_at_1_percent"
] = (
    100.0
    * height_summary[
        "top1_correct_count"
    ]
    / height_summary[
        "query_count"
    ]
)

height_summary[
    "failure_rate_percent"
] = (
    100.0
    * height_summary[
        "top1_failure_count"
    ]
    / height_summary[
        "query_count"
    ]
)

height_summary_csv = (
    OUTPUT_ROOT
    / "failure_summary_by_height.csv"
)

height_summary.to_csv(
    height_summary_csv,
    index=False,
)

# ------------------------------------------------------------
# 4. Failure summary by location
# ------------------------------------------------------------

location_summary = (
    query_df
    .groupby(
        "query_location_id"
    )
    .agg(
        query_count=(
            "frame_id",
            "count",
        ),
        top1_correct_count=(
            "top1_correct",
            "sum",
        ),
        mean_positive_rank=(
            "positive_rank",
            "mean",
        ),
        maximum_positive_rank=(
            "positive_rank",
            "max",
        ),
        mean_margin=(
            "top1_top2_margin",
            "mean",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "query_location_id": (
                "location_id"
            )
        }
    )
)

location_summary[
    "top1_failure_count"
] = (
    location_summary["query_count"]
    - location_summary[
        "top1_correct_count"
    ]
)

location_summary[
    "recall_at_1_percent"
] = (
    100.0
    * location_summary[
        "top1_correct_count"
    ]
    / location_summary[
        "query_count"
    ]
)

location_summary[
    "failure_rate_percent"
] = (
    100.0
    * location_summary[
        "top1_failure_count"
    ]
    / location_summary[
        "query_count"
    ]
)

worst_locations = (
    location_summary
    .sort_values(
        [
            "failure_rate_percent",
            "mean_positive_rank",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

worst_locations_csv = (
    OUTPUT_ROOT
    / "worst_test_locations.csv"
)

worst_locations.to_csv(
    worst_locations_csv,
    index=False,
)

# ------------------------------------------------------------
# 5. Confidence-margin comparison
# ------------------------------------------------------------

def describe_values(values):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "count": int(
            len(values)
        ),
        "mean": float(
            np.mean(values)
        ),
        "median": float(
            np.median(values)
        ),
        "p05": float(
            np.percentile(values, 5)
        ),
        "p25": float(
            np.percentile(values, 25)
        ),
        "p75": float(
            np.percentile(values, 75)
        ),
        "p95": float(
            np.percentile(values, 95)
        ),
    }

correct_margin_stats = describe_values(
    correct_df[
        "top1_top2_margin"
    ]
)

failure_margin_stats = describe_values(
    failure_df[
        "top1_top2_margin"
    ]
)

# ------------------------------------------------------------
# 6. Candidate ambiguity thresholds
# ------------------------------------------------------------

candidate_thresholds = [
    0.001,
    0.002,
    0.005,
    0.010,
    0.020,
    0.050,
]

threshold_records = []

for threshold in candidate_thresholds:

    accepted = (
        query_df[
            "top1_top2_margin"
        ] >= threshold
    )

    accepted_count = int(
        accepted.sum()
    )

    rejected_count = int(
        (~accepted).sum()
    )

    if accepted_count > 0:

        accepted_accuracy = float(
            100.0
            * query_df.loc[
                accepted,
                "top1_correct",
            ].mean()
        )

    else:
        accepted_accuracy = None

    threshold_records.append({
        "margin_threshold": threshold,
        "accepted_query_count": (
            accepted_count
        ),
        "ambiguous_query_count": (
            rejected_count
        ),
        "coverage_percent": float(
            100.0
            * accepted_count
            / len(query_df)
        ),
        "accepted_top1_accuracy_percent": (
            accepted_accuracy
        ),
    })

threshold_df = pd.DataFrame(
    threshold_records
)

threshold_csv = (
    OUTPUT_ROOT
    / "margin_threshold_analysis.csv"
)

threshold_df.to_csv(
    threshold_csv,
    index=False,
)

# ------------------------------------------------------------
# 7. Positive-rank distribution
# ------------------------------------------------------------

rank_distribution = (
    query_df[
        "positive_rank"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "positive_rank"
    )
    .reset_index(
        name="query_count"
    )
)

rank_distribution[
    "query_percent"
] = (
    100.0
    * rank_distribution[
        "query_count"
    ]
    / len(query_df)
)

rank_distribution_csv = (
    OUTPUT_ROOT
    / "positive_rank_distribution.csv"
)

rank_distribution.to_csv(
    rank_distribution_csv,
    index=False,
)

# ------------------------------------------------------------
# 8. Save failure-analysis JSON
# ------------------------------------------------------------

failure_report = {
    "model_name": (
        "AdvancedEdgeGeoLPN_"
        "LOCAL_REIMPLEMENTATION"
    ),
    "provenance_label": (
        "LOCAL_REIMPLEMENTATION"
    ),
    "benchmark_protocol": (
        "PROJECT_DEFINED_"
        "LOCATION_DISJOINT_60_20_20"
    ),
    "total_queries": int(
        len(query_df)
    ),
    "top1_correct_queries": int(
        len(correct_df)
    ),
    "top1_failure_queries": int(
        len(failure_df)
    ),
    "overall_recall_at_1_percent": float(
        100.0
        * query_df[
            "top1_correct"
        ].mean()
    ),
    "maximum_positive_rank": int(
        query_df[
            "positive_rank"
        ].max()
    ),
    "correct_margin_statistics": (
        correct_margin_stats
    ),
    "failure_margin_statistics": (
        failure_margin_stats
    ),
    "worst_height": int(
        height_summary
        .sort_values(
            "recall_at_1_percent"
        )
        .iloc[0][
            "nominal_height"
        ]
    ),
    "files": {
        "top1_failures": str(
            failure_csv
        ),
        "height_summary": str(
            height_summary_csv
        ),
        "worst_locations": str(
            worst_locations_csv
        ),
        "margin_thresholds": str(
            threshold_csv
        ),
        "rank_distribution": str(
            rank_distribution_csv
        ),
    },
}

failure_report_json = (
    OUTPUT_ROOT
    / "failure_analysis.json"
)

failure_report_json.write_text(
    json.dumps(
        failure_report,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 9. Final output
# ------------------------------------------------------------

print("=" * 76)
print("✅ FAILURE ANALYSIS COMPLETE")
print("=" * 76)

print("\nTotal queries:")
print(len(query_df))

print("\nTop-1 correct:")
print(len(correct_df))

print("\nTop-1 failures:")
print(len(failure_df))

print("\nMaximum positive rank:")
print(
    query_df[
        "positive_rank"
    ].max()
)

print("\nFailure summary by height:")
print(
    height_summary[
        [
            "nominal_height",
            "query_count",
            "top1_failure_count",
            "recall_at_1_percent",
            "failure_rate_percent",
            "mean_positive_rank",
        ]
    ].to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda value: f"{value:.2f}"
            ),
            "failure_rate_percent": (
                lambda value: f"{value:.2f}"
            ),
            "mean_positive_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nTop 10 worst locations:")
print(
    worst_locations[
        [
            "location_id",
            "query_count",
            "top1_failure_count",
            "recall_at_1_percent",
            "mean_positive_rank",
            "maximum_positive_rank",
        ]
    ]
    .head(10)
    .to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda value: f"{value:.2f}"
            ),
            "mean_positive_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nMargin threshold analysis:")
print(
    threshold_df.to_string(
        index=False,
        formatters={
            "margin_threshold": (
                lambda value: f"{value:.3f}"
            ),
            "coverage_percent": (
                lambda value: f"{value:.2f}"
            ),
            "accepted_top1_accuracy_percent": (
                lambda value: (
                    "None"
                    if value is None
                    else f"{value:.2f}"
                )
            ),
        },
    )
)

print("\nTop-1 failures CSV:")
print(failure_csv)

print("\nWorst locations CSV:")
print(worst_locations_csv)

print("\nFailure report JSON:")
print(failure_report_json)

✅ FAILURE ANALYSIS COMPLETE

Total queries:
8000

Top-1 correct:
7566

Top-1 failures:
434

Maximum positive rank:
6

Failure summary by height:
 nominal_height  query_count  top1_failure_count recall_at_1_percent failure_rate_percent mean_positive_rank
            150         2000                 133               93.35                 6.65              1.103
            200         2000                 109               94.55                 5.45              1.083
            250         2000                  91               95.45                 4.55              1.077
            300         2000                 101               94.95                 5.05              1.066

Top 10 worst locations:
location_id  query_count  top1_failure_count recall_at_1_percent mean_positive_rank  maximum_positive_rank
       0071          200                 200                0.00              2.000                      2
       0007          200                 147               26.50       

In [22]:
# ============================================================
# PHASE 2 — CELL 11
# Final report and evidence package
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json
import zipfile
import hashlib

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "reports"
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Input files
# ------------------------------------------------------------

METRICS_CSV = (
    OUTPUT_ROOT
    / "corrected_location_disjoint_metrics.csv"
)

QUERY_RESULTS_CSV = (
    OUTPUT_ROOT
    / "corrected_query_results.csv"
)

FAILURE_CASES_CSV = (
    OUTPUT_ROOT
    / "top1_failure_cases.csv"
)

THRESHOLD_CSV = (
    OUTPUT_ROOT
    / "margin_threshold_analysis.csv"
)

WORST_LOCATIONS_CSV = (
    OUTPUT_ROOT
    / "worst_test_locations.csv"
)

required_files = [
    METRICS_CSV,
    QUERY_RESULTS_CSV,
    FAILURE_CASES_CSV,
    THRESHOLD_CSV,
    WORST_LOCATIONS_CSV,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 2. Load results
# ------------------------------------------------------------

metrics_df = pd.read_csv(
    METRICS_CSV
)

query_df = pd.read_csv(
    QUERY_RESULTS_CSV,
    dtype={
        "query_location_id": str,
        "top1_location_id": str,
    },
)

failure_df = pd.read_csv(
    FAILURE_CASES_CSV,
    dtype={
        "query_location_id": str,
        "top1_location_id": str,
    },
)

threshold_df = pd.read_csv(
    THRESHOLD_CSV
)

worst_locations_df = pd.read_csv(
    WORST_LOCATIONS_CSV,
    dtype={
        "location_id": str,
    },
)

query_df["query_location_id"] = (
    query_df["query_location_id"]
    .str.zfill(4)
)

query_df["top1_location_id"] = (
    query_df["top1_location_id"]
    .str.zfill(4)
)

failure_df["query_location_id"] = (
    failure_df["query_location_id"]
    .str.zfill(4)
)

failure_df["top1_location_id"] = (
    failure_df["top1_location_id"]
    .str.zfill(4)
)

worst_locations_df["location_id"] = (
    worst_locations_df["location_id"]
    .str.zfill(4)
)

# ------------------------------------------------------------
# 3. Confusion-pair analysis
# ------------------------------------------------------------

confusion_pairs_df = (
    failure_df
    .groupby(
        [
            "query_location_id",
            "top1_location_id",
        ]
    )
    .agg(
        confusion_count=(
            "frame_id",
            "count",
        ),
        mean_top1_score=(
            "top1_score",
            "mean",
        ),
        mean_margin=(
            "top1_top2_margin",
            "mean",
        ),
        maximum_positive_rank=(
            "positive_rank",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "confusion_count",
            "query_location_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

confusion_pairs_csv = (
    OUTPUT_ROOT
    / "top1_confusion_pairs.csv"
)

confusion_pairs_df.to_csv(
    confusion_pairs_csv,
    index=False,
)

# Confusions separated by nominal height.
height_confusions_df = (
    failure_df
    .groupby(
        [
            "nominal_height",
            "query_location_id",
            "top1_location_id",
        ]
    )
    .size()
    .reset_index(
        name="confusion_count"
    )
    .sort_values(
        [
            "confusion_count",
            "nominal_height",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

height_confusions_csv = (
    OUTPUT_ROOT
    / "top1_confusion_pairs_by_height.csv"
)

height_confusions_df.to_csv(
    height_confusions_csv,
    index=False,
)

# ------------------------------------------------------------
# 4. Extract headline results
# ------------------------------------------------------------

overall = metrics_df[
    metrics_df["group"] == "overall"
].iloc[0]

height_rows = (
    metrics_df[
        metrics_df["group"] != "overall"
    ]
    .copy()
    .sort_values(
        "nominal_height"
    )
)

best_threshold_row = (
    threshold_df
    .sort_values(
        [
            "accepted_top1_accuracy_percent",
            "coverage_percent",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .iloc[0]
)

top_confusion = (
    confusion_pairs_df.iloc[0]
    if not confusion_pairs_df.empty
    else None
)

# ------------------------------------------------------------
# 5. Create Markdown report
# ------------------------------------------------------------

height_table_lines = []

for _, row in height_rows.iterrows():

    height_table_lines.append(
        "| "
        f"{int(row['nominal_height'])} | "
        f"{int(row['query_count'])} | "
        f"{row['recall_at_1_percent']:.2f}% | "
        f"{row['recall_at_5_percent']:.2f}% | "
        f"{row['recall_at_10_percent']:.2f}% | "
        f"{row['mean_average_precision_percent']:.2f}% |"
    )

height_table = "\n".join(
    height_table_lines
)

worst_location_lines = []

for _, row in (
    worst_locations_df
    .head(10)
    .iterrows()
):

    worst_location_lines.append(
        "| "
        f"{row['location_id']} | "
        f"{int(row['query_count'])} | "
        f"{int(row['top1_failure_count'])} | "
        f"{row['recall_at_1_percent']:.2f}% | "
        f"{row['mean_positive_rank']:.3f} | "
        f"{int(row['maximum_positive_rank'])} |"
    )

worst_location_table = "\n".join(
    worst_location_lines
)

confusion_lines = []

for _, row in (
    confusion_pairs_df
    .head(10)
    .iterrows()
):

    confusion_lines.append(
        "| "
        f"{row['query_location_id']} | "
        f"{row['top1_location_id']} | "
        f"{int(row['confusion_count'])} | "
        f"{row['mean_margin']:.6f} | "
        f"{int(row['maximum_positive_rank'])} |"
    )

confusion_table = "\n".join(
    confusion_lines
)

REPORT_PATH = (
    OUTPUT_ROOT
    / "phase2_corrected_sues200_report.md"
)

report_text = f"""# Phase 2 — Corrected SUES-200 Benchmark

## Provenance

- Model: `AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION`
- Provenance label: `LOCAL_REIMPLEMENTATION`
- Protocol: `PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20`
- Official MobileGeo result: No
- Official SUES-200 protocol: No
- Task measured: drone-to-satellite image retrieval
- Verified geographic UAV pose: No

## Split

- Training locations: 120
- Validation locations: 40
- Test locations: 40
- Test drone queries: 8,000
- Test satellite gallery images: 40
- Train/test location overlap: 0
- Validation/test location overlap: 0

## Overall results

- Recall@1: {overall['recall_at_1_percent']:.2f}%
- Recall@5: {overall['recall_at_5_percent']:.2f}%
- Recall@10: {overall['recall_at_10_percent']:.2f}%
- mAP: {overall['mean_average_precision_percent']:.2f}%
- Mean positive rank: {overall['mean_positive_rank']:.3f}

## Results by nominal height

| Nominal height | Queries | R@1 | R@5 | R@10 | mAP |
|---:|---:|---:|---:|---:|---:|
{height_table}

The height values are dataset folder labels and are not verified
as AGL or MSL altitude.

## Failure analysis

- Top-1 correct queries: 7,566
- Top-1 failures: 434
- Maximum positive rank: 6
- Weakest nominal height: 150
- Location 0071: 0% Recall@1
- Location 0007: 26.50% Recall@1

## Worst test locations

| Location | Queries | Failures | R@1 | Mean positive rank | Maximum rank |
|---|---:|---:|---:|---:|---:|
{worst_location_table}

## Dominant confusion pairs

| Query location | Predicted location | Count | Mean margin | Maximum positive rank |
|---|---|---:|---:|---:|
{confusion_table}

## Confidence-margin result

Using a Top-1/Top-2 margin threshold of
`{best_threshold_row['margin_threshold']:.3f}`:

- Coverage: {best_threshold_row['coverage_percent']:.2f}%
- Accepted Top-1 accuracy:
  {best_threshold_row['accepted_top1_accuracy_percent']:.2f}%
- Ambiguous queries:
  {int(best_threshold_row['ambiguous_query_count'])}

This is an experimental ambiguity threshold. It is not yet a
calibrated probability or an out-of-map rejection model.

## Timing interpretation

The recorded descriptor extraction used batched GPU inference.
It is not a batch-one Jetson Orin Nano deployment result.

Google Drive image decoding and loading dominated the recorded
end-to-end extraction time.

## Reporting restrictions

This result must not be described as:

- an official MobileGeo result
- an official SUES-200 benchmark result
- verified geographic positioning
- verified UAV pose
- deployment latency on Jetson Orin Nano

## Next phase

Run the common retrieval benchmark comparing reproducible
backends under one evaluator.
"""

REPORT_PATH.write_text(
    report_text,
    encoding="utf-8",
)

# ------------------------------------------------------------
# 6. Create summary JSON
# ------------------------------------------------------------

summary = {
    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "phase": (
        "PHASE_2_CORRECTED_SUES200_"
        "LOCATION_DISJOINT_BENCHMARK"
    ),

    "phase_complete": True,

    "model_name": (
        "AdvancedEdgeGeoLPN_"
        "LOCAL_REIMPLEMENTATION"
    ),

    "provenance_label": (
        "LOCAL_REIMPLEMENTATION"
    ),

    "protocol": (
        "PROJECT_DEFINED_"
        "LOCATION_DISJOINT_60_20_20"
    ),

    "official_mobilegeo_result": False,
    "official_sues200_protocol": False,

    "overall_metrics": {
        "recall_at_1_percent": float(
            overall[
                "recall_at_1_percent"
            ]
        ),
        "recall_at_5_percent": float(
            overall[
                "recall_at_5_percent"
            ]
        ),
        "recall_at_10_percent": float(
            overall[
                "recall_at_10_percent"
            ]
        ),
        "mean_average_precision_percent": float(
            overall[
                "mean_average_precision_percent"
            ]
        ),
    },

    "top1_correct_queries": 7566,
    "top1_failure_queries": 434,
    "maximum_positive_rank": 6,

    "recommended_experimental_margin_threshold": float(
        best_threshold_row[
            "margin_threshold"
        ]
    ),

    "threshold_coverage_percent": float(
        best_threshold_row[
            "coverage_percent"
        ]
    ),

    "threshold_accepted_accuracy_percent": float(
        best_threshold_row[
            "accepted_top1_accuracy_percent"
        ]
    ),

    "next_phase": (
        "PHASE_3_COMMON_RETRIEVAL_BENCHMARK"
    ),
}

SUMMARY_JSON = (
    OUTPUT_ROOT
    / "phase2_summary.json"
)

SUMMARY_JSON.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 7. File checksum helper
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()

# ------------------------------------------------------------
# 8. Create evidence package
# ------------------------------------------------------------

ZIP_PATH = (
    REPORT_ROOT
    / "mobilegeo_phase2_corrected_sues200.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

excluded_directories = {
    "embeddings",
}

included_files = []

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    # Add Phase 2 result files.
    for file_path in sorted(
        OUTPUT_ROOT.rglob("*")
    ):

        if not file_path.is_file():
            continue

        relative_parts = (
            file_path
            .relative_to(
                OUTPUT_ROOT
            )
            .parts
        )

        if any(
            directory in excluded_directories
            for directory in relative_parts
        ):
            continue

        archive_path = (
            Path("phase2_results")
            / file_path.relative_to(
                OUTPUT_ROOT
            )
        )

        archive.write(
            file_path,
            arcname=str(
                archive_path
            ),
        )

        included_files.append(
            file_path
        )

    # Add corrected manifests.
    for file_path in sorted(
        MANIFEST_ROOT.rglob("*")
    ):

        if not file_path.is_file():
            continue

        archive_path = (
            Path("phase2_manifests")
            / file_path.relative_to(
                MANIFEST_ROOT
            )
        )

        archive.write(
            file_path,
            arcname=str(
                archive_path
            ),
        )

        included_files.append(
            file_path
        )

# ------------------------------------------------------------
# 9. Validate ZIP and save checksum
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as archive:

    corrupt_file = archive.testzip()
    archived_names = archive.namelist()

if corrupt_file is not None:
    raise RuntimeError(
        f"Corrupt ZIP member: {corrupt_file}"
    )

ZIP_SHA256 = sha256_file(
    ZIP_PATH
)

CHECKSUM_PATH = (
    REPORT_ROOT
    / "mobilegeo_phase2_corrected_sues200.zip.sha256"
)

CHECKSUM_PATH.write_text(
    f"{ZIP_SHA256}  {ZIP_PATH.name}\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# 10. Final output
# ------------------------------------------------------------

print("=" * 76)
print("✅ PHASE 2 CORRECTED SUES-200 BENCHMARK COMPLETE")
print("=" * 76)

print("\nOverall Recall@1:")
print(
    f"{overall['recall_at_1_percent']:.2f}%"
)

print("\nOverall Recall@5:")
print(
    f"{overall['recall_at_5_percent']:.2f}%"
)

print("\nOverall Recall@10:")
print(
    f"{overall['recall_at_10_percent']:.2f}%"
)

print("\nOverall mAP:")
print(
    f"{overall['mean_average_precision_percent']:.2f}%"
)

print("\nTop confusion pairs:")
print(
    confusion_pairs_df[
        [
            "query_location_id",
            "top1_location_id",
            "confusion_count",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nFinal report:")
print(REPORT_PATH)

print("\nSummary JSON:")
print(SUMMARY_JSON)

print("\nEvidence ZIP:")
print(ZIP_PATH)

print("\nFiles inside ZIP:")
print(len(archived_names))

print("\nZIP SHA-256:")
print(ZIP_SHA256)

print("\nEmbedding files remain separately at:")
print(
    OUTPUT_ROOT
    / "embeddings"
)

print("\nNext notebook:")
print(
    PROJECT_ROOT
    / "notebooks"
    / "03_common_retrieval_benchmark.ipynb"
)

✅ PHASE 2 CORRECTED SUES-200 BENCHMARK COMPLETE

Overall Recall@1:
94.58%

Overall Recall@5:
99.99%

Overall Recall@10:
100.00%

Overall mAP:
96.93%

Top confusion pairs:
query_location_id top1_location_id  confusion_count
             0071             0058              200
             0007             0058               91
             0007             0008               56
             0088             0152               17
             0063             0056               14
             0155             0144               12
             0151             0186               11
             0057             0164                5
             0027             0140                4
             0169             0144                4

Final report:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/phase2_corrected_sues200_report.md

Summary JSON:
/content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/phase2_summary.json

Evidence ZIP:
/content/drive/MyDrive/

In [23]:
# ============================================================
# PHASE 3 — CELL 1
# Common retrieval benchmark workspace setup
# ============================================================

from google.colab import drive
drive.mount(
    "/content/drive",
    force_remount=False,
)

from pathlib import Path
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPOSITORIES_ROOT = (
    PROJECT_ROOT
    / "repositories"
)

CHECKPOINTS_ROOT = (
    PROJECT_ROOT
    / "checkpoints"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

for folder in [
    PHASE3_ROOT,
    REPOSITORIES_ROOT,
    CHECKPOINTS_ROOT,
    EMBEDDING_ROOT,
    RANKING_ROOT,
    REPORT_ROOT,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

# Required Phase 2 evidence.
required_files = {
    "test_queries": (
        MANIFEST_ROOT
        / "sues200_test_queries.csv"
    ),
    "test_gallery": (
        MANIFEST_ROOT
        / "sues200_test_gallery.csv"
    ),
    "local_metrics": (
        PHASE2_ROOT
        / "corrected_location_disjoint_metrics.csv"
    ),
    "local_rankings": (
        PHASE2_ROOT
        / "corrected_location_disjoint_rankings.csv"
    ),
    "local_query_embeddings": (
        PHASE2_ROOT
        / "embeddings"
        / "sues200_test_query_embeddings.npz"
    ),
    "local_gallery_embeddings": (
        PHASE2_ROOT
        / "embeddings"
        / "sues200_test_gallery_embeddings.npz"
    ),
}

print("=" * 72)
print("PHASE 3 COMMON BENCHMARK SETUP")
print("=" * 72)

all_available = True

for name, path in required_files.items():

    exists = path.is_file()

    print(
        "✅" if exists else "❌",
        name,
        "→",
        path,
    )

    if not exists:
        all_available = False

if not all_available:
    raise FileNotFoundError(
        "One or more required Phase 2 files are missing."
    )

benchmark_definition = {
    "benchmark_name": (
        "COMMON_SUES200_LOCATION_DISJOINT_BENCHMARK"
    ),

    "protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "query_count": 8000,
    "gallery_count": 40,
    "test_location_count": 40,

    "nominal_heights": [
        150,
        200,
        250,
        300,
    ],

    "required_metrics": [
        "Recall@1",
        "Recall@5",
        "Recall@10",
        "mAP",
    ],

    "candidate_methods": {
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION": {
            "status": "COMPLETED",
            "provenance": "LOCAL_REIMPLEMENTATION",
        },

        "Sample4Geo": {
            "status": "PENDING_RELEASE_AUDIT",
            "provenance": "UNRESOLVED",
        },

        "UltraVPR": {
            "status": "PENDING_RELEASE_AUDIT",
            "provenance": "UNRESOLVED",
        },

        "MobileGeo": {
            "status": (
                "PRECOMPUTED_FEATURES_ONLY_"
                "NOT_DIRECTLY_COMPARABLE_YET"
            ),
            "provenance": (
                "PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION"
            ),
        },
    },

    "comparison_rule": (
        "A method enters the common accuracy table only "
        "after descriptors are generated from the exact same "
        "8,000 test queries and 40 test gallery images."
    ),

    "reporting_restrictions": [
        (
            "Do not compare MobileGeo published MAT metrics "
            "directly with the project-defined split."
        ),
        (
            "Do not label local model results as official MobileGeo."
        ),
        (
            "Do not report retrieval as verified UAV pose."
        ),
    ],
}

definition_path = (
    PHASE3_ROOT
    / "common_benchmark_definition.json"
)

definition_path.write_text(
    json.dumps(
        benchmark_definition,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n✅ Phase 3 workspace ready")

print("\nOutput root:")
print(PHASE3_ROOT)

print("\nBenchmark definition:")
print(definition_path)

print("\nFirst model to add:")
print("Sample4Geo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PHASE 3 COMMON BENCHMARK SETUP
✅ test_queries → /content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_test_queries.csv
✅ test_gallery → /content/drive/MyDrive/mobilegeo_project/manifests/sues200_corrected/sues200_test_gallery.csv
✅ local_metrics → /content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/corrected_location_disjoint_metrics.csv
✅ local_rankings → /content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/corrected_location_disjoint_rankings.csv
✅ local_query_embeddings → /content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/embeddings/sues200_test_query_embeddings.npz
✅ local_gallery_embeddings → /content/drive/MyDrive/mobilegeo_project/results/sues200_corrected/embeddings/sues200_test_gallery_embeddings.npz

✅ Phase 3 workspace ready

Output root:
/content/drive/MyDrive/mobilegeo_project/re

In [24]:
# ============================================================
# PHASE 3 — CELL 2
# Clone and audit the official Sample4Geo repository
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import hashlib
import json
import re

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPOSITORIES_ROOT = (
    PROJECT_ROOT
    / "repositories"
)

SAMPLE4GEO_REPO = (
    REPOSITORIES_ROOT
    / "Sample4Geo"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "sample4geo_audit"
)

REPOSITORY_URL = (
    "https://github.com/Skyy93/Sample4Geo.git"
)

REPOSITORIES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. Command helper
# ------------------------------------------------------------

def run_command(command, cwd=None):

    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:

        print("STDOUT:")
        print(result.stdout)

        print("STDERR:")
        print(result.stderr)

        raise RuntimeError(
            "Command failed:\n"
            + " ".join(command)
        )

    return result.stdout.strip()

# ------------------------------------------------------------
# 3. Clone repository
# ------------------------------------------------------------

if not (
    SAMPLE4GEO_REPO
    / ".git"
).is_dir():

    if SAMPLE4GEO_REPO.exists():
        shutil.rmtree(
            SAMPLE4GEO_REPO
        )

    print(
        "Cloning official Sample4Geo repository..."
    )

    run_command(
        [
            "git",
            "clone",
            REPOSITORY_URL,
            str(SAMPLE4GEO_REPO),
        ]
    )

else:
    print(
        "Sample4Geo repository already exists."
    )

# ------------------------------------------------------------
# 4. Record exact repository state
# ------------------------------------------------------------

commit_hash = run_command(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=SAMPLE4GEO_REPO,
)

short_commit = run_command(
    [
        "git",
        "rev-parse",
        "--short",
        "HEAD",
    ],
    cwd=SAMPLE4GEO_REPO,
)

branch_name = run_command(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=SAMPLE4GEO_REPO,
)

commit_date = run_command(
    [
        "git",
        "show",
        "-s",
        "--format=%cI",
        "HEAD",
    ],
    cwd=SAMPLE4GEO_REPO,
)

commit_subject = run_command(
    [
        "git",
        "show",
        "-s",
        "--format=%s",
        "HEAD",
    ],
    cwd=SAMPLE4GEO_REPO,
)

# ------------------------------------------------------------
# 5. File hash helper
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

# ------------------------------------------------------------
# 6. Inventory repository files
# ------------------------------------------------------------

checkpoint_extensions = {
    ".pth",
    ".pt",
    ".ckpt",
    ".bin",
    ".safetensors",
    ".onnx",
    ".engine",
}

config_extensions = {
    ".yaml",
    ".yml",
    ".json",
    ".toml",
}

inventory_records = []

checkpoint_files = []
python_files = []
config_files = []
license_files = []
readme_files = []
requirement_files = []

for file_path in sorted(
    SAMPLE4GEO_REPO.rglob("*")
):

    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(
        SAMPLE4GEO_REPO
    )

    if ".git" in relative_path.parts:
        continue

    if "__pycache__" in relative_path.parts:
        continue

    suffix = file_path.suffix.lower()
    filename_lower = file_path.name.lower()

    if suffix in checkpoint_extensions:
        category = "checkpoint"
        checkpoint_files.append(
            str(relative_path)
        )

    elif suffix == ".py":
        category = "python_source"
        python_files.append(
            str(relative_path)
        )

    elif suffix in config_extensions:
        category = "configuration"
        config_files.append(
            str(relative_path)
        )

    elif (
        filename_lower.startswith("license")
        or filename_lower == "copying"
    ):
        category = "license"
        license_files.append(
            str(relative_path)
        )

    elif filename_lower.startswith("readme"):
        category = "readme"
        readme_files.append(
            str(relative_path)
        )

    elif (
        "requirement" in filename_lower
        or filename_lower.startswith(
            "environment"
        )
    ):
        category = "environment"
        requirement_files.append(
            str(relative_path)
        )

    else:
        category = "other"

    inventory_records.append({
        "relative_path": str(
            relative_path
        ),
        "filename": file_path.name,
        "extension": suffix,
        "category": category,
        "size_bytes": file_path.stat().st_size,
        "sha256": sha256_file(
            file_path
        ),
    })

# ------------------------------------------------------------
# 7. Search README/source for model-download evidence
# ------------------------------------------------------------

text_extensions = {
    ".md",
    ".txt",
    ".py",
    ".yaml",
    ".yml",
    ".json",
    ".sh",
}

search_patterns = {
    "checkpoint_reference": (
        r"\.(pth|pt|ckpt|safetensors)\b"
        r"|checkpoint"
        r"|weight"
    ),

    "download_link": (
        r"https?://"
        r"|drive\.google"
        r"|huggingface"
        r"|dropbox"
        r"|baidu"
    ),

    "sues200_reference": (
        r"SUES[-_ ]?200"
    ),

    "university1652_reference": (
        r"University[-_ ]?1652"
    ),

    "inference_reference": (
        r"eval"
        r"|test"
        r"|inference"
        r"|extract"
        r"|predict"
    ),
}

evidence_records = []

for file_path in sorted(
    SAMPLE4GEO_REPO.rglob("*")
):

    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(
        SAMPLE4GEO_REPO
    )

    if ".git" in relative_path.parts:
        continue

    if file_path.suffix.lower() not in text_extensions:
        continue

    try:
        lines = file_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

    except Exception:
        continue

    for line_number, line in enumerate(
        lines,
        start=1,
    ):

        for category, pattern in (
            search_patterns.items()
        ):

            if re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            ):

                evidence_records.append({
                    "category": category,
                    "relative_path": str(
                        relative_path
                    ),
                    "line_number": line_number,
                    "line_text": line.strip(),
                })

# ------------------------------------------------------------
# 8. Save CSV files
# ------------------------------------------------------------

import pandas as pd

inventory_df = pd.DataFrame(
    inventory_records
)

evidence_df = pd.DataFrame(
    evidence_records
)

inventory_csv = (
    AUDIT_ROOT
    / "sample4geo_repository_inventory.csv"
)

evidence_csv = (
    AUDIT_ROOT
    / "sample4geo_source_evidence.csv"
)

inventory_df.to_csv(
    inventory_csv,
    index=False,
)

evidence_df.to_csv(
    evidence_csv,
    index=False,
)

# ------------------------------------------------------------
# 9. Initial reproducibility classification
# ------------------------------------------------------------

has_checkpoint_in_repo = (
    len(checkpoint_files) > 0
)

has_python_code = (
    len(python_files) > 0
)

has_license = (
    len(license_files) > 0
)

has_sues_reference = False

if not evidence_df.empty:

    has_sues_reference = bool(
        (
            evidence_df["category"]
            == "sues200_reference"
        ).any()
    )

audit_status = (
    "REQUIRES_CHECKPOINT_AND_PIPELINE_VALIDATION"
)

audit_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "repository": (
        "Skyy93/Sample4Geo"
    ),

    "repository_url": REPOSITORY_URL,

    "local_path": str(
        SAMPLE4GEO_REPO
    ),

    "branch": branch_name,

    "commit_hash": commit_hash,

    "short_commit": short_commit,

    "commit_date": commit_date,

    "commit_subject": commit_subject,

    "python_source_file_count": len(
        python_files
    ),

    "configuration_file_count": len(
        config_files
    ),

    "checkpoint_files_inside_repository": (
        checkpoint_files
    ),

    "checkpoint_found_inside_repository": (
        has_checkpoint_in_repo
    ),

    "license_files": license_files,

    "license_found": has_license,

    "readme_files": readme_files,

    "environment_files": (
        requirement_files
    ),

    "sues200_reference_found": (
        has_sues_reference
    ),

    "audit_status": audit_status,

    "common_benchmark_ready": False,

    "next_action": (
        "Identify the official checkpoint, model "
        "configuration and image preprocessing."
    ),
}

audit_json = (
    AUDIT_ROOT
    / "sample4geo_initial_audit.json"
)

audit_json.write_text(
    json.dumps(
        audit_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 10. Print important evidence
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("✅ SAMPLE4GEO REPOSITORY AUDIT COMPLETE")
print("=" * 72)

print("\nRepository:")
print(SAMPLE4GEO_REPO)

print("\nBranch:")
print(branch_name)

print("\nPinned commit:")
print(commit_hash)

print("\nCommit date:")
print(commit_date)

print("\nPython source files:")
print(len(python_files))

print("\nConfiguration files:")
print(len(config_files))

print("\nCheckpoint files inside repository:")
print(len(checkpoint_files))

for path in checkpoint_files:
    print("-", path)

print("\nLicence files:")
print(len(license_files))

for path in license_files:
    print("-", path)

print("\nEnvironment files:")
print(len(requirement_files))

for path in requirement_files:
    print("-", path)

print("\nSUES-200 references found:")
print(has_sues_reference)

print("\nTop-level repository files:")

for path in sorted(
    SAMPLE4GEO_REPO.iterdir()
):

    if path.name == ".git":
        continue

    print(
        "-",
        path.name,
    )

print("\nRelevant checkpoint/download evidence:")

if evidence_df.empty:

    print("No text evidence found.")

else:

    selected_evidence = evidence_df[
        evidence_df["category"].isin(
            [
                "checkpoint_reference",
                "download_link",
                "sues200_reference",
            ]
        )
    ].head(40)

    for _, row in selected_evidence.iterrows():

        print(
            f"{row['relative_path']}:"
            f"{row['line_number']} — "
            f"{row['line_text']}"
        )

print("\nInventory CSV:")
print(inventory_csv)

print("\nEvidence CSV:")
print(evidence_csv)

print("\nAudit JSON:")
print(audit_json)

print("\nNext action:")
print(
    "Validate the official Sample4Geo checkpoint "
    "and its SUES-200 compatibility."
)

Cloning official Sample4Geo repository...

✅ SAMPLE4GEO REPOSITORY AUDIT COMPLETE

Repository:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo

Branch:
main

Pinned commit:
498c6511478053ddb4a43f24a9f4942fab238f09

Commit date:
2025-08-08T10:18:16+02:00

Python source files:
24

Configuration files:
0

Checkpoint files inside repository:
0

Licence files:
0

Environment files:
1
- requirements.txt

SUES-200 references found:
False

Top-level repository files:
- .gitignore
- README.md
- calc_distance_cvact.py
- calc_distance_cvusa.py
- calc_distance_vigor.py
- eval_cvact.py
- eval_cvusa.py
- eval_university.py
- eval_vigor_cross.py
- eval_vigor_same.py
- images
- requirements.txt
- sample4geo
- train_cvact.py
- train_cvusa.py
- train_university.py
- train_vigor.py

Relevant checkpoint/download evidence:
README.md:6 — [Paper](https://arxiv.org/abs/2303.11851)
README.md:8 — [Weights](https://drive.google.com/drive/folders/1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA?usp=drive_link)


In [25]:
# ============================================================
# PHASE 3 — CELL 3
# Determine Sample4Geo compatibility with our SUES-200 split
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

SAMPLE4GEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "Sample4Geo"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "sample4geo_audit"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

README_PATH = (
    SAMPLE4GEO_REPO
    / "README.md"
)

EVAL_UNIVERSITY_PATH = (
    SAMPLE4GEO_REPO
    / "eval_university.py"
)

if not SAMPLE4GEO_REPO.is_dir():
    raise FileNotFoundError(
        "Sample4Geo repository is missing. "
        "Run Phase 3 Cell 2 first."
    )

if not EVAL_UNIVERSITY_PATH.is_file():
    raise FileNotFoundError(
        EVAL_UNIVERSITY_PATH
    )

# ------------------------------------------------------------
# 1. Read official evaluation configuration
# ------------------------------------------------------------

eval_text = EVAL_UNIVERSITY_PATH.read_text(
    encoding="utf-8",
    errors="replace",
)

readme_text = (
    README_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )
    if README_PATH.is_file()
    else ""
)

def extract_value(pattern, text):
    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE,
    )

    if match:
        return match.group(1)

    return None


model_name = extract_value(
    r"model\s*:\s*str\s*=\s*['\"]([^'\"]+)",
    eval_text,
)

image_size = extract_value(
    r"img_size\s*:\s*int\s*=\s*(\d+)",
    eval_text,
)

checkpoint_relative_path = extract_value(
    r"checkpoint_start\s*=\s*['\"]([^'\"]+)",
    eval_text,
)

dataset_name = extract_value(
    r"dataset\s*:\s*str\s*=\s*['\"]([^'\"]+)",
    eval_text,
)

# ------------------------------------------------------------
# 2. Search repository for dataset support
# ------------------------------------------------------------

supported_dataset_evidence = {
    "University-1652": [],
    "SUES-200": [],
    "CVUSA": [],
    "CVACT": [],
    "VIGOR": [],
}

text_extensions = {
    ".py",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".json",
}

patterns = {
    "University-1652": r"University[-_ ]?1652|U1652",
    "SUES-200": r"SUES[-_ ]?200",
    "CVUSA": r"CVUSA",
    "CVACT": r"CVACT",
    "VIGOR": r"VIGOR",
}

for file_path in SAMPLE4GEO_REPO.rglob("*"):

    if not file_path.is_file():
        continue

    if ".git" in file_path.parts:
        continue

    if file_path.suffix.lower() not in text_extensions:
        continue

    text = file_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for dataset_label, pattern in patterns.items():

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):
            supported_dataset_evidence[
                dataset_label
            ].append(
                str(
                    file_path.relative_to(
                        SAMPLE4GEO_REPO
                    )
                )
            )

for dataset_label in supported_dataset_evidence:

    supported_dataset_evidence[
        dataset_label
    ] = sorted(
        set(
            supported_dataset_evidence[
                dataset_label
            ]
        )
    )

sues_specific_files = (
    supported_dataset_evidence[
        "SUES-200"
    ]
)

sues_specific_pipeline_found = (
    len(sues_specific_files) > 0
)

# ------------------------------------------------------------
# 3. Resolve expected checkpoint path
# ------------------------------------------------------------

if checkpoint_relative_path:

    expected_checkpoint_path = (
        SAMPLE4GEO_REPO
        / checkpoint_relative_path
    )

else:
    expected_checkpoint_path = None

checkpoint_already_available = bool(
    expected_checkpoint_path
    and expected_checkpoint_path.is_file()
)

# ------------------------------------------------------------
# 4. Assign fair benchmark provenance
# ------------------------------------------------------------

if sues_specific_pipeline_found:

    benchmark_status = (
        "SUES200_PIPELINE_REQUIRES_VALIDATION"
    )

    planned_provenance = (
        "OFFICIAL_CHECKPOINT_REPRODUCTION"
    )

else:

    benchmark_status = (
        "ZERO_SHOT_TRANSFER_REQUIRED"
    )

    planned_provenance = (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    )

compatibility_decision = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "Sample4Geo",

    "repository": "Skyy93/Sample4Geo",

    "official_repository_code": True,

    "official_sues200_pipeline_found": (
        sues_specific_pipeline_found
    ),

    "official_configuration": {
        "model_name": model_name,
        "image_size": (
            int(image_size)
            if image_size
            else None
        ),
        "configured_dataset": dataset_name,
        "checkpoint_relative_path": (
            checkpoint_relative_path
        ),
        "expected_checkpoint_path": (
            str(expected_checkpoint_path)
            if expected_checkpoint_path
            else None
        ),
        "checkpoint_already_available": (
            checkpoint_already_available
        ),
    },

    "dataset_evidence": (
        supported_dataset_evidence
    ),

    "common_benchmark_status": (
        benchmark_status
    ),

    "planned_provenance_label": (
        planned_provenance
    ),

    "comparison_protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "reporting_restrictions": [
        (
            "Do not describe zero-shot SUES-200 evaluation "
            "as an official Sample4Geo SUES-200 result."
        ),
        (
            "Use the exact same 8,000 queries and "
            "40 gallery images as the local baseline."
        ),
        (
            "Record the official checkpoint hash."
        ),
        (
            "Do not report retrieval as verified UAV pose."
        ),
    ],

    "next_action": (
        "Download the official University-1652 "
        "Sample4Geo checkpoint."
    ),
}

decision_path = (
    AUDIT_ROOT
    / "sample4geo_sues200_compatibility.json"
)

decision_path.write_text(
    json.dumps(
        compatibility_decision,
        indent=2,
    ),
    encoding="utf-8",
)

# Save expected checkpoint path separately.
checkpoint_path_config = (
    PROJECT_ROOT
    / "configs"
    / "sample4geo_expected_checkpoint_path.txt"
)

if expected_checkpoint_path:

    checkpoint_path_config.write_text(
        str(expected_checkpoint_path),
        encoding="utf-8",
    )

# ------------------------------------------------------------
# 5. Final output
# ------------------------------------------------------------

print("=" * 74)
print("✅ SAMPLE4GEO COMPATIBILITY DECISION COMPLETE")
print("=" * 74)

print("\nOfficial model:")
print(model_name)

print("\nOfficial input size:")
print(image_size)

print("\nOfficial configured dataset:")
print(dataset_name)

print("\nExpected checkpoint:")
print(expected_checkpoint_path)

print("\nCheckpoint already available:")
print(checkpoint_already_available)

print("\nSUES-200-specific files found:")
print(len(sues_specific_files))

for path in sues_specific_files:
    print("-", path)

print("\nCommon benchmark status:")
print(benchmark_status)

print("\nPlanned provenance:")
print(planned_provenance)

print("\nDecision JSON:")
print(decision_path)

print("\nNext action:")
print(
    "Download and validate the official "
    "Sample4Geo University-1652 checkpoint."
)

✅ SAMPLE4GEO COMPATIBILITY DECISION COMPLETE

Official model:
convnext_base.fb_in22k_ft_in1k_384

Official input size:
384

Official configured dataset:
U1652-D2S

Expected checkpoint:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth

Checkpoint already available:
False

SUES-200-specific files found:
0

Common benchmark status:
ZERO_SHOT_TRANSFER_REQUIRED

Planned provenance:
OFFICIAL_SAMPLE4GEO_CHECKPOINT_ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT

Decision JSON:
/content/drive/MyDrive/mobilegeo_project/results/common_benchmark/sample4geo_audit/sample4geo_sues200_compatibility.json

Next action:
Download and validate the official Sample4Geo University-1652 checkpoint.


In [29]:
# ============================================================
# PHASE 3 — CELL 4B
# Extract and validate Sample4Geo checkpoint from pretrained.zip
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import zipfile
import hashlib
import json
import shutil
import torch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

SAMPLE4GEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "Sample4Geo"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "sample4geo_audit"
)

DOWNLOADED_ZIP = Path(
    "/content/sample4geo_official_weights/pretrained.zip"
)

EXPECTED_CHECKPOINT = (
    SAMPLE4GEO_REPO
    / "pretrained"
    / "university"
    / "convnext_base.fb_in22k_ft_in1k_384"
    / "weights_e1_0.9515.pth"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_CHECKPOINT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not DOWNLOADED_ZIP.is_file():
    raise FileNotFoundError(
        "Downloaded archive was not found:\n"
        f"{DOWNLOADED_ZIP}"
    )

print("Downloaded archive:")
print(DOWNLOADED_ZIP)

print("\nArchive size:")
print(
    f"{DOWNLOADED_ZIP.stat().st_size / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# 2. Inspect ZIP contents
# ------------------------------------------------------------

with zipfile.ZipFile(
    DOWNLOADED_ZIP,
    "r",
) as archive:

    all_members = archive.namelist()

    print("\nFiles inside archive:")
    print(len(all_members))

    checkpoint_members = [
        member
        for member in all_members
        if member.lower().endswith(
            (
                ".pth",
                ".pt",
                ".ckpt",
            )
        )
    ]

    print("\nCheckpoint files inside archive:")
    print(len(checkpoint_members))

    for member in checkpoint_members:
        print("-", member)

    # Exact expected filename.
    exact_matches = [
        member
        for member in checkpoint_members
        if Path(member).name
        == "weights_e1_0.9515.pth"
    ]

    # Prefer University-1652 + ConvNeXt-Base path.
    preferred_matches = [
        member
        for member in exact_matches
        if (
            "university" in member.lower()
            and "convnext_base" in member.lower()
        )
    ]

    if preferred_matches:
        selected_member = preferred_matches[0]

    elif len(exact_matches) == 1:
        selected_member = exact_matches[0]

    else:
        likely_matches = [
            member
            for member in checkpoint_members
            if (
                "university" in member.lower()
                and "convnext_base" in member.lower()
            )
        ]

        if len(likely_matches) == 1:
            selected_member = likely_matches[0]

        else:
            raise FileNotFoundError(
                "Could not uniquely identify the required "
                "Sample4Geo University-1652 checkpoint.\n"
                f"Exact filename matches: {len(exact_matches)}\n"
                f"Likely path matches: {len(likely_matches)}"
            )

    print("\nSelected archive member:")
    print(selected_member)

    # Extract only the selected checkpoint to a temporary folder.
    TEMP_EXTRACT_ROOT = Path(
        "/content/sample4geo_selected_checkpoint"
    )

    if TEMP_EXTRACT_ROOT.exists():
        shutil.rmtree(
            TEMP_EXTRACT_ROOT
        )

    TEMP_EXTRACT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    archive.extract(
        selected_member,
        TEMP_EXTRACT_ROOT,
    )

extracted_checkpoint = (
    TEMP_EXTRACT_ROOT
    / selected_member
)

if not extracted_checkpoint.is_file():
    raise FileNotFoundError(
        "Checkpoint extraction failed:\n"
        f"{extracted_checkpoint}"
    )

print("\nExtracted checkpoint:")
print(extracted_checkpoint)

print("\nExtracted size:")
print(
    f"{extracted_checkpoint.stat().st_size / (1024**2):.2f} MB"
)

# ------------------------------------------------------------
# 3. Copy to the repository's expected location
# ------------------------------------------------------------

shutil.copy2(
    extracted_checkpoint,
    EXPECTED_CHECKPOINT,
)

if not EXPECTED_CHECKPOINT.is_file():
    raise FileNotFoundError(
        EXPECTED_CHECKPOINT
    )

print("\n✅ Checkpoint copied to:")
print(EXPECTED_CHECKPOINT)

# Remove temporary extracted folder.
shutil.rmtree(
    TEMP_EXTRACT_ROOT,
    ignore_errors=True,
)

# ------------------------------------------------------------
# 4. Calculate SHA-256
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


checkpoint_sha256 = sha256_file(
    EXPECTED_CHECKPOINT
)

checkpoint_size_bytes = (
    EXPECTED_CHECKPOINT.stat().st_size
)

checkpoint_size_mb = (
    checkpoint_size_bytes
    / (1024 * 1024)
)

# ------------------------------------------------------------
# 5. Load checkpoint
# ------------------------------------------------------------

safe_loading_error = None

try:
    checkpoint = torch.load(
        EXPECTED_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        EXPECTED_CHECKPOINT,
        map_location="cpu",
    )

except Exception as error:
    safe_loading_error = repr(error)

    print("\nWeights-only loading failed:")
    print(safe_loading_error)

    checkpoint = torch.load(
        EXPECTED_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

print("\nCheckpoint object type:")
print(type(checkpoint))

if not isinstance(checkpoint, dict):
    raise RuntimeError(
        "Unsupported checkpoint type:\n"
        f"{type(checkpoint)}"
    )

container_keys = [
    str(key)
    for key in checkpoint.keys()
]

# ------------------------------------------------------------
# 6. Resolve model state dictionary
# ------------------------------------------------------------

if (
    "state_dict" in checkpoint
    and isinstance(
        checkpoint["state_dict"],
        dict,
    )
):
    state_dict = checkpoint[
        "state_dict"
    ]

    state_dict_source = (
        "checkpoint['state_dict']"
    )

elif (
    "model_state_dict" in checkpoint
    and isinstance(
        checkpoint["model_state_dict"],
        dict,
    )
):
    state_dict = checkpoint[
        "model_state_dict"
    ]

    state_dict_source = (
        "checkpoint['model_state_dict']"
    )

elif (
    "model" in checkpoint
    and isinstance(
        checkpoint["model"],
        dict,
    )
):
    state_dict = checkpoint[
        "model"
    ]

    state_dict_source = (
        "checkpoint['model']"
    )

elif all(
    torch.is_tensor(value)
    for value in checkpoint.values()
):
    state_dict = checkpoint
    state_dict_source = "checkpoint_root"

else:
    print("\nCheckpoint container keys:")

    for key in container_keys:
        print("-", key)

    raise RuntimeError(
        "No recognizable model state dictionary found."
    )

# ------------------------------------------------------------
# 7. Clean state-dictionary prefixes
# ------------------------------------------------------------

clean_state_dict = {}

for key, value in state_dict.items():

    clean_key = str(key)

    if clean_key.startswith("module."):
        clean_key = clean_key[7:]

    clean_state_dict[
        clean_key
    ] = value

tensor_items = [
    (key, tensor)
    for key, tensor in clean_state_dict.items()
    if torch.is_tensor(tensor)
]

if not tensor_items:
    raise RuntimeError(
        "No tensor parameters were found."
    )

total_tensor_values = sum(
    tensor.numel()
    for _, tensor in tensor_items
)

prefix_counts = {}

for key, _ in tensor_items:

    prefix = key.split(".")[0]

    prefix_counts[prefix] = (
        prefix_counts.get(
            prefix,
            0,
        )
        + 1
    )

# ------------------------------------------------------------
# 8. Save checkpoint path and audit
# ------------------------------------------------------------

checkpoint_config_path = (
    PROJECT_ROOT
    / "configs"
    / "sample4geo_checkpoint_path.txt"
)

checkpoint_config_path.write_text(
    str(EXPECTED_CHECKPOINT),
    encoding="utf-8",
)

audit_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "Sample4Geo",

    "repository": "Skyy93/Sample4Geo",

    "archive_path": str(
        DOWNLOADED_ZIP
    ),

    "archive_member": (
        selected_member
    ),

    "checkpoint_origin": (
        "OFFICIAL_SAMPLE4GEO_GOOGLE_DRIVE_ARCHIVE"
    ),

    "checkpoint_path": str(
        EXPECTED_CHECKPOINT
    ),

    "checkpoint_size_bytes": int(
        checkpoint_size_bytes
    ),

    "checkpoint_size_mb": float(
        checkpoint_size_mb
    ),

    "checkpoint_sha256": (
        checkpoint_sha256
    ),

    "checkpoint_object_type": str(
        type(checkpoint)
    ),

    "checkpoint_container_keys": (
        container_keys
    ),

    "state_dict_source": (
        state_dict_source
    ),

    "tensor_entry_count": len(
        tensor_items
    ),

    "total_tensor_values": int(
        total_tensor_values
    ),

    "parameter_prefix_counts": (
        prefix_counts
    ),

    "official_model_name": (
        "convnext_base.fb_in22k_ft_in1k_384"
    ),

    "official_input_size": 384,

    "official_training_dataset": (
        "University-1652"
    ),

    "common_benchmark_usage": (
        "ZERO_SHOT_TRANSFER_TO_"
        "PROJECT_DEFINED_SUES200_SPLIT"
    ),

    "provenance_label": (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "safe_loading_error": (
        safe_loading_error
    ),

    "model_loading_validated": False,
}

audit_json = (
    AUDIT_ROOT
    / "sample4geo_checkpoint_audit.json"
)

audit_json.write_text(
    json.dumps(
        audit_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 9. Final result
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("✅ SAMPLE4GEO OFFICIAL CHECKPOINT VALIDATED")
print("=" * 76)

print("\nCheckpoint:")
print(EXPECTED_CHECKPOINT)

print("\nCheckpoint size:")
print(f"{checkpoint_size_mb:.2f} MB")

print("\nCheckpoint SHA-256:")
print(checkpoint_sha256)

print("\nState dictionary source:")
print(state_dict_source)

print("\nTensor entries:")
print(len(tensor_items))

print("\nTotal tensor values:")
print(f"{total_tensor_values:,}")

print("\nParameter prefix counts:")

for prefix, count in sorted(
    prefix_counts.items()
):
    print(
        f"- {prefix}: {count}"
    )

print("\nFirst 30 parameter keys:")

for key, tensor in tensor_items[:30]:
    print(
        f"- {key}: {tuple(tensor.shape)}"
    )

print("\nSaved checkpoint path:")
print(checkpoint_config_path)

print("\nAudit JSON:")
print(audit_json)

print("\nNext action:")
print(
    "Construct the Sample4Geo ConvNeXt-Base model "
    "and test strict checkpoint loading."
)

Downloaded archive:
/content/sample4geo_official_weights/pretrained.zip

Archive size:
1.51 GB

Files inside archive:
26

Checkpoint files inside archive:
5
- pretrained/cvact/convnext_base.fb_in22k_ft_in1k_384/weights_e36_90.8149.pth
- pretrained/cvusa/convnext_base.fb_in22k_ft_in1k_384/weights_e40_98.6830.pth
- pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth
- pretrained/vigor_cross/convnext_base.fb_in22k_ft_in1k_384/weights_e40_0.6109.pth
- pretrained/vigor_same/convnext_base.fb_in22k_ft_in1k_384/weights_e40_0.7786.pth

Selected archive member:
pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth

Extracted checkpoint:
/content/sample4geo_selected_checkpoint/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth

Extracted size:
334.17 MB

✅ Checkpoint copied to:
/content/drive/MyDrive/mobilegeo_project/repositories/Sample4Geo/pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pt

In [30]:
# ============================================================
# PHASE 3 — CELL 5
# Construct Sample4Geo model, strict-load checkpoint,
# reproduce official validation preprocessing,
# and run a descriptor smoke test
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2

# ------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

SAMPLE4GEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "Sample4Geo"
)

CHECKPOINT_PATH = (
    SAMPLE4GEO_REPO
    / "pretrained"
    / "university"
    / "convnext_base.fb_in22k_ft_in1k_384"
    / "weights_e1_0.9515.pth"
)

TEST_QUERY_MANIFEST = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
    / "sues200_test_queries.csv"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "sample4geo_audit"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = (
    "convnext_base.fb_in22k_ft_in1k_384"
)

IMAGE_SIZE = 384

SAMPLE4GEO_DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if not SAMPLE4GEO_REPO.is_dir():
    raise FileNotFoundError(
        SAMPLE4GEO_REPO
    )

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        CHECKPOINT_PATH
    )

if not TEST_QUERY_MANIFEST.is_file():
    raise FileNotFoundError(
        TEST_QUERY_MANIFEST
    )

# ------------------------------------------------------------
# 2. Ensure timm is installed
# ------------------------------------------------------------

try:
    import timm

except ImportError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "timm",
        ],
        check=True,
    )

    import timm

print("PyTorch version:")
print(torch.__version__)

print("\ntimm version:")
print(timm.__version__)

print("\nDevice:")
print(SAMPLE4GEO_DEVICE)

# ------------------------------------------------------------
# 3. Import official repository model class
# ------------------------------------------------------------

repo_string = str(
    SAMPLE4GEO_REPO
)

if repo_string not in sys.path:
    sys.path.insert(
        0,
        repo_string,
    )

from sample4geo.model import TimmModel

# ------------------------------------------------------------
# 4. Construct official architecture
# ------------------------------------------------------------
# pretrained=False avoids downloading an unrelated ImageNet
# initialization. The complete official Sample4Geo checkpoint
# will be loaded strictly immediately afterward.

sample4geo_model = TimmModel(
    model_name=MODEL_NAME,
    pretrained=False,
    img_size=IMAGE_SIZE,
)

model_key_count_before_load = len(
    sample4geo_model.state_dict()
)

model_parameter_count = sum(
    parameter.numel()
    for parameter in sample4geo_model.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in sample4geo_model.parameters()
    if parameter.requires_grad
)

print("\nConstructed model:")
print(type(sample4geo_model))

print("\nModel state-dictionary entries:")
print(model_key_count_before_load)

print("\nModel parameters:")
print(f"{model_parameter_count:,}")

# ------------------------------------------------------------
# 5. Load official checkpoint
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

if not isinstance(
    checkpoint,
    dict,
):
    raise RuntimeError(
        "Checkpoint is not a state dictionary."
    )

checkpoint_key_count = len(
    checkpoint
)

print("\nCheckpoint entries:")
print(checkpoint_key_count)

# This raises an exception if any key or tensor shape differs.
strict_load_result = (
    sample4geo_model.load_state_dict(
        checkpoint,
        strict=True,
    )
)

missing_keys = list(
    strict_load_result.missing_keys
)

unexpected_keys = list(
    strict_load_result.unexpected_keys
)

strict_load_success = (
    len(missing_keys) == 0
    and len(unexpected_keys) == 0
)

if not strict_load_success:
    raise RuntimeError(
        "Strict checkpoint loading did not succeed."
    )

sample4geo_model = (
    sample4geo_model
    .to(SAMPLE4GEO_DEVICE)
    .eval()
)

# Keep the official logit scale for provenance.
sample4geo_logit_scale_log = float(
    sample4geo_model
    .logit_scale
    .detach()
    .cpu()
    .item()
)

sample4geo_logit_scale_exp = float(
    sample4geo_model
    .logit_scale
    .detach()
    .exp()
    .cpu()
    .item()
)

# ------------------------------------------------------------
# 6. Resolve official model normalization
# ------------------------------------------------------------

sample4geo_data_config = (
    sample4geo_model.get_config()
)

mean = np.asarray(
    sample4geo_data_config["mean"],
    dtype=np.float32,
)

std = np.asarray(
    sample4geo_data_config["std"],
    dtype=np.float32,
)

if mean.shape != (3,):
    raise RuntimeError(
        f"Unexpected mean shape: {mean.shape}"
    )

if std.shape != (3,):
    raise RuntimeError(
        f"Unexpected standard-deviation shape: {std.shape}"
    )

if not hasattr(
    cv2,
    "INTER_LINEAR_EXACT",
):
    raise RuntimeError(
        "This OpenCV build does not provide "
        "INTER_LINEAR_EXACT."
    )

# ------------------------------------------------------------
# 7. Exact evaluation preprocessing
# ------------------------------------------------------------
# Equivalent to the repository's University-1652 validation:
# BGR -> RGB
# Resize to 384x384 with cv2.INTER_LINEAR_EXACT
# Convert to [0,1]
# Normalize with resolved model mean/std
# HWC -> CHW tensor

def sample4geo_preprocess(
    image_path,
):
    image_path = Path(
        image_path
    )

    image_bgr = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )

    if image_bgr is None:
        raise RuntimeError(
            f"Could not read image: {image_path}"
        )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )

    image_rgb = cv2.resize(
        image_rgb,
        (
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        interpolation=(
            cv2.INTER_LINEAR_EXACT
        ),
    )

    image_float = (
        image_rgb.astype(
            np.float32
        )
        / 255.0
    )

    image_float = (
        image_float
        - mean.reshape(
            1,
            1,
            3,
        )
    ) / std.reshape(
        1,
        1,
        3,
    )

    image_tensor = (
        torch.from_numpy(
            image_float
        )
        .permute(
            2,
            0,
            1,
        )
        .contiguous()
    )

    return image_tensor

# ------------------------------------------------------------
# 8. Select one real SUES-200 test query
# ------------------------------------------------------------

query_manifest_df = pd.read_csv(
    TEST_QUERY_MANIFEST
)

path_candidates = [
    "absolute_path",
    "image_path",
    "query_path",
    "path",
    "filepath",
    "file_path",
]

query_path_column = next(
    (
        column
        for column in path_candidates
        if column
        in query_manifest_df.columns
    ),
    None,
)

if query_path_column is None:
    print(
        "\nManifest columns:"
    )

    print(
        query_manifest_df.columns.tolist()
    )

    raise KeyError(
        "No image-path column was identified "
        "in the test-query manifest."
    )

smoke_image_path = Path(
    str(
        query_manifest_df.iloc[0][
            query_path_column
        ]
    )
)

if not smoke_image_path.is_file():
    raise FileNotFoundError(
        smoke_image_path
    )

smoke_tensor = (
    sample4geo_preprocess(
        smoke_image_path
    )
    .unsqueeze(0)
    .to(
        SAMPLE4GEO_DEVICE,
        non_blocking=True,
    )
)

# ------------------------------------------------------------
# 9. Descriptor smoke test
# ------------------------------------------------------------

with torch.inference_mode():

    # CUDA warm-up.
    for _ in range(3):
        _ = sample4geo_model(
            smoke_tensor
        )

    if SAMPLE4GEO_DEVICE.type == "cuda":
        torch.cuda.synchronize()

    inference_times_ms = []

    descriptor_raw = None

    for _ in range(10):

        start_time = time.perf_counter()

        descriptor_raw = (
            sample4geo_model(
                smoke_tensor
            )
        )

        if SAMPLE4GEO_DEVICE.type == "cuda":
            torch.cuda.synchronize()

        elapsed_ms = (
            time.perf_counter()
            - start_time
        ) * 1000.0

        inference_times_ms.append(
            elapsed_ms
        )

    descriptor_normalized = (
        F.normalize(
            descriptor_raw.float(),
            p=2,
            dim=1,
        )
    )

descriptor_shape = tuple(
    descriptor_raw.shape
)

descriptor_dimension = int(
    descriptor_raw.shape[1]
)

descriptor_raw_norm = float(
    descriptor_raw
    .float()
    .norm(
        p=2,
        dim=1,
    )
    .item()
)

descriptor_normalized_norm = float(
    descriptor_normalized
    .norm(
        p=2,
        dim=1,
    )
    .item()
)

descriptor_finite = bool(
    torch.isfinite(
        descriptor_normalized
    ).all().item()
)

median_batch1_inference_ms = float(
    np.median(
        inference_times_ms
    )
)

mean_batch1_inference_ms = float(
    np.mean(
        inference_times_ms
    )
)

if descriptor_shape[0] != 1:
    raise RuntimeError(
        f"Unexpected batch dimension: {descriptor_shape}"
    )

if descriptor_dimension != 1024:
    raise RuntimeError(
        "Unexpected Sample4Geo descriptor dimension: "
        f"{descriptor_dimension}"
    )

if not descriptor_finite:
    raise RuntimeError(
        "Descriptor contains NaN or infinity."
    )

if not np.isclose(
    descriptor_normalized_norm,
    1.0,
    atol=1e-5,
):
    raise RuntimeError(
        "L2-normalized descriptor norm is not 1."
    )

SAMPLE4GEO_DESCRIPTOR_DIM = (
    descriptor_dimension
)

# ------------------------------------------------------------
# 10. JSON-safe conversion helper
# ------------------------------------------------------------

def json_safe(
    value,
):
    if isinstance(
        value,
        Path,
    ):
        return str(value)

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        tuple,
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(
        value,
        list,
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    return value

# ------------------------------------------------------------
# 11. Save strict-loading audit
# ------------------------------------------------------------

model_audit = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "Sample4Geo",

    "repository_model_class": (
        "sample4geo.model.TimmModel"
    ),

    "model_name": MODEL_NAME,

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_entries": int(
        checkpoint_key_count
    ),

    "model_state_dict_entries": int(
        model_key_count_before_load
    ),

    "strict_loading_requested": True,

    "strict_loading_successful": (
        strict_load_success
    ),

    "missing_keys": missing_keys,

    "unexpected_keys": (
        unexpected_keys
    ),

    "parameter_count": int(
        model_parameter_count
    ),

    "trainable_parameter_count": int(
        trainable_parameter_count
    ),

    "descriptor_dimension": int(
        descriptor_dimension
    ),

    "image_size": [
        IMAGE_SIZE,
        IMAGE_SIZE,
    ],

    "data_config": json_safe(
        sample4geo_data_config
    ),

    "validation_preprocessing": {
        "color_conversion": (
            "OpenCV BGR_TO_RGB"
        ),
        "resize": [
            IMAGE_SIZE,
            IMAGE_SIZE,
        ],
        "interpolation": (
            "cv2.INTER_LINEAR_EXACT"
        ),
        "pixel_scaling": (
            "uint8_to_float32_divide_by_255"
        ),
        "mean": mean.tolist(),
        "std": std.tolist(),
        "l2_normalize_descriptor": True,
    },

    "logit_scale_log": (
        sample4geo_logit_scale_log
    ),

    "logit_scale_exp": (
        sample4geo_logit_scale_exp
    ),

    "smoke_test": {
        "image_path": str(
            smoke_image_path
        ),
        "input_shape": list(
            smoke_tensor.shape
        ),
        "descriptor_shape": list(
            descriptor_shape
        ),
        "raw_descriptor_norm": (
            descriptor_raw_norm
        ),
        "normalized_descriptor_norm": (
            descriptor_normalized_norm
        ),
        "finite": descriptor_finite,
        "median_batch1_inference_ms": (
            median_batch1_inference_ms
        ),
        "mean_batch1_inference_ms": (
            mean_batch1_inference_ms
        ),
        "timing_label": (
            "COLAB_RUNTIME_BATCH1_SMOKE_TEST_"
            "NOT_DEPLOYMENT_LATENCY"
        ),
    },

    "provenance_label": (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "official_sues200_result": False,

    "common_benchmark_ready": True,
}

MODEL_AUDIT_JSON = (
    AUDIT_ROOT
    / "sample4geo_strict_model_load_audit.json"
)

MODEL_AUDIT_JSON.write_text(
    json.dumps(
        model_audit,
        indent=2,
    ),
    encoding="utf-8",
)

# Update the previous checkpoint audit when available.
previous_audit_path = (
    AUDIT_ROOT
    / "sample4geo_checkpoint_audit.json"
)

if previous_audit_path.is_file():

    previous_audit = json.loads(
        previous_audit_path.read_text(
            encoding="utf-8"
        )
    )

    previous_audit[
        "model_loading_validated"
    ] = True

    previous_audit[
        "strict_model_loading_successful"
    ] = True

    previous_audit[
        "descriptor_dimension"
    ] = descriptor_dimension

    previous_audit[
        "strict_model_load_audit"
    ] = str(
        MODEL_AUDIT_JSON
    )

    previous_audit_path.write_text(
        json.dumps(
            previous_audit,
            indent=2,
        ),
        encoding="utf-8",
    )

# ------------------------------------------------------------
# 12. Final output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("✅ SAMPLE4GEO STRICT MODEL LOAD COMPLETE")
print("=" * 76)

print("\nOfficial repository class:")
print(
    "sample4geo.model.TimmModel"
)

print("\nModel:")
print(MODEL_NAME)

print("\nStrict loading successful:")
print(strict_load_success)

print("\nMissing keys:")
print(missing_keys)

print("\nUnexpected keys:")
print(unexpected_keys)

print("\nModel parameters:")
print(f"{model_parameter_count:,}")

print("\nInput shape:")
print(tuple(smoke_tensor.shape))

print("\nDescriptor shape:")
print(descriptor_shape)

print("\nRaw descriptor norm:")
print(f"{descriptor_raw_norm:.6f}")

print("\nNormalized descriptor norm:")
print(
    f"{descriptor_normalized_norm:.6f}"
)

print("\nDescriptor finite:")
print(descriptor_finite)

print("\nResolved mean:")
print(mean.tolist())

print("\nResolved std:")
print(std.tolist())

print("\nLearned logit scale exp:")
print(
    f"{sample4geo_logit_scale_exp:.6f}"
)

print("\nMedian batch-one smoke inference:")
print(
    f"{median_batch1_inference_ms:.3f} ms"
)

print("\nTiming interpretation:")
print(
    "COLAB_RUNTIME_BATCH1_SMOKE_TEST_"
    "NOT_DEPLOYMENT_LATENCY"
)

print("\nAudit JSON:")
print(MODEL_AUDIT_JSON)

print("\nNext action:")
print(
    "Extract Sample4Geo descriptors for the same "
    "8,000 queries and 40 gallery images."
)

PyTorch version:
2.11.0+cu128

timm version:
1.0.28

Device:
cuda

Constructed model:
<class 'sample4geo.model.TimmModel'>

Model state-dictionary entries:
343

Model parameters:
87,566,465

Checkpoint entries:
343

✅ SAMPLE4GEO STRICT MODEL LOAD COMPLETE

Official repository class:
sample4geo.model.TimmModel

Model:
convnext_base.fb_in22k_ft_in1k_384

Strict loading successful:
True

Missing keys:
[]

Unexpected keys:
[]

Model parameters:
87,566,465

Input shape:
(1, 3, 384, 384)

Descriptor shape:
(1, 1024)

Raw descriptor norm:
28.954470

Normalized descriptor norm:
1.000000

Descriptor finite:
True

Resolved mean:
[0.48500001430511475, 0.4560000002384186, 0.4059999883174896]

Resolved std:
[0.2290000021457672, 0.2240000069141388, 0.22499999403953552]

Learned logit scale exp:
14.298084

Median batch-one smoke inference:
39.561 ms

Timing interpretation:
COLAB_RUNTIME_BATCH1_SMOKE_TEST_NOT_DEPLOYMENT_LATENCY

Audit JSON:
/content/drive/MyDrive/mobilegeo_project/results/common_bench

In [31]:
# ============================================================
# PHASE 3 — CELL 6
# Extract Sample4Geo descriptors for the common SUES-200 split
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import sys
import time
import json

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

SAMPLE4GEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "Sample4Geo"
)

CHECKPOINT_PATH = (
    SAMPLE4GEO_REPO
    / "pretrained"
    / "university"
    / "convnext_base.fb_in22k_ft_in1k_384"
    / "weights_e1_0.9515.pth"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

QUERY_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "sues200_test_queries.csv"
)

GALLERY_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "sues200_test_gallery.csv"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

EMBEDDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

QUERY_OUTPUT_PATH = (
    EMBEDDING_ROOT
    / "sample4geo_sues200_test_query_embeddings.npz"
)

GALLERY_OUTPUT_PATH = (
    EMBEDDING_ROOT
    / "sample4geo_sues200_test_gallery_embeddings.npz"
)

EXTRACTION_REPORT_PATH = (
    REPORT_ROOT
    / "sample4geo_descriptor_extraction_report.json"
)

MODEL_NAME = (
    "convnext_base.fb_in22k_ft_in1k_384"
)

IMAGE_SIZE = 384
BATCH_SIZE = 16
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

for required_path in [
    SAMPLE4GEO_REPO,
    CHECKPOINT_PATH,
    QUERY_MANIFEST_PATH,
    GALLERY_MANIFEST_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

print("Device:")
print(DEVICE)

print("\nBatch size:")
print(BATCH_SIZE)

# ------------------------------------------------------------
# 2. Import official Sample4Geo model class
# ------------------------------------------------------------

repo_string = str(
    SAMPLE4GEO_REPO
)

if repo_string not in sys.path:
    sys.path.insert(
        0,
        repo_string,
    )

from sample4geo.model import TimmModel

# ------------------------------------------------------------
# 3. Construct and strict-load official checkpoint
# ------------------------------------------------------------

sample4geo_model = TimmModel(
    model_name=MODEL_NAME,
    pretrained=False,
    img_size=IMAGE_SIZE,
)

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

load_result = (
    sample4geo_model.load_state_dict(
        checkpoint,
        strict=True,
    )
)

if (
    load_result.missing_keys
    or load_result.unexpected_keys
):
    raise RuntimeError(
        "Strict checkpoint loading failed."
    )

sample4geo_model = (
    sample4geo_model
    .to(DEVICE)
    .eval()
)

data_config = (
    sample4geo_model.get_config()
)

NORMALIZATION_MEAN = np.asarray(
    data_config["mean"],
    dtype=np.float32,
)

NORMALIZATION_STD = np.asarray(
    data_config["std"],
    dtype=np.float32,
)

print("\nStrict checkpoint loading:")
print("Successful")

print("\nNormalization mean:")
print(NORMALIZATION_MEAN.tolist())

print("\nNormalization std:")
print(NORMALIZATION_STD.tolist())

# ------------------------------------------------------------
# 4. Manifest helpers
# ------------------------------------------------------------

def find_column(
    dataframe,
    candidates,
    purpose,
):
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    raise KeyError(
        f"Could not find {purpose} column.\n"
        f"Available columns: "
        f"{dataframe.columns.tolist()}"
    )


query_df = pd.read_csv(
    QUERY_MANIFEST_PATH,
    dtype=str,
)

gallery_df = pd.read_csv(
    GALLERY_MANIFEST_PATH,
    dtype=str,
)

query_path_column = find_column(
    query_df,
    [
        "absolute_path",
        "image_path",
        "query_path",
        "path",
        "filepath",
        "file_path",
    ],
    "query image path",
)

gallery_path_column = find_column(
    gallery_df,
    [
        "absolute_path",
        "image_path",
        "gallery_path",
        "path",
        "filepath",
        "file_path",
    ],
    "gallery image path",
)

query_frame_column = find_column(
    query_df,
    [
        "frame_id",
        "query_id",
        "image_id",
        "id",
    ],
    "query frame ID",
)

query_location_column = find_column(
    query_df,
    [
        "location_id",
        "query_location_id",
        "class_id",
    ],
    "query location ID",
)

query_height_column = find_column(
    query_df,
    [
        "nominal_height",
        "height",
        "altitude",
    ],
    "query nominal height",
)

positive_tile_column = find_column(
    query_df,
    [
        "positive_tile_id",
        "tile_id",
        "gallery_tile_id",
    ],
    "positive gallery tile ID",
)

gallery_tile_column = find_column(
    gallery_df,
    [
        "tile_id",
        "gallery_tile_id",
        "image_id",
        "id",
    ],
    "gallery tile ID",
)

gallery_location_column = find_column(
    gallery_df,
    [
        "location_id",
        "gallery_location_id",
        "class_id",
    ],
    "gallery location ID",
)

if len(query_df) != 8000:
    raise RuntimeError(
        f"Expected 8,000 queries, found {len(query_df)}."
    )

if len(gallery_df) != 40:
    raise RuntimeError(
        f"Expected 40 gallery images, found {len(gallery_df)}."
    )

print("\nQuery images:")
print(len(query_df))

print("\nGallery images:")
print(len(gallery_df))

# ------------------------------------------------------------
# 5. Dataset
# ------------------------------------------------------------

class Sample4GeoImageDataset(Dataset):

    def __init__(
        self,
        dataframe,
        path_column,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.path_column = path_column

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(
        self,
        index,
    ):
        image_path = Path(
            self.dataframe.iloc[index][
                self.path_column
            ]
        )

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            raise RuntimeError(
                f"Could not read image: {image_path}"
            )

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        image_rgb = cv2.resize(
            image_rgb,
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            interpolation=(
                cv2.INTER_LINEAR_EXACT
            ),
        )

        image_float = (
            image_rgb.astype(
                np.float32
            )
            / 255.0
        )

        image_float = (
            image_float
            - NORMALIZATION_MEAN.reshape(
                1,
                1,
                3,
            )
        ) / NORMALIZATION_STD.reshape(
            1,
            1,
            3,
        )

        image_tensor = (
            torch.from_numpy(
                image_float
            )
            .permute(
                2,
                0,
                1,
            )
            .contiguous()
        )

        return {
            "image": image_tensor,
            "index": index,
        }

# ------------------------------------------------------------
# 6. Extraction function
# ------------------------------------------------------------

def extract_descriptors(
    dataframe,
    path_column,
    dataset_label,
):
    dataset = Sample4GeoImageDataset(
        dataframe=dataframe,
        path_column=path_column,
    )

    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
        persistent_workers=False,
    )

    descriptor_batches = []
    output_indices = []

    model_time_seconds = 0.0

    overall_start = time.perf_counter()

    with torch.inference_mode():

        for batch_number, batch in enumerate(
            dataloader,
            start=1,
        ):
            images = batch[
                "image"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            indices = batch[
                "index"
            ].numpy()

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            model_start = time.perf_counter()

            raw_descriptors = (
                sample4geo_model(
                    images
                )
            )

            descriptors = F.normalize(
                raw_descriptors.float(),
                p=2,
                dim=1,
            )

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            model_time_seconds += (
                time.perf_counter()
                - model_start
            )

            descriptor_batches.append(
                descriptors
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            output_indices.extend(
                indices.tolist()
            )

            processed = min(
                batch_number * BATCH_SIZE,
                len(dataset),
            )

            if (
                batch_number == 1
                or batch_number % 50 == 0
                or processed == len(dataset)
            ):
                print(
                    f"{dataset_label}: "
                    f"{processed}/{len(dataset)}"
                )

    end_to_end_seconds = (
        time.perf_counter()
        - overall_start
    )

    descriptors = np.concatenate(
        descriptor_batches,
        axis=0,
    )

    output_indices = np.asarray(
        output_indices,
        dtype=np.int64,
    )

    # Restore exact manifest ordering defensively.
    sort_order = np.argsort(
        output_indices
    )

    descriptors = descriptors[
        sort_order
    ]

    output_indices = output_indices[
        sort_order
    ]

    expected_indices = np.arange(
        len(dataframe),
        dtype=np.int64,
    )

    if not np.array_equal(
        output_indices,
        expected_indices,
    ):
        raise RuntimeError(
            f"{dataset_label} output order is invalid."
        )

    if descriptors.shape[0] != len(dataframe):
        raise RuntimeError(
            f"{dataset_label} descriptor count mismatch."
        )

    if descriptors.shape[1] != 1024:
        raise RuntimeError(
            f"Unexpected descriptor dimension: "
            f"{descriptors.shape}"
        )

    if not np.isfinite(
        descriptors
    ).all():
        raise RuntimeError(
            f"{dataset_label} descriptors contain "
            "NaN or infinity."
        )

    descriptor_norms = np.linalg.norm(
        descriptors,
        axis=1,
    )

    if not np.allclose(
        descriptor_norms,
        1.0,
        atol=1e-4,
    ):
        raise RuntimeError(
            f"{dataset_label} descriptors are not "
            "properly L2-normalized."
        )

    timing = {
        "image_count": int(
            len(dataframe)
        ),
        "batch_size": int(
            BATCH_SIZE
        ),
        "model_inference_seconds": float(
            model_time_seconds
        ),
        "model_inference_ms_per_image": float(
            1000.0
            * model_time_seconds
            / len(dataframe)
        ),
        "end_to_end_seconds": float(
            end_to_end_seconds
        ),
        "end_to_end_ms_per_image": float(
            1000.0
            * end_to_end_seconds
            / len(dataframe)
        ),
    }

    return descriptors, timing

# ------------------------------------------------------------
# 7. CUDA warm-up
# ------------------------------------------------------------

if DEVICE.type == "cuda":

    warmup_tensor = torch.zeros(
        (
            1,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    with torch.inference_mode():
        for _ in range(3):
            _ = sample4geo_model(
                warmup_tensor
            )

    torch.cuda.synchronize()

    del warmup_tensor

# ------------------------------------------------------------
# 8. Extract gallery descriptors
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXTRACTING SAMPLE4GEO GALLERY DESCRIPTORS")
print("=" * 72)

gallery_descriptors, gallery_timing = (
    extract_descriptors(
        dataframe=gallery_df,
        path_column=gallery_path_column,
        dataset_label="Gallery",
    )
)

# ------------------------------------------------------------
# 9. Extract query descriptors
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXTRACTING SAMPLE4GEO QUERY DESCRIPTORS")
print("=" * 72)

query_descriptors, query_timing = (
    extract_descriptors(
        dataframe=query_df,
        path_column=query_path_column,
        dataset_label="Queries",
    )
)

# ------------------------------------------------------------
# 10. Save embeddings
# ------------------------------------------------------------
# Explicit Unicode dtypes avoid object arrays and allow:
# np.load(path, allow_pickle=False)

np.savez_compressed(
    GALLERY_OUTPUT_PATH,
    descriptors=gallery_descriptors,
    tile_ids=np.asarray(
        gallery_df[
            gallery_tile_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
    location_ids=np.asarray(
        gallery_df[
            gallery_location_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
    absolute_paths=np.asarray(
        gallery_df[
            gallery_path_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
)

np.savez_compressed(
    QUERY_OUTPUT_PATH,
    descriptors=query_descriptors,
    frame_ids=np.asarray(
        query_df[
            query_frame_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
    location_ids=np.asarray(
        query_df[
            query_location_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
    nominal_heights=np.asarray(
        query_df[
            query_height_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
    positive_tile_ids=np.asarray(
        query_df[
            positive_tile_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
    absolute_paths=np.asarray(
        query_df[
            query_path_column
        ].astype(str).tolist(),
        dtype=np.str_,
    ),
)

# ------------------------------------------------------------
# 11. Reload files with pickle disabled
# ------------------------------------------------------------

gallery_validation = np.load(
    GALLERY_OUTPUT_PATH,
    allow_pickle=False,
)

query_validation = np.load(
    QUERY_OUTPUT_PATH,
    allow_pickle=False,
)

if (
    gallery_validation[
        "descriptors"
    ].shape
    != (40, 1024)
):
    raise RuntimeError(
        "Saved gallery embedding shape is invalid."
    )

if (
    query_validation[
        "descriptors"
    ].shape
    != (8000, 1024)
):
    raise RuntimeError(
        "Saved query embedding shape is invalid."
    )

gallery_validation.close()
query_validation.close()

# ------------------------------------------------------------
# 12. Save extraction report
# ------------------------------------------------------------

report = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "Sample4Geo",

    "model_name": MODEL_NAME,

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_loading": (
        "STRICT_SUCCESS"
    ),

    "provenance_label": (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "official_sample4geo_sues200_result": False,

    "benchmark_protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "input_size": [
        IMAGE_SIZE,
        IMAGE_SIZE,
    ],

    "descriptor_dimension": 1024,

    "descriptor_normalization": (
        "L2"
    ),

    "preprocessing": {
        "color_conversion": (
            "OpenCV_BGR_TO_RGB"
        ),
        "resize_interpolation": (
            "cv2.INTER_LINEAR_EXACT"
        ),
        "mean": (
            NORMALIZATION_MEAN.tolist()
        ),
        "std": (
            NORMALIZATION_STD.tolist()
        ),
    },

    "gallery": {
        "manifest": str(
            GALLERY_MANIFEST_PATH
        ),
        "embedding_file": str(
            GALLERY_OUTPUT_PATH
        ),
        "descriptor_shape": list(
            gallery_descriptors.shape
        ),
        "timing": gallery_timing,
    },

    "queries": {
        "manifest": str(
            QUERY_MANIFEST_PATH
        ),
        "embedding_file": str(
            QUERY_OUTPUT_PATH
        ),
        "descriptor_shape": list(
            query_descriptors.shape
        ),
        "timing": query_timing,
    },

    "timing_interpretation": (
        "BATCHED_COLAB_RUNTIME_MEASUREMENT_"
        "NOT_JETSON_ORIN_NANO_DEPLOYMENT_LATENCY"
    ),
}

EXTRACTION_REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 13. Final output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("✅ SAMPLE4GEO DESCRIPTOR EXTRACTION COMPLETE")
print("=" * 76)

print("\nGallery descriptor shape:")
print(gallery_descriptors.shape)

print("\nQuery descriptor shape:")
print(query_descriptors.shape)

print("\nGallery inference time:")
print(
    f"{gallery_timing['model_inference_ms_per_image']:.3f} "
    "ms/image"
)

print("\nGallery end-to-end time:")
print(
    f"{gallery_timing['end_to_end_ms_per_image']:.3f} "
    "ms/image"
)

print("\nQuery inference time:")
print(
    f"{query_timing['model_inference_ms_per_image']:.3f} "
    "ms/image"
)

print("\nQuery end-to-end time:")
print(
    f"{query_timing['end_to_end_ms_per_image']:.3f} "
    "ms/image"
)

print("\nGallery embeddings:")
print(GALLERY_OUTPUT_PATH)

print("\nQuery embeddings:")
print(QUERY_OUTPUT_PATH)

print("\nExtraction report:")
print(EXTRACTION_REPORT_PATH)

print("\nPickle-free loading validation:")
print("Successful")

print("\nNext action:")
print(
    "Evaluate Sample4Geo retrieval on the same "
    "8,000-query and 40-gallery protocol."
)

Device:
cuda

Batch size:
16

Strict checkpoint loading:
Successful

Normalization mean:
[0.48500001430511475, 0.4560000002384186, 0.4059999883174896]

Normalization std:
[0.2290000021457672, 0.2240000069141388, 0.22499999403953552]

Query images:
8000

Gallery images:
40

EXTRACTING SAMPLE4GEO GALLERY DESCRIPTORS
Gallery: 16/40
Gallery: 40/40

EXTRACTING SAMPLE4GEO QUERY DESCRIPTORS
Queries: 16/8000
Queries: 800/8000
Queries: 1600/8000
Queries: 2400/8000
Queries: 3200/8000
Queries: 4000/8000
Queries: 4800/8000
Queries: 5600/8000
Queries: 6400/8000
Queries: 7200/8000
Queries: 8000/8000

✅ SAMPLE4GEO DESCRIPTOR EXTRACTION COMPLETE

Gallery descriptor shape:
(40, 1024)

Query descriptor shape:
(8000, 1024)

Gallery inference time:
137.017 ms/image

Gallery end-to-end time:
169.310 ms/image

Query inference time:
39.092 ms/image

Query end-to-end time:
39.399 ms/image

Gallery embeddings:
/content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/sample4geo_sues200_test_

In [32]:
# ============================================================
# PHASE 3 — CELL 7
# Evaluate Sample4Geo on the common SUES-200 test protocol
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time
import json

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

RANKING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

GALLERY_EMBEDDING_PATH = (
    EMBEDDING_ROOT
    / "sample4geo_sues200_test_gallery_embeddings.npz"
)

QUERY_EMBEDDING_PATH = (
    EMBEDDING_ROOT
    / "sample4geo_sues200_test_query_embeddings.npz"
)

METRICS_CSV = (
    REPORT_ROOT
    / "sample4geo_location_disjoint_metrics.csv"
)

METRICS_JSON = (
    REPORT_ROOT
    / "sample4geo_location_disjoint_metrics.json"
)

QUERY_RESULTS_CSV = (
    RANKING_ROOT
    / "sample4geo_query_results.csv"
)

COMPLETE_RANKINGS_CSV = (
    RANKING_ROOT
    / "sample4geo_complete_rankings.csv"
)

PER_LOCATION_METRICS_CSV = (
    REPORT_ROOT
    / "sample4geo_per_location_metrics.csv"
)

for path in [
    GALLERY_EMBEDDING_PATH,
    QUERY_EMBEDDING_PATH,
]:
    if not path.is_file():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 2. Load pickle-free embedding files
# ------------------------------------------------------------

gallery_data = np.load(
    GALLERY_EMBEDDING_PATH,
    allow_pickle=False,
)

query_data = np.load(
    QUERY_EMBEDDING_PATH,
    allow_pickle=False,
)

gallery_descriptors = (
    gallery_data["descriptors"]
    .astype(
        np.float32,
        copy=False,
    )
)

query_descriptors = (
    query_data["descriptors"]
    .astype(
        np.float32,
        copy=False,
    )
)

gallery_tile_ids = (
    gallery_data["tile_ids"]
    .astype(str)
)

gallery_location_ids = (
    gallery_data["location_ids"]
    .astype(str)
)

gallery_paths = (
    gallery_data["absolute_paths"]
    .astype(str)
)

query_frame_ids = (
    query_data["frame_ids"]
    .astype(str)
)

query_location_ids = (
    query_data["location_ids"]
    .astype(str)
)

query_nominal_heights = (
    query_data["nominal_heights"]
    .astype(str)
)

query_positive_tile_ids = (
    query_data["positive_tile_ids"]
    .astype(str)
)

query_paths = (
    query_data["absolute_paths"]
    .astype(str)
)

gallery_data.close()
query_data.close()

print("Gallery descriptors:")
print(gallery_descriptors.shape)

print("\nQuery descriptors:")
print(query_descriptors.shape)

# ------------------------------------------------------------
# 3. Validate embedding dimensions and metadata
# ------------------------------------------------------------

if gallery_descriptors.shape != (40, 1024):
    raise RuntimeError(
        "Unexpected gallery descriptor shape: "
        f"{gallery_descriptors.shape}"
    )

if query_descriptors.shape != (8000, 1024):
    raise RuntimeError(
        "Unexpected query descriptor shape: "
        f"{query_descriptors.shape}"
    )

if len(np.unique(gallery_tile_ids)) != 40:
    raise RuntimeError(
        "Gallery tile IDs are not unique."
    )

if not np.isfinite(
    gallery_descriptors
).all():
    raise RuntimeError(
        "Gallery descriptors contain NaN or infinity."
    )

if not np.isfinite(
    query_descriptors
).all():
    raise RuntimeError(
        "Query descriptors contain NaN or infinity."
    )

# Defensively normalize again.
gallery_norms = np.linalg.norm(
    gallery_descriptors,
    axis=1,
    keepdims=True,
)

query_norms = np.linalg.norm(
    query_descriptors,
    axis=1,
    keepdims=True,
)

gallery_descriptors = (
    gallery_descriptors
    / np.clip(
        gallery_norms,
        1e-12,
        None,
    )
)

query_descriptors = (
    query_descriptors
    / np.clip(
        query_norms,
        1e-12,
        None,
    )
)

# ------------------------------------------------------------
# 4. Resolve each query's positive gallery index
# ------------------------------------------------------------

tile_to_gallery_index = {
    tile_id: index
    for index, tile_id
    in enumerate(gallery_tile_ids)
}

missing_positive_tiles = sorted(
    set(query_positive_tile_ids)
    - set(gallery_tile_ids)
)

if missing_positive_tiles:
    raise RuntimeError(
        "Some positive tile IDs are missing from the gallery:\n"
        f"{missing_positive_tiles[:20]}"
    )

positive_gallery_indices = np.asarray(
    [
        tile_to_gallery_index[
            tile_id
        ]
        for tile_id in query_positive_tile_ids
    ],
    dtype=np.int64,
)

# ------------------------------------------------------------
# 5. Exact cosine-similarity retrieval
# ------------------------------------------------------------

search_start = time.perf_counter()

similarity_matrix = (
    query_descriptors
    @ gallery_descriptors.T
)

ranking_indices = np.argsort(
    -similarity_matrix,
    axis=1,
    kind="stable",
)

ranked_scores = np.take_along_axis(
    similarity_matrix,
    ranking_indices,
    axis=1,
)

search_seconds = (
    time.perf_counter()
    - search_start
)

search_ms_per_query = (
    1000.0
    * search_seconds
    / len(query_descriptors)
)

# ------------------------------------------------------------
# 6. Positive ranks and Top-1 results
# ------------------------------------------------------------

positive_matches = (
    ranking_indices
    == positive_gallery_indices[:, None]
)

if not positive_matches.any(axis=1).all():
    raise RuntimeError(
        "At least one query has no positive gallery match."
    )

positive_ranks = (
    np.argmax(
        positive_matches,
        axis=1,
    )
    + 1
)

top1_gallery_indices = (
    ranking_indices[:, 0]
)

top2_gallery_indices = (
    ranking_indices[:, 1]
)

top1_scores = (
    ranked_scores[:, 0]
)

top2_scores = (
    ranked_scores[:, 1]
)

top1_top2_margins = (
    top1_scores
    - top2_scores
)

top1_tile_ids = (
    gallery_tile_ids[
        top1_gallery_indices
    ]
)

top1_location_ids = (
    gallery_location_ids[
        top1_gallery_indices
    ]
)

top2_tile_ids = (
    gallery_tile_ids[
        top2_gallery_indices
    ]
)

top2_location_ids = (
    gallery_location_ids[
        top2_gallery_indices
    ]
)

top1_correct = (
    top1_gallery_indices
    == positive_gallery_indices
)

# ------------------------------------------------------------
# 7. Metric helper
# ------------------------------------------------------------

def calculate_metrics(
    ranks,
    margins,
):
    ranks = np.asarray(
        ranks,
        dtype=np.int64,
    )

    margins = np.asarray(
        margins,
        dtype=np.float64,
    )

    return {
        "query_count": int(
            len(ranks)
        ),

        "recall_at_1_percent": float(
            100.0
            * np.mean(ranks <= 1)
        ),

        "recall_at_5_percent": float(
            100.0
            * np.mean(ranks <= 5)
        ),

        "recall_at_10_percent": float(
            100.0
            * np.mean(ranks <= 10)
        ),

        # One positive gallery image per query:
        # AP equals reciprocal rank.
        "mean_average_precision_percent": float(
            100.0
            * np.mean(
                1.0 / ranks
            )
        ),

        "mean_positive_rank": float(
            np.mean(ranks)
        ),

        "median_positive_rank": float(
            np.median(ranks)
        ),

        "p95_positive_rank": float(
            np.percentile(
                ranks,
                95,
            )
        ),

        "maximum_positive_rank": int(
            np.max(ranks)
        ),

        "mean_top1_top2_margin": float(
            np.mean(margins)
        ),

        "median_top1_top2_margin": float(
            np.median(margins)
        ),
    }

# ------------------------------------------------------------
# 8. Overall and per-height metrics
# ------------------------------------------------------------

metric_records = []

overall_metrics = calculate_metrics(
    positive_ranks,
    top1_top2_margins,
)

metric_records.append({
    "group": "overall",
    "nominal_height": "all",
    **overall_metrics,
})

unique_heights = sorted(
    np.unique(
        query_nominal_heights
    ),
    key=lambda value: int(value),
)

for nominal_height in unique_heights:

    mask = (
        query_nominal_heights
        == nominal_height
    )

    height_metrics = calculate_metrics(
        positive_ranks[mask],
        top1_top2_margins[mask],
    )

    metric_records.append({
        "group": (
            f"height_{nominal_height}"
        ),
        "nominal_height": (
            nominal_height
        ),
        **height_metrics,
    })

metrics_df = pd.DataFrame(
    metric_records
)

# ------------------------------------------------------------
# 9. Compact per-query results
# ------------------------------------------------------------

query_results_df = pd.DataFrame({
    "frame_id": query_frame_ids,
    "query_location_id": (
        query_location_ids
    ),
    "nominal_height": (
        query_nominal_heights
    ),
    "positive_tile_id": (
        query_positive_tile_ids
    ),
    "positive_rank": positive_ranks,
    "top1_tile_id": top1_tile_ids,
    "top1_location_id": (
        top1_location_ids
    ),
    "top1_score": top1_scores,
    "top2_tile_id": top2_tile_ids,
    "top2_location_id": (
        top2_location_ids
    ),
    "top2_score": top2_scores,
    "top1_top2_margin": (
        top1_top2_margins
    ),
    "top1_correct": top1_correct,
    "absolute_path": query_paths,
})

# ------------------------------------------------------------
# 10. Complete 320,000-row ranking table
# ------------------------------------------------------------

query_count = len(
    query_frame_ids
)

gallery_count = len(
    gallery_tile_ids
)

flat_gallery_indices = (
    ranking_indices.reshape(-1)
)

complete_rankings_df = pd.DataFrame({
    "frame_id": np.repeat(
        query_frame_ids,
        gallery_count,
    ),

    "query_location_id": np.repeat(
        query_location_ids,
        gallery_count,
    ),

    "nominal_height": np.repeat(
        query_nominal_heights,
        gallery_count,
    ),

    "positive_tile_id": np.repeat(
        query_positive_tile_ids,
        gallery_count,
    ),

    "rank": np.tile(
        np.arange(
            1,
            gallery_count + 1,
            dtype=np.int16,
        ),
        query_count,
    ),

    "retrieved_tile_id": (
        gallery_tile_ids[
            flat_gallery_indices
        ]
    ),

    "retrieved_location_id": (
        gallery_location_ids[
            flat_gallery_indices
        ]
    ),

    "similarity": (
        ranked_scores.reshape(-1)
    ),
})

complete_rankings_df[
    "is_positive"
] = (
    complete_rankings_df[
        "retrieved_tile_id"
    ]
    == complete_rankings_df[
        "positive_tile_id"
    ]
)

if len(complete_rankings_df) != 320000:
    raise RuntimeError(
        "Complete ranking row count is incorrect."
    )

# ------------------------------------------------------------
# 11. Per-location metrics
# ------------------------------------------------------------

per_location_records = []

for location_id in sorted(
    np.unique(
        query_location_ids
    )
):

    location_mask = (
        query_location_ids
        == location_id
    )

    location_metrics = calculate_metrics(
        positive_ranks[
            location_mask
        ],
        top1_top2_margins[
            location_mask
        ],
    )

    per_location_records.append({
        "location_id": location_id,
        **location_metrics,
    })

per_location_df = pd.DataFrame(
    per_location_records
)

# ------------------------------------------------------------
# 12. Save CSV files
# ------------------------------------------------------------

metrics_df.to_csv(
    METRICS_CSV,
    index=False,
)

query_results_df.to_csv(
    QUERY_RESULTS_CSV,
    index=False,
)

complete_rankings_df.to_csv(
    COMPLETE_RANKINGS_CSV,
    index=False,
)

per_location_df.to_csv(
    PER_LOCATION_METRICS_CSV,
    index=False,
)

# ------------------------------------------------------------
# 13. Save evaluation JSON
# ------------------------------------------------------------

evaluation_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "Sample4Geo",

    "model_name": (
        "convnext_base.fb_in22k_ft_in1k_384"
    ),

    "checkpoint_training_dataset": (
        "University-1652"
    ),

    "checkpoint_provenance": (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT"
    ),

    "evaluation_provenance_label": (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "official_sample4geo_sues200_result": False,

    "benchmark_protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "query_count": int(
        query_count
    ),

    "gallery_count": int(
        gallery_count
    ),

    "descriptor_dimension": int(
        query_descriptors.shape[1]
    ),

    "similarity": (
        "COSINE_SIMILARITY_ON_L2_NORMALIZED_DESCRIPTORS"
    ),

    "ranking": (
        "EXACT_FULL_GALLERY_SORT"
    ),

    "positive_definition": (
        "ONE_MATCHING_SATELLITE_TILE_PER_QUERY"
    ),

    "overall_metrics": (
        overall_metrics
    ),

    "per_height_metrics": {
        str(row["nominal_height"]): {
            key: (
                value.item()
                if isinstance(
                    value,
                    np.generic,
                )
                else value
            )
            for key, value in row.items()
            if key not in [
                "group",
                "nominal_height",
            ]
        }
        for _, row in metrics_df[
            metrics_df["group"]
            != "overall"
        ].iterrows()
    },

    "search_timing": {
        "total_seconds": float(
            search_seconds
        ),
        "milliseconds_per_query": float(
            search_ms_per_query
        ),
        "scope": (
            "SIMILARITY_MATRIX_AND_FULL_SORT_ONLY"
        ),
        "descriptor_extraction_excluded": True,
    },

    "files": {
        "metrics_csv": str(
            METRICS_CSV
        ),
        "query_results_csv": str(
            QUERY_RESULTS_CSV
        ),
        "complete_rankings_csv": str(
            COMPLETE_RANKINGS_CSV
        ),
        "per_location_metrics_csv": str(
            PER_LOCATION_METRICS_CSV
        ),
    },

    "reporting_restrictions": [
        (
            "Do not describe this as an official "
            "Sample4Geo SUES-200 result."
        ),
        (
            "This is zero-shot transfer from a "
            "University-1652 checkpoint."
        ),
        (
            "Do not interpret retrieval as verified "
            "geographic UAV pose."
        ),
    ],
}

METRICS_JSON.write_text(
    json.dumps(
        evaluation_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 14. Final output
# ------------------------------------------------------------

display_columns = [
    "group",
    "nominal_height",
    "query_count",
    "recall_at_1_percent",
    "recall_at_5_percent",
    "recall_at_10_percent",
    "mean_average_precision_percent",
    "mean_positive_rank",
]

display_df = metrics_df[
    display_columns
].copy()

print("\n" + "=" * 78)
print("✅ SAMPLE4GEO COMMON BENCHMARK EVALUATION COMPLETE")
print("=" * 78)

print()

print(
    display_df.to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda value: f"{value:.2f}"
            ),
            "recall_at_5_percent": (
                lambda value: f"{value:.2f}"
            ),
            "recall_at_10_percent": (
                lambda value: f"{value:.2f}"
            ),
            "mean_average_precision_percent": (
                lambda value: f"{value:.2f}"
            ),
            "mean_positive_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nSearch time:")
print(
    f"{search_seconds:.6f} seconds"
)

print("\nSearch time per query:")
print(
    f"{search_ms_per_query:.6f} ms"
)

print("\nTop-1 correct queries:")
print(
    int(top1_correct.sum())
)

print("\nTop-1 failures:")
print(
    int((~top1_correct).sum())
)

print("\nMaximum positive rank:")
print(
    int(positive_ranks.max())
)

print("\nComplete ranking rows:")
print(
    len(complete_rankings_df)
)

print("\nMetrics CSV:")
print(METRICS_CSV)

print("\nMetrics JSON:")
print(METRICS_JSON)

print("\nQuery results:")
print(QUERY_RESULTS_CSV)

print("\nComplete rankings:")
print(COMPLETE_RANKINGS_CSV)

print("\nPer-location metrics:")
print(PER_LOCATION_METRICS_CSV)

print("\nNext action:")
print(
    "Compare Sample4Geo against "
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION "
    "under the common evaluator."
)

Gallery descriptors:
(40, 1024)

Query descriptors:
(8000, 1024)

✅ SAMPLE4GEO COMMON BENCHMARK EVALUATION COMPLETE

     group nominal_height  query_count recall_at_1_percent recall_at_5_percent recall_at_10_percent mean_average_precision_percent mean_positive_rank
   overall            all         8000               85.90               98.74                99.78                          91.79              1.274
height_150            150         2000               78.55               96.50                99.10                          86.61              1.563
height_200            200         2000               85.85               99.05               100.00                          91.75              1.252
height_250            250         2000               88.35               99.45               100.00                          93.54              1.176
height_300            300         2000               90.85               99.95               100.00                          95.25   

In [34]:
# ============================================================
# PHASE 3 — CELL 8
# Common comparison:
# Local reimplementation vs Sample4Geo
# Corrected for optional image-path columns
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RANKING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_METRICS_CSV = (
    PHASE2_ROOT
    / "corrected_location_disjoint_metrics.csv"
)

LOCAL_QUERY_RESULTS_CSV = (
    PHASE2_ROOT
    / "corrected_query_results.csv"
)

SAMPLE4GEO_METRICS_CSV = (
    REPORT_ROOT
    / "sample4geo_location_disjoint_metrics.csv"
)

SAMPLE4GEO_QUERY_RESULTS_CSV = (
    RANKING_ROOT
    / "sample4geo_query_results.csv"
)

COMMON_METRICS_CSV = (
    REPORT_ROOT
    / "common_benchmark_metrics.csv"
)

COMMON_METRICS_JSON = (
    REPORT_ROOT
    / "common_benchmark_metrics.json"
)

PAIRWISE_SUMMARY_CSV = (
    REPORT_ROOT
    / "local_vs_sample4geo_pairwise_summary.csv"
)

PAIRED_QUERY_RESULTS_CSV = (
    RANKING_ROOT
    / "local_vs_sample4geo_paired_query_results.csv"
)

HEIGHT_COMPARISON_CSV = (
    REPORT_ROOT
    / "local_vs_sample4geo_height_comparison.csv"
)

COMPARISON_REPORT_MD = (
    REPORT_ROOT
    / "local_vs_sample4geo_comparison_report.md"
)

for required_path in [
    LOCAL_METRICS_CSV,
    LOCAL_QUERY_RESULTS_CSV,
    SAMPLE4GEO_METRICS_CSV,
    SAMPLE4GEO_QUERY_RESULTS_CSV,
]:
    if not required_path.is_file():
        raise FileNotFoundError(
            required_path
        )

# ------------------------------------------------------------
# 2. Load metric files
# ------------------------------------------------------------

local_metrics = pd.read_csv(
    LOCAL_METRICS_CSV
)

sample_metrics = pd.read_csv(
    SAMPLE4GEO_METRICS_CSV
)

required_metric_columns = {
    "group",
    "nominal_height",
    "query_count",
    "recall_at_1_percent",
    "recall_at_5_percent",
    "recall_at_10_percent",
    "mean_average_precision_percent",
    "mean_positive_rank",
}

for label, dataframe in [
    ("local", local_metrics),
    ("sample4geo", sample_metrics),
]:
    missing_columns = (
        required_metric_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{label} metrics are missing: "
            f"{sorted(missing_columns)}"
        )

# ------------------------------------------------------------
# 3. Create shared metrics table
# ------------------------------------------------------------

local_common = (
    local_metrics.copy()
)

local_common.insert(
    0,
    "method",
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION",
)

local_common.insert(
    1,
    "provenance",
    "LOCAL_REIMPLEMENTATION",
)

sample_common = (
    sample_metrics.copy()
)

sample_common.insert(
    0,
    "method",
    "Sample4Geo",
)

sample_common.insert(
    1,
    "provenance",
    (
        "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_FROM_UNIVERSITY1652"
    ),
)

common_metrics = pd.concat(
    [
        local_common,
        sample_common,
    ],
    ignore_index=True,
)

common_metrics.to_csv(
    COMMON_METRICS_CSV,
    index=False,
)

# ------------------------------------------------------------
# 4. Load per-query results
# ------------------------------------------------------------

local_queries = pd.read_csv(
    LOCAL_QUERY_RESULTS_CSV,
    dtype=str,
)

sample_queries = pd.read_csv(
    SAMPLE4GEO_QUERY_RESULTS_CSV,
    dtype=str,
)

print("Local query-result columns:")
print(local_queries.columns.tolist())

print("\nSample4Geo query-result columns:")
print(sample_queries.columns.tolist())

required_query_columns = {
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
    "positive_rank",
    "top1_tile_id",
    "top1_location_id",
    "top1_score",
    "top1_top2_margin",
    "top1_correct",
}

for label, dataframe in [
    ("local", local_queries),
    ("sample4geo", sample_queries),
]:
    missing_columns = (
        required_query_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{label} query results are missing: "
            f"{sorted(missing_columns)}"
        )

if len(local_queries) != 8000:
    raise RuntimeError(
        "Expected 8,000 local query results, "
        f"found {len(local_queries)}."
    )

if len(sample_queries) != 8000:
    raise RuntimeError(
        "Expected 8,000 Sample4Geo query results, "
        f"found {len(sample_queries)}."
    )

# ------------------------------------------------------------
# 5. Type conversion helpers
# ------------------------------------------------------------

def convert_boolean(
    series,
):
    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    result = normalized.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    })

    if result.isna().any():
        invalid_values = sorted(
            normalized[
                result.isna()
            ].unique().tolist()
        )

        raise ValueError(
            "Unsupported boolean values: "
            f"{invalid_values}"
        )

    return result.astype(bool)


numeric_columns = [
    "positive_rank",
    "top1_score",
    "top1_top2_margin",
]

for dataframe in [
    local_queries,
    sample_queries,
]:
    dataframe[
        "frame_id"
    ] = dataframe[
        "frame_id"
    ].astype(str)

    dataframe[
        "query_location_id"
    ] = (
        dataframe[
            "query_location_id"
        ]
        .astype(str)
        .str.zfill(4)
    )

    dataframe[
        "top1_location_id"
    ] = (
        dataframe[
            "top1_location_id"
        ]
        .astype(str)
        .str.zfill(4)
    )

    dataframe[
        "nominal_height"
    ] = (
        dataframe[
            "nominal_height"
        ]
        .astype(str)
        .str.replace(
            ".0",
            "",
            regex=False,
        )
    )

    dataframe[
        "positive_tile_id"
    ] = dataframe[
        "positive_tile_id"
    ].astype(str)

    dataframe[
        "top1_tile_id"
    ] = dataframe[
        "top1_tile_id"
    ].astype(str)

    dataframe[
        "top1_correct"
    ] = convert_boolean(
        dataframe[
            "top1_correct"
        ]
    )

    for column in numeric_columns:
        dataframe[
            column
        ] = pd.to_numeric(
            dataframe[
                column
            ],
            errors="raise",
        )

# ------------------------------------------------------------
# 6. Validate unique frame IDs
# ------------------------------------------------------------

if not local_queries[
    "frame_id"
].is_unique:
    raise RuntimeError(
        "Local frame IDs are not unique."
    )

if not sample_queries[
    "frame_id"
].is_unique:
    raise RuntimeError(
        "Sample4Geo frame IDs are not unique."
    )

# ------------------------------------------------------------
# 7. Validate identical query identities
# ------------------------------------------------------------
# Image paths are intentionally not required because the local
# query-results file does not contain absolute_path.

alignment_columns = [
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
]

local_alignment = (
    local_queries[
        alignment_columns
    ]
    .sort_values(
        "frame_id"
    )
    .reset_index(drop=True)
)

sample_alignment = (
    sample_queries[
        alignment_columns
    ]
    .sort_values(
        "frame_id"
    )
    .reset_index(drop=True)
)

if not local_alignment.equals(
    sample_alignment
):
    comparison = (
        local_alignment
        != sample_alignment
    )

    mismatch_count = int(
        comparison.any(
            axis=1
        ).sum()
    )

    print("\nFirst mismatched rows:")

    mismatch_indices = np.where(
        comparison.any(
            axis=1
        ).to_numpy()
    )[0][:10]

    for index in mismatch_indices:
        print("\nLocal:")
        print(
            local_alignment.iloc[
                index
            ].to_dict()
        )

        print("Sample4Geo:")
        print(
            sample_alignment.iloc[
                index
            ].to_dict()
        )

    raise RuntimeError(
        "Query metadata are not identical. "
        f"Mismatched rows: {mismatch_count}"
    )

print("\nQuery alignment:")
print("Successful — 8,000 identical query identities")

# ------------------------------------------------------------
# 8. Detect an optional image path
# ------------------------------------------------------------

path_candidates = [
    "absolute_path",
    "query_path",
    "image_path",
    "path",
    "filepath",
    "file_path",
]

local_path_column = next(
    (
        column
        for column in path_candidates
        if column in local_queries.columns
    ),
    None,
)

sample_path_column = next(
    (
        column
        for column in path_candidates
        if column in sample_queries.columns
    ),
    None,
)

print("\nLocal optional path column:")
print(local_path_column)

print("\nSample4Geo optional path column:")
print(sample_path_column)

# ------------------------------------------------------------
# 9. Prepare columns for paired merge
# ------------------------------------------------------------

local_columns = [
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
    "positive_rank",
    "top1_tile_id",
    "top1_location_id",
    "top1_score",
    "top1_top2_margin",
    "top1_correct",
]

if local_path_column:
    local_columns.append(
        local_path_column
    )

local_selected = (
    local_queries[
        local_columns
    ]
    .copy()
    .rename(
        columns={
            "positive_rank": (
                "local_positive_rank"
            ),
            "top1_tile_id": (
                "local_top1_tile_id"
            ),
            "top1_location_id": (
                "local_top1_location_id"
            ),
            "top1_score": (
                "local_top1_score"
            ),
            "top1_top2_margin": (
                "local_top1_top2_margin"
            ),
            "top1_correct": (
                "local_top1_correct"
            ),
        }
    )
)

if local_path_column:
    local_selected = (
        local_selected.rename(
            columns={
                local_path_column: (
                    "local_image_path"
                )
            }
        )
    )

sample_columns = [
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
    "positive_rank",
    "top1_tile_id",
    "top1_location_id",
    "top1_score",
    "top1_top2_margin",
    "top1_correct",
]

if sample_path_column:
    sample_columns.append(
        sample_path_column
    )

sample_selected = (
    sample_queries[
        sample_columns
    ]
    .copy()
    .rename(
        columns={
            "positive_rank": (
                "sample4geo_positive_rank"
            ),
            "top1_tile_id": (
                "sample4geo_top1_tile_id"
            ),
            "top1_location_id": (
                "sample4geo_top1_location_id"
            ),
            "top1_score": (
                "sample4geo_top1_score"
            ),
            "top1_top2_margin": (
                "sample4geo_top1_top2_margin"
            ),
            "top1_correct": (
                "sample4geo_top1_correct"
            ),
        }
    )
)

if sample_path_column:
    sample_selected = (
        sample_selected.rename(
            columns={
                sample_path_column: (
                    "sample4geo_image_path"
                )
            }
        )
    )

# ------------------------------------------------------------
# 10. Merge paired results
# ------------------------------------------------------------

merge_keys = [
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
]

paired_queries = (
    local_selected.merge(
        sample_selected,
        on=merge_keys,
        how="inner",
        validate="one_to_one",
    )
)

if len(paired_queries) != 8000:
    raise RuntimeError(
        "Paired merge did not return 8,000 rows. "
        f"Rows returned: {len(paired_queries)}"
    )

# Create one optional image-path output column.
if (
    "local_image_path"
    in paired_queries.columns
):
    paired_queries[
        "image_path"
    ] = paired_queries[
        "local_image_path"
    ]

elif (
    "sample4geo_image_path"
    in paired_queries.columns
):
    paired_queries[
        "image_path"
    ] = paired_queries[
        "sample4geo_image_path"
    ]

else:
    paired_queries[
        "image_path"
    ] = ""

# ------------------------------------------------------------
# 11. Paired outcomes
# ------------------------------------------------------------

local_correct = paired_queries[
    "local_top1_correct"
].to_numpy(
    dtype=bool
)

sample_correct = paired_queries[
    "sample4geo_top1_correct"
].to_numpy(
    dtype=bool
)

both_correct = (
    local_correct
    & sample_correct
)

local_only_correct = (
    local_correct
    & ~sample_correct
)

sample_only_correct = (
    ~local_correct
    & sample_correct
)

both_wrong = (
    ~local_correct
    & ~sample_correct
)

paired_queries[
    "paired_outcome"
] = np.select(
    [
        both_correct,
        local_only_correct,
        sample_only_correct,
        both_wrong,
    ],
    [
        "both_correct",
        "local_only_correct",
        "sample4geo_only_correct",
        "both_wrong",
    ],
    default="invalid",
)

paired_queries[
    "positive_rank_difference_sample_minus_local"
] = (
    paired_queries[
        "sample4geo_positive_rank"
    ]
    - paired_queries[
        "local_positive_rank"
    ]
)

paired_queries[
    "same_top1_prediction"
] = (
    paired_queries[
        "local_top1_tile_id"
    ]
    == paired_queries[
        "sample4geo_top1_tile_id"
    ]
)

paired_queries.to_csv(
    PAIRED_QUERY_RESULTS_CSV,
    index=False,
)

# ------------------------------------------------------------
# 12. Exact McNemar test
# ------------------------------------------------------------

local_only_count = int(
    local_only_correct.sum()
)

sample_only_count = int(
    sample_only_correct.sum()
)

discordant_count = (
    local_only_count
    + sample_only_count
)

try:
    from scipy.stats import binomtest

    if discordant_count > 0:
        mcnemar_p_value = float(
            binomtest(
                min(
                    local_only_count,
                    sample_only_count,
                ),
                n=discordant_count,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )
    else:
        mcnemar_p_value = 1.0

except ImportError:
    mcnemar_p_value = None

if discordant_count > 0:
    mcnemar_chi_square = float(
        (
            abs(
                local_only_count
                - sample_only_count
            )
            - 1
        ) ** 2
        / discordant_count
    )
else:
    mcnemar_chi_square = 0.0

# ------------------------------------------------------------
# 13. Overall metrics
# ------------------------------------------------------------

local_overall = (
    local_metrics[
        local_metrics["group"]
        == "overall"
    ]
    .iloc[0]
)

sample_overall = (
    sample_metrics[
        sample_metrics["group"]
        == "overall"
    ]
    .iloc[0]
)

overall_comparison = {
    "local_recall_at_1_percent": float(
        local_overall[
            "recall_at_1_percent"
        ]
    ),

    "sample4geo_recall_at_1_percent": float(
        sample_overall[
            "recall_at_1_percent"
        ]
    ),

    "recall_at_1_difference": float(
        local_overall[
            "recall_at_1_percent"
        ]
        - sample_overall[
            "recall_at_1_percent"
        ]
    ),

    "local_recall_at_5_percent": float(
        local_overall[
            "recall_at_5_percent"
        ]
    ),

    "sample4geo_recall_at_5_percent": float(
        sample_overall[
            "recall_at_5_percent"
        ]
    ),

    "recall_at_5_difference": float(
        local_overall[
            "recall_at_5_percent"
        ]
        - sample_overall[
            "recall_at_5_percent"
        ]
    ),

    "local_recall_at_10_percent": float(
        local_overall[
            "recall_at_10_percent"
        ]
    ),

    "sample4geo_recall_at_10_percent": float(
        sample_overall[
            "recall_at_10_percent"
        ]
    ),

    "recall_at_10_difference": float(
        local_overall[
            "recall_at_10_percent"
        ]
        - sample_overall[
            "recall_at_10_percent"
        ]
    ),

    "local_map_percent": float(
        local_overall[
            "mean_average_precision_percent"
        ]
    ),

    "sample4geo_map_percent": float(
        sample_overall[
            "mean_average_precision_percent"
        ]
    ),

    "map_difference": float(
        local_overall[
            "mean_average_precision_percent"
        ]
        - sample_overall[
            "mean_average_precision_percent"
        ]
    ),

    "local_mean_positive_rank": float(
        local_overall[
            "mean_positive_rank"
        ]
    ),

    "sample4geo_mean_positive_rank": float(
        sample_overall[
            "mean_positive_rank"
        ]
    ),
}

# ------------------------------------------------------------
# 14. Per-height comparison
# ------------------------------------------------------------

local_height = (
    local_metrics[
        local_metrics["group"]
        != "overall"
    ]
    .copy()
)

sample_height = (
    sample_metrics[
        sample_metrics["group"]
        != "overall"
    ]
    .copy()
)

local_height[
    "nominal_height"
] = (
    local_height[
        "nominal_height"
    ]
    .astype(str)
    .str.replace(
        ".0",
        "",
        regex=False,
    )
)

sample_height[
    "nominal_height"
] = (
    sample_height[
        "nominal_height"
    ]
    .astype(str)
    .str.replace(
        ".0",
        "",
        regex=False,
    )
)

height_comparison = (
    local_height[
        [
            "nominal_height",
            "query_count",
            "recall_at_1_percent",
            "recall_at_5_percent",
            "recall_at_10_percent",
            "mean_average_precision_percent",
            "mean_positive_rank",
        ]
    ]
    .merge(
        sample_height[
            [
                "nominal_height",
                "query_count",
                "recall_at_1_percent",
                "recall_at_5_percent",
                "recall_at_10_percent",
                "mean_average_precision_percent",
                "mean_positive_rank",
            ]
        ],
        on="nominal_height",
        suffixes=(
            "_local",
            "_sample4geo",
        ),
        validate="one_to_one",
    )
)

height_comparison[
    "recall_at_1_difference_local_minus_sample4geo"
] = (
    height_comparison[
        "recall_at_1_percent_local"
    ]
    - height_comparison[
        "recall_at_1_percent_sample4geo"
    ]
)

height_comparison[
    "map_difference_local_minus_sample4geo"
] = (
    height_comparison[
        "mean_average_precision_percent_local"
    ]
    - height_comparison[
        "mean_average_precision_percent_sample4geo"
    ]
)

height_comparison = (
    height_comparison
    .sort_values(
        "nominal_height",
        key=lambda series: (
            series.astype(int)
        ),
    )
    .reset_index(drop=True)
)

height_comparison.to_csv(
    HEIGHT_COMPARISON_CSV,
    index=False,
)

# ------------------------------------------------------------
# 15. Paired summary
# ------------------------------------------------------------

pairwise_summary = pd.DataFrame([
    {
        "outcome": "both_correct",
        "query_count": int(
            both_correct.sum()
        ),
    },
    {
        "outcome": "local_only_correct",
        "query_count": (
            local_only_count
        ),
    },
    {
        "outcome": "sample4geo_only_correct",
        "query_count": (
            sample_only_count
        ),
    },
    {
        "outcome": "both_wrong",
        "query_count": int(
            both_wrong.sum()
        ),
    },
])

pairwise_summary[
    "query_percent"
] = (
    100.0
    * pairwise_summary[
        "query_count"
    ]
    / 8000
)

pairwise_summary.to_csv(
    PAIRWISE_SUMMARY_CSV,
    index=False,
)

# ------------------------------------------------------------
# 16. Save JSON
# ------------------------------------------------------------

comparison_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "benchmark": (
        "COMMON_SUES200_LOCATION_DISJOINT_BENCHMARK"
    ),

    "protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "query_count": 8000,
    "gallery_count": 40,

    "overall_comparison": (
        overall_comparison
    ),

    "paired_outcomes": {
        "both_correct": int(
            both_correct.sum()
        ),
        "local_only_correct": (
            local_only_count
        ),
        "sample4geo_only_correct": (
            sample_only_count
        ),
        "both_wrong": int(
            both_wrong.sum()
        ),
        "discordant_count": int(
            discordant_count
        ),
    },

    "mcnemar_test": {
        "exact_two_sided_p_value": (
            mcnemar_p_value
        ),
        "continuity_corrected_chi_square": (
            mcnemar_chi_square
        ),
    },

    "provenance": {
        "local": (
            "LOCAL_REIMPLEMENTATION"
        ),
        "sample4geo": (
            "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
            "ZERO_SHOT_TRANSFER_FROM_UNIVERSITY1652"
        ),
    },

    "files": {
        "common_metrics": str(
            COMMON_METRICS_CSV
        ),
        "paired_results": str(
            PAIRED_QUERY_RESULTS_CSV
        ),
        "paired_summary": str(
            PAIRWISE_SUMMARY_CSV
        ),
        "height_comparison": str(
            HEIGHT_COMPARISON_CSV
        ),
    },
}

COMMON_METRICS_JSON.write_text(
    json.dumps(
        comparison_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 17. Markdown report
# ------------------------------------------------------------

height_lines = []

for _, row in height_comparison.iterrows():
    height_lines.append(
        "| "
        f"{row['nominal_height']} | "
        f"{row['recall_at_1_percent_local']:.2f}% | "
        f"{row['recall_at_1_percent_sample4geo']:.2f}% | "
        f"{row['recall_at_1_difference_local_minus_sample4geo']:.2f} | "
        f"{row['mean_average_precision_percent_local']:.2f}% | "
        f"{row['mean_average_precision_percent_sample4geo']:.2f}% |"
    )

height_table = "\n".join(
    height_lines
)

report_text = f"""# Local Model vs Sample4Geo

## Protocol

- Queries: 8,000
- Gallery images: 40
- Split: `PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20`
- Retrieval: exact cosine similarity

## Overall results

| Metric | Local reimplementation | Sample4Geo | Difference |
|---|---:|---:|---:|
| Recall@1 | {overall_comparison['local_recall_at_1_percent']:.2f}% | {overall_comparison['sample4geo_recall_at_1_percent']:.2f}% | {overall_comparison['recall_at_1_difference']:.2f} |
| Recall@5 | {overall_comparison['local_recall_at_5_percent']:.2f}% | {overall_comparison['sample4geo_recall_at_5_percent']:.2f}% | {overall_comparison['recall_at_5_difference']:.2f} |
| Recall@10 | {overall_comparison['local_recall_at_10_percent']:.2f}% | {overall_comparison['sample4geo_recall_at_10_percent']:.2f}% | {overall_comparison['recall_at_10_difference']:.2f} |
| mAP | {overall_comparison['local_map_percent']:.2f}% | {overall_comparison['sample4geo_map_percent']:.2f}% | {overall_comparison['map_difference']:.2f} |

## Nominal-height comparison

| Height | Local R@1 | Sample4Geo R@1 | Difference | Local mAP | Sample4Geo mAP |
|---:|---:|---:|---:|---:|---:|
{height_table}

## Paired Top-1 outcomes

- Both correct: {int(both_correct.sum())}
- Local only correct: {local_only_count}
- Sample4Geo only correct: {sample_only_count}
- Both wrong: {int(both_wrong.sum())}
- McNemar exact p-value: {mcnemar_p_value}

## Interpretation

The local reimplementation performs better on this particular
project-defined test split. This does not isolate architecture because
the checkpoints have different training datasets and training regimes.

Sample4Geo is evaluated as zero-shot transfer from University-1652.
The local model is not an official MobileGeo implementation.
"""

COMPARISON_REPORT_MD.write_text(
    report_text,
    encoding="utf-8",
)

# ------------------------------------------------------------
# 18. Display results
# ------------------------------------------------------------

overall_display = pd.DataFrame([
    {
        "method": (
            "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
        ),
        "Recall@1": (
            overall_comparison[
                "local_recall_at_1_percent"
            ]
        ),
        "Recall@5": (
            overall_comparison[
                "local_recall_at_5_percent"
            ]
        ),
        "Recall@10": (
            overall_comparison[
                "local_recall_at_10_percent"
            ]
        ),
        "mAP": (
            overall_comparison[
                "local_map_percent"
            ]
        ),
        "mean_rank": (
            overall_comparison[
                "local_mean_positive_rank"
            ]
        ),
    },
    {
        "method": "Sample4Geo",
        "Recall@1": (
            overall_comparison[
                "sample4geo_recall_at_1_percent"
            ]
        ),
        "Recall@5": (
            overall_comparison[
                "sample4geo_recall_at_5_percent"
            ]
        ),
        "Recall@10": (
            overall_comparison[
                "sample4geo_recall_at_10_percent"
            ]
        ),
        "mAP": (
            overall_comparison[
                "sample4geo_map_percent"
            ]
        ),
        "mean_rank": (
            overall_comparison[
                "sample4geo_mean_positive_rank"
            ]
        ),
    },
])

print("\n" + "=" * 78)
print("✅ LOCAL VS SAMPLE4GEO COMMON COMPARISON COMPLETE")
print("=" * 78)

print("\nOverall comparison:")

print(
    overall_display.to_string(
        index=False,
        formatters={
            "Recall@1": (
                lambda value: f"{value:.2f}"
            ),
            "Recall@5": (
                lambda value: f"{value:.2f}"
            ),
            "Recall@10": (
                lambda value: f"{value:.2f}"
            ),
            "mAP": (
                lambda value: f"{value:.2f}"
            ),
            "mean_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nDifference, local minus Sample4Geo:")

print(
    "Recall@1:",
    f"{overall_comparison['recall_at_1_difference']:.2f}",
    "percentage points",
)

print(
    "mAP:",
    f"{overall_comparison['map_difference']:.2f}",
    "percentage points",
)

print("\nPer-height Recall@1 comparison:")

print(
    height_comparison[
        [
            "nominal_height",
            "recall_at_1_percent_local",
            "recall_at_1_percent_sample4geo",
            (
                "recall_at_1_difference_"
                "local_minus_sample4geo"
            ),
        ]
    ].to_string(
        index=False,
        formatters={
            "recall_at_1_percent_local": (
                lambda value: f"{value:.2f}"
            ),
            "recall_at_1_percent_sample4geo": (
                lambda value: f"{value:.2f}"
            ),
            (
                "recall_at_1_difference_"
                "local_minus_sample4geo"
            ): (
                lambda value: f"{value:.2f}"
            ),
        },
    )
)

print("\nPaired Top-1 outcomes:")

print(
    pairwise_summary.to_string(
        index=False,
        formatters={
            "query_percent": (
                lambda value: f"{value:.2f}"
            ),
        },
    )
)

print("\nMcNemar exact p-value:")
print(mcnemar_p_value)

print("\nSame Top-1 prediction count:")
print(
    int(
        paired_queries[
            "same_top1_prediction"
        ].sum()
    )
)

print("\nCommon metrics:")
print(COMMON_METRICS_CSV)

print("\nHeight comparison:")
print(HEIGHT_COMPARISON_CSV)

print("\nPaired query results:")
print(PAIRED_QUERY_RESULTS_CSV)

print("\nComparison report:")
print(COMPARISON_REPORT_MD)

print("\nNext action:")
print(
    "Audit UltraVPR for an official checkpoint "
    "and compatible descriptor pipeline."
)

Local query-result columns:
['frame_id', 'query_location_id', 'nominal_height', 'positive_tile_id', 'positive_rank', 'reciprocal_rank', 'top1_tile_id', 'top1_location_id', 'top1_score', 'top2_tile_id', 'top2_score', 'top1_top2_margin', 'top1_correct']

Sample4Geo query-result columns:
['frame_id', 'query_location_id', 'nominal_height', 'positive_tile_id', 'positive_rank', 'top1_tile_id', 'top1_location_id', 'top1_score', 'top2_tile_id', 'top2_location_id', 'top2_score', 'top1_top2_margin', 'top1_correct', 'absolute_path']

Query alignment:
Successful — 8,000 identical query identities

Local optional path column:
None

Sample4Geo optional path column:
absolute_path

✅ LOCAL VS SAMPLE4GEO COMMON COMPARISON COMPLETE

Overall comparison:
                                   method Recall@1 Recall@5 Recall@10   mAP mean_rank
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION    94.58    99.99    100.00 96.93     1.082
                               Sample4Geo    85.90    98.74     99.78 91.79     1.2

In [35]:
# ============================================================
# PHASE 3 — CELL 9
# Clone and audit the official UltraVPR repository
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import hashlib
import json
import re

import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPOSITORIES_ROOT = (
    PROJECT_ROOT
    / "repositories"
)

ULTRAVPR_REPO = (
    REPOSITORIES_ROOT
    / "UltraVPR"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "ultravpr_audit"
)

REPOSITORY_URL = (
    "https://github.com/cbbhuxx/UltraVPR.git"
)

REPOSITORIES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. Command helper
# ------------------------------------------------------------

def run_command(
    command,
    cwd=None,
):
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        print("Command:")
        print(" ".join(command))

        print("\nSTDOUT:")
        print(result.stdout)

        print("\nSTDERR:")
        print(result.stderr)

        raise RuntimeError(
            "Command failed."
        )

    return result.stdout.strip()

# ------------------------------------------------------------
# 3. Clone repository
# ------------------------------------------------------------

if not (
    ULTRAVPR_REPO
    / ".git"
).is_dir():

    if ULTRAVPR_REPO.exists():
        shutil.rmtree(
            ULTRAVPR_REPO
        )

    print(
        "Cloning official UltraVPR repository..."
    )

    run_command(
        [
            "git",
            "clone",
            REPOSITORY_URL,
            str(ULTRAVPR_REPO),
        ]
    )

else:
    print(
        "UltraVPR repository already exists."
    )

    # Fetch remote metadata without silently replacing
    # the pinned local working tree.
    run_command(
        [
            "git",
            "fetch",
            "--all",
            "--tags",
        ],
        cwd=ULTRAVPR_REPO,
    )

# ------------------------------------------------------------
# 4. Record repository state
# ------------------------------------------------------------

commit_hash = run_command(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=ULTRAVPR_REPO,
)

short_commit = run_command(
    [
        "git",
        "rev-parse",
        "--short",
        "HEAD",
    ],
    cwd=ULTRAVPR_REPO,
)

branch_name = run_command(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=ULTRAVPR_REPO,
)

commit_date = run_command(
    [
        "git",
        "show",
        "-s",
        "--format=%cI",
        "HEAD",
    ],
    cwd=ULTRAVPR_REPO,
)

commit_subject = run_command(
    [
        "git",
        "show",
        "-s",
        "--format=%s",
        "HEAD",
    ],
    cwd=ULTRAVPR_REPO,
)

remote_url = run_command(
    [
        "git",
        "remote",
        "get-url",
        "origin",
    ],
    cwd=ULTRAVPR_REPO,
)

# ------------------------------------------------------------
# 5. SHA-256 helper
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()

# ------------------------------------------------------------
# 6. Repository inventory
# ------------------------------------------------------------

checkpoint_extensions = {
    ".pth",
    ".pt",
    ".ckpt",
    ".bin",
    ".safetensors",
}

deployment_extensions = {
    ".onnx",
    ".engine",
    ".trt",
}

configuration_extensions = {
    ".yaml",
    ".yml",
    ".json",
    ".toml",
}

inventory_records = []

checkpoint_files = []
deployment_files = []
python_files = []
configuration_files = []
license_files = []
readme_files = []
requirement_files = []
evaluation_files = []
export_files = []

for file_path in sorted(
    ULTRAVPR_REPO.rglob("*")
):
    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(
        ULTRAVPR_REPO
    )

    if ".git" in relative_path.parts:
        continue

    if "__pycache__" in relative_path.parts:
        continue

    suffix = file_path.suffix.lower()
    filename_lower = (
        file_path.name.lower()
    )

    if suffix in checkpoint_extensions:
        category = "checkpoint"

        checkpoint_files.append(
            str(relative_path)
        )

    elif suffix in deployment_extensions:
        category = "deployment_model"

        deployment_files.append(
            str(relative_path)
        )

    elif suffix == ".py":
        category = "python_source"

        python_files.append(
            str(relative_path)
        )

        if filename_lower.startswith(
            "eval"
        ):
            evaluation_files.append(
                str(relative_path)
            )

        if (
            "export" in filename_lower
            or "onnx" in filename_lower
        ):
            export_files.append(
                str(relative_path)
            )

    elif suffix in configuration_extensions:
        category = "configuration"

        configuration_files.append(
            str(relative_path)
        )

    elif filename_lower.startswith(
        "license"
    ):
        category = "license"

        license_files.append(
            str(relative_path)
        )

    elif filename_lower.startswith(
        "readme"
    ):
        category = "readme"

        readme_files.append(
            str(relative_path)
        )

    elif (
        "requirement" in filename_lower
        or filename_lower.startswith(
            "environment"
        )
    ):
        category = "environment"

        requirement_files.append(
            str(relative_path)
        )

    else:
        category = "other"

    inventory_records.append({
        "relative_path": str(
            relative_path
        ),
        "filename": file_path.name,
        "extension": suffix,
        "category": category,
        "size_bytes": int(
            file_path.stat().st_size
        ),
        "sha256": sha256_file(
            file_path
        ),
    })

inventory_df = pd.DataFrame(
    inventory_records
)

# ------------------------------------------------------------
# 7. Search source and README evidence
# ------------------------------------------------------------

text_extensions = {
    ".md",
    ".txt",
    ".py",
    ".yaml",
    ".yml",
    ".json",
    ".sh",
}

search_patterns = {
    "weights_reference": (
        r"weight"
        r"|checkpoint"
        r"|pretrained"
        r"|\.pth\b"
        r"|\.pt\b"
    ),

    "onnx_reference": (
        r"onnx"
        r"|tensorrt"
        r"|\.engine\b"
    ),

    "uav_visloc_reference": (
        r"UAV[-_ ]?VisLoc"
    ),

    "vpair_reference": (
        r"VPAir"
    ),

    "sues200_reference": (
        r"SUES[-_ ]?200"
    ),

    "evaluation_reference": (
        r"\beval"
        r"|\btest"
        r"|inference"
        r"|descriptor"
        r"|feature"
    ),

    "model_loading_reference": (
        r"torch\.load"
        r"|load_state_dict"
        r"|resume"
        r"|model_path"
    ),

    "input_size_reference": (
        r"image_size"
        r"|img_size"
        r"|resize"
        r"|crop"
        r"|input_size"
    ),

    "download_link": (
        r"https?://"
        r"|pan\.baidu"
        r"|cloud\.tsinghua"
        r"|drive\.google"
        r"|huggingface"
    ),
}

evidence_records = []

for file_path in sorted(
    ULTRAVPR_REPO.rglob("*")
):
    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(
        ULTRAVPR_REPO
    )

    if ".git" in relative_path.parts:
        continue

    if (
        file_path.suffix.lower()
        not in text_extensions
    ):
        continue

    lines = file_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        for category, pattern in (
            search_patterns.items()
        ):
            if re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            ):
                evidence_records.append({
                    "category": category,
                    "relative_path": str(
                        relative_path
                    ),
                    "line_number": int(
                        line_number
                    ),
                    "line_text": (
                        line.strip()
                    ),
                })

evidence_df = pd.DataFrame(
    evidence_records
)

# ------------------------------------------------------------
# 8. Extract URLs from repository text
# ------------------------------------------------------------

url_records = []

url_pattern = re.compile(
    r"https?://[^\s<>\"]+"
)

for file_path in sorted(
    ULTRAVPR_REPO.rglob("*")
):
    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(
        ULTRAVPR_REPO
    )

    if ".git" in relative_path.parts:
        continue

    if (
        file_path.suffix.lower()
        not in text_extensions
    ):
        continue

    lines = file_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        urls = url_pattern.findall(
            line
        )

        for url in urls:
            cleaned_url = url.rstrip(
                ".,;:)]}'"
            )

            url_records.append({
                "relative_path": str(
                    relative_path
                ),
                "line_number": int(
                    line_number
                ),
                "url": cleaned_url,
                "line_text": (
                    line.strip()
                ),
            })

urls_df = pd.DataFrame(
    url_records
)

# ------------------------------------------------------------
# 9. Determine release compatibility
# ------------------------------------------------------------

has_checkpoint_inside_repo = (
    len(checkpoint_files) > 0
)

has_deployment_model_inside_repo = (
    len(deployment_files) > 0
)

has_evaluation_scripts = (
    len(evaluation_files) > 0
)

has_export_scripts = (
    len(export_files) > 0
)

has_license = (
    len(license_files) > 0
)

if evidence_df.empty:
    has_sues200_reference = False
    has_weights_reference = False
    has_onnx_reference = False

else:
    has_sues200_reference = bool(
        (
            evidence_df["category"]
            == "sues200_reference"
        ).any()
    )

    has_weights_reference = bool(
        (
            evidence_df["category"]
            == "weights_reference"
        ).any()
    )

    has_onnx_reference = bool(
        (
            evidence_df["category"]
            == "onnx_reference"
        ).any()
    )

if has_sues200_reference:
    benchmark_status = (
        "SUES200_PIPELINE_REQUIRES_VALIDATION"
    )

    planned_provenance = (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "SUES200_PIPELINE_REQUIRES_VALIDATION"
    )

else:
    benchmark_status = (
        "OFFICIAL_CHECKPOINT_DISCOVERY_REQUIRED_"
        "THEN_ZERO_SHOT_TRANSFER"
    )

    planned_provenance = (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    )

# ------------------------------------------------------------
# 10. Save audit files
# ------------------------------------------------------------

INVENTORY_CSV = (
    AUDIT_ROOT
    / "ultravpr_repository_inventory.csv"
)

EVIDENCE_CSV = (
    AUDIT_ROOT
    / "ultravpr_source_evidence.csv"
)

URLS_CSV = (
    AUDIT_ROOT
    / "ultravpr_detected_urls.csv"
)

AUDIT_JSON = (
    AUDIT_ROOT
    / "ultravpr_initial_audit.json"
)

inventory_df.to_csv(
    INVENTORY_CSV,
    index=False,
)

evidence_df.to_csv(
    EVIDENCE_CSV,
    index=False,
)

urls_df.to_csv(
    URLS_CSV,
    index=False,
)

audit_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "UltraVPR",

    "repository": (
        "cbbhuxx/UltraVPR"
    ),

    "repository_url": (
        REPOSITORY_URL
    ),

    "remote_url": remote_url,

    "local_path": str(
        ULTRAVPR_REPO
    ),

    "branch": branch_name,

    "commit_hash": commit_hash,

    "short_commit": short_commit,

    "commit_date": commit_date,

    "commit_subject": (
        commit_subject
    ),

    "python_source_file_count": int(
        len(python_files)
    ),

    "configuration_file_count": int(
        len(configuration_files)
    ),

    "checkpoint_files_inside_repository": (
        checkpoint_files
    ),

    "checkpoint_found_inside_repository": (
        has_checkpoint_inside_repo
    ),

    "deployment_model_files": (
        deployment_files
    ),

    "deployment_model_found_inside_repository": (
        has_deployment_model_inside_repo
    ),

    "evaluation_files": (
        evaluation_files
    ),

    "export_files": export_files,

    "evaluation_scripts_found": (
        has_evaluation_scripts
    ),

    "export_scripts_found": (
        has_export_scripts
    ),

    "weights_reference_found": (
        has_weights_reference
    ),

    "onnx_reference_found": (
        has_onnx_reference
    ),

    "license_files": license_files,

    "license_found": has_license,

    "readme_files": readme_files,

    "environment_files": (
        requirement_files
    ),

    "sues200_reference_found": (
        has_sues200_reference
    ),

    "common_benchmark_status": (
        benchmark_status
    ),

    "planned_provenance_label": (
        planned_provenance
    ),

    "common_benchmark_ready": False,

    "next_action": (
        "Resolve the official UltraVPR checkpoint "
        "download URL and inspect model loading, "
        "input preprocessing and descriptor output."
    ),
}

AUDIT_JSON.write_text(
    json.dumps(
        audit_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 11. Print concise audit output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("✅ ULTRAVPR REPOSITORY AUDIT COMPLETE")
print("=" * 76)

print("\nRepository:")
print(ULTRAVPR_REPO)

print("\nBranch:")
print(branch_name)

print("\nPinned commit:")
print(commit_hash)

print("\nCommit date:")
print(commit_date)

print("\nPython source files:")
print(len(python_files))

print("\nCheckpoint files inside repository:")
print(len(checkpoint_files))

for path in checkpoint_files:
    print("-", path)

print("\nDeployment model files inside repository:")
print(len(deployment_files))

for path in deployment_files:
    print("-", path)

print("\nEvaluation scripts:")

for path in evaluation_files:
    print("-", path)

print("\nExport scripts:")

for path in export_files:
    print("-", path)

print("\nLicense files:")

for path in license_files:
    print("-", path)

print("\nEnvironment files:")

for path in requirement_files:
    print("-", path)

print("\nSUES-200 references found:")
print(has_sues200_reference)

print("\nCommon benchmark status:")
print(benchmark_status)

print("\nPlanned provenance:")
print(planned_provenance)

print("\nRelevant release evidence:")

if evidence_df.empty:
    print("No evidence records found.")

else:
    selected_evidence = (
        evidence_df[
            evidence_df[
                "category"
            ].isin(
                [
                    "weights_reference",
                    "onnx_reference",
                    "uav_visloc_reference",
                    "vpair_reference",
                    "sues200_reference",
                    "model_loading_reference",
                ]
            )
        ]
        .drop_duplicates(
            subset=[
                "relative_path",
                "line_number",
                "line_text",
            ]
        )
        .head(60)
    )

    for _, row in (
        selected_evidence.iterrows()
    ):
        print(
            f"{row['relative_path']}:"
            f"{row['line_number']} — "
            f"{row['line_text']}"
        )

print("\nDetected download URLs:")

if urls_df.empty:
    print("No URLs detected.")

else:
    selected_urls = (
        urls_df
        .drop_duplicates(
            subset=["url"]
        )
    )

    for _, row in (
        selected_urls.iterrows()
    ):
        print(
            f"- {row['url']}"
        )

print("\nInventory CSV:")
print(INVENTORY_CSV)

print("\nEvidence CSV:")
print(EVIDENCE_CSV)

print("\nDetected URLs CSV:")
print(URLS_CSV)

print("\nAudit JSON:")
print(AUDIT_JSON)

print("\nNext action:")
print(
    "Resolve and download the official UltraVPR "
    "checkpoint without selecting an unrelated "
    "backbone or deployment file."
)

Cloning official UltraVPR repository...

✅ ULTRAVPR REPOSITORY AUDIT COMPLETE

Repository:
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR

Branch:
main

Pinned commit:
e8456de13661e7d7584396c9ac3610ec020d0aaf

Commit date:
2025-12-17T11:18:19+08:00

Python source files:
15

Checkpoint files inside repository:
0

Deployment model files inside repository:
0

Evaluation scripts:
- eval_UAV-VisLoc.py
- eval_VPAir.py

Export scripts:
- export_ultravpr.py

License files:
- LICENSE

Environment files:
- requirements.txt

SUES-200 references found:
False

Common benchmark status:
OFFICIAL_CHECKPOINT_DISCOVERY_REQUIRED_THEN_ZERO_SHOT_TRANSFER

Planned provenance:
OFFICIAL_ULTRAVPR_CHECKPOINT_ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT

Relevant release evidence:
README.md:3 — <h3 align="center"><a href="https://ieeexplore.ieee.org/document/11091472" target='_blank'>UltraVPR: Unsupervised Lightweight Rotation-Invariant Aerial Visual Place Recognition</a> </h3>
README.md:14 — - [✅

In [36]:
# ============================================================
# PHASE 3 — CELL 10
# Discover, download and inspect the official UltraVPR checkpoint
# from the public Tsinghua/Seafile share
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import posixpath
import hashlib
import json
import os

import requests
import torch

# ------------------------------------------------------------
# 1. Paths and official share information
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "ultravpr_audit"
)

CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "checkpoints"
    / "ultravpr"
)

CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "model_best.pth.tar"
)

CHECKPOINT_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "ultravpr_checkpoint_path.txt"
)

AUDIT_JSON = (
    AUDIT_ROOT
    / "ultravpr_checkpoint_audit.json"
)

TSINGHUA_BASE_URL = (
    "https://cloud.tsinghua.edu.cn"
)

SHARE_TOKEN = (
    "ea66ca1bbb334132bcd1"
)

PUBLIC_SHARE_URL = (
    f"{TSINGHUA_BASE_URL}/d/"
    f"{SHARE_TOKEN}/"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_CONFIG_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. HTTP session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(X11; Linux x86_64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/130.0 Safari/537.36"
    ),
    "Accept": (
        "application/json,text/plain,"
        "text/html,*/*"
    ),
    "Referer": PUBLIC_SHARE_URL,
})

# Visit the public share once to establish any required cookies.
landing_response = session.get(
    PUBLIC_SHARE_URL,
    timeout=60,
    allow_redirects=True,
)

print("Public share landing status:")
print(landing_response.status_code)

print("\nPublic share final URL:")
print(landing_response.url)

# ------------------------------------------------------------
# 3. Flexible Seafile directory-entry parser
# ------------------------------------------------------------

def normalize_entries(
    payload,
):
    """
    Normalize several possible Seafile API response formats
    into a simple list of entry dictionaries.
    """

    if isinstance(payload, list):
        return payload

    if not isinstance(payload, dict):
        return []

    possible_keys = [
        "dirent_list",
        "entries",
        "data",
        "items",
        "results",
    ]

    for key in possible_keys:

        value = payload.get(key)

        if isinstance(value, list):
            return value

    return []


def entry_name(
    entry,
):
    for key in [
        "name",
        "filename",
        "file_name",
    ]:
        value = entry.get(key)

        if value:
            return str(value)

    return None


def entry_is_directory(
    entry,
):
    entry_type = str(
        entry.get(
            "type",
            entry.get(
                "is_dir",
                entry.get(
                    "kind",
                    "",
                ),
            ),
        )
    ).lower()

    return entry_type in {
        "dir",
        "directory",
        "d",
        "true",
        "1",
    }


# ------------------------------------------------------------
# 4. List one public directory
# ------------------------------------------------------------

def list_public_directory(
    directory_path,
):
    """
    Try both commonly deployed Seafile public-share APIs.
    """

    attempts = []

    # Older public download-link API.
    old_api_url = (
        f"{TSINGHUA_BASE_URL}/api2/d/"
        f"{SHARE_TOKEN}/dir/"
    )

    try:
        response = session.get(
            old_api_url,
            params={
                "p": directory_path,
            },
            timeout=60,
        )

        attempts.append({
            "url": response.url,
            "status": response.status_code,
            "content_type": response.headers.get(
                "content-type",
                "",
            ),
        })

        if response.ok:
            payload = response.json()
            entries = normalize_entries(
                payload
            )

            if entries or directory_path == "/":
                return entries, attempts

    except Exception as error:
        attempts.append({
            "url": old_api_url,
            "error": repr(error),
        })

    # Newer v2.1 share-link API.
    new_api_url = (
        f"{TSINGHUA_BASE_URL}/api/v2.1/"
        f"share-links/{SHARE_TOKEN}/dirents/"
    )

    for parameter_name in [
        "path",
        "p",
    ]:
        try:
            response = session.get(
                new_api_url,
                params={
                    parameter_name: (
                        directory_path
                    ),
                },
                timeout=60,
            )

            attempts.append({
                "url": response.url,
                "status": response.status_code,
                "content_type": (
                    response.headers.get(
                        "content-type",
                        "",
                    )
                ),
            })

            if response.ok:
                payload = response.json()
                entries = normalize_entries(
                    payload
                )

                if entries or directory_path == "/":
                    return entries, attempts

        except Exception as error:
            attempts.append({
                "url": new_api_url,
                "parameter": parameter_name,
                "error": repr(error),
            })

    raise RuntimeError(
        "Could not list the Tsinghua public share.\n"
        + json.dumps(
            attempts,
            indent=2,
        )
    )


# ------------------------------------------------------------
# 5. Recursively inventory the official share
# ------------------------------------------------------------

inventory = []
api_attempt_log = []

directories_to_scan = [
    "/",
]

scanned_directories = set()

MAX_DIRECTORIES = 100

while directories_to_scan:

    current_directory = (
        directories_to_scan.pop(0)
    )

    if current_directory in scanned_directories:
        continue

    scanned_directories.add(
        current_directory
    )

    if len(scanned_directories) > MAX_DIRECTORIES:
        raise RuntimeError(
            "Unexpectedly large public directory tree."
        )

    entries, attempts = (
        list_public_directory(
            current_directory
        )
    )

    api_attempt_log.extend(
        attempts
    )

    print(
        f"Scanning {current_directory}: "
        f"{len(entries)} entries"
    )

    for entry in entries:

        name = entry_name(
            entry
        )

        if not name:
            continue

        full_path = posixpath.normpath(
            posixpath.join(
                current_directory,
                name,
            )
        )

        if not full_path.startswith("/"):
            full_path = "/" + full_path

        is_directory = entry_is_directory(
            entry
        )

        size_value = entry.get(
            "size",
            entry.get(
                "file_size",
                None,
            ),
        )

        try:
            size_bytes = (
                int(size_value)
                if size_value is not None
                else None
            )

        except Exception:
            size_bytes = None

        inventory.append({
            "path": full_path,
            "name": name,
            "is_directory": (
                is_directory
            ),
            "size_bytes": size_bytes,
            "raw_entry": entry,
        })

        if is_directory:
            directories_to_scan.append(
                full_path
            )

# ------------------------------------------------------------
# 6. Print released model files
# ------------------------------------------------------------

model_extensions = (
    ".pth",
    ".pt",
    ".pth.tar",
    ".ckpt",
    ".onnx",
    ".engine",
)

model_files = [
    item
    for item in inventory
    if (
        not item["is_directory"]
        and item["path"]
        .lower()
        .endswith(model_extensions)
    )
]

print("\n" + "=" * 76)
print("MODEL FILES FOUND IN OFFICIAL ULTRAVPR SHARE")
print("=" * 76)

for item in model_files:

    size_text = (
        "unknown size"
        if item["size_bytes"] is None
        else (
            f"{item['size_bytes'] / (1024**2):.2f} MB"
        )
    )

    print(
        "-",
        item["path"],
        f"({size_text})",
    )

if not model_files:
    raise FileNotFoundError(
        "No checkpoint or deployment model was found "
        "inside the official UltraVPR share."
    )

# ------------------------------------------------------------
# 7. Select the official PyTorch retrieval checkpoint
# ------------------------------------------------------------

def checkpoint_priority(
    item,
):
    path_lower = item[
        "path"
    ].lower()

    score = 0

    if path_lower.endswith(
        "model_best.pth.tar"
    ):
        score += 1000

    if (
        "e2resnet50_c8_se2gem_32"
        in path_lower
    ):
        score += 500

    if "checkpoint" in path_lower:
        score += 100

    if "ultravpr" in path_lower:
        score += 50

    if path_lower.endswith(
        ".onnx"
    ):
        score -= 1000

    return score


pytorch_candidates = [
    item
    for item in model_files
    if item["path"].lower().endswith(
        (
            ".pth",
            ".pt",
            ".pth.tar",
            ".ckpt",
        )
    )
]

if not pytorch_candidates:
    raise FileNotFoundError(
        "The official share contains no PyTorch checkpoint."
    )

pytorch_candidates = sorted(
    pytorch_candidates,
    key=checkpoint_priority,
    reverse=True,
)

selected_checkpoint = (
    pytorch_candidates[0]
)

selected_share_path = (
    selected_checkpoint["path"]
)

selected_priority = checkpoint_priority(
    selected_checkpoint
)

if selected_priority < 1000:
    raise RuntimeError(
        "The expected official file "
        "`model_best.pth.tar` was not found. "
        "No lower-confidence checkpoint will be "
        "selected automatically."
    )

print("\nSelected official checkpoint:")
print(selected_share_path)

# ------------------------------------------------------------
# 8. Download only the selected checkpoint
# ------------------------------------------------------------

def file_looks_like_html(
    file_path,
):
    with open(
        file_path,
        "rb",
    ) as file:
        prefix = file.read(
            1024
        ).lstrip().lower()

    return (
        prefix.startswith(b"<!doctype html")
        or prefix.startswith(b"<html")
    )


if CHECKPOINT_PATH.is_file():

    print("\nCheckpoint already exists:")
    print(CHECKPOINT_PATH)

else:

    temporary_path = (
        CHECKPOINT_PATH.parent
        / (
            CHECKPOINT_PATH.name
            + ".partial"
        )
    )

    if temporary_path.exists():
        temporary_path.unlink()

    download_url = (
        f"{TSINGHUA_BASE_URL}/d/"
        f"{SHARE_TOKEN}/files/"
    )

    print("\nDownloading selected checkpoint...")

    download_response = session.get(
        download_url,
        params={
            "p": selected_share_path,
            "dl": "1",
        },
        timeout=300,
        stream=True,
        allow_redirects=True,
    )

    print("Download status:")
    print(download_response.status_code)

    print("\nDownload final URL:")
    print(download_response.url)

    print("\nDownload content type:")
    print(
        download_response.headers.get(
            "content-type",
            "unknown",
        )
    )

    download_response.raise_for_status()

    with open(
        temporary_path,
        "wb",
    ) as output_file:

        for chunk in (
            download_response.iter_content(
                chunk_size=1024 * 1024
            )
        ):
            if chunk:
                output_file.write(
                    chunk
                )

    if temporary_path.stat().st_size < (
        1024 * 1024
    ):
        raise RuntimeError(
            "Downloaded checkpoint is unexpectedly small: "
            f"{temporary_path.stat().st_size} bytes"
        )

    if file_looks_like_html(
        temporary_path
    ):
        preview = temporary_path.read_text(
            encoding="utf-8",
            errors="replace",
        )[:1000]

        temporary_path.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "The download returned HTML instead of "
            "a checkpoint.\n"
            + preview
        )

    os.replace(
        temporary_path,
        CHECKPOINT_PATH,
    )

    print("\n✅ Checkpoint downloaded:")
    print(CHECKPOINT_PATH)

# ------------------------------------------------------------
# 9. SHA-256 checksum
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file:

        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


checkpoint_size_bytes = (
    CHECKPOINT_PATH.stat().st_size
)

checkpoint_size_mb = (
    checkpoint_size_bytes
    / (1024 * 1024)
)

checkpoint_sha256 = sha256_file(
    CHECKPOINT_PATH
)

# ------------------------------------------------------------
# 10. Inspect checkpoint container
# ------------------------------------------------------------

safe_loading_error = None

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

except Exception as error:
    safe_loading_error = repr(
        error
    )

    print("\nWeights-only loading failed:")
    print(safe_loading_error)

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

print("\nCheckpoint object type:")
print(type(checkpoint))

if not isinstance(
    checkpoint,
    dict,
):
    raise RuntimeError(
        "UltraVPR checkpoint is not a dictionary."
    )

container_keys = [
    str(key)
    for key in checkpoint.keys()
]

if (
    "state_dict" in checkpoint
    and isinstance(
        checkpoint["state_dict"],
        dict,
    )
):
    state_dict = checkpoint[
        "state_dict"
    ]

    state_dict_source = (
        "checkpoint['state_dict']"
    )

elif (
    "model_state_dict" in checkpoint
    and isinstance(
        checkpoint["model_state_dict"],
        dict,
    )
):
    state_dict = checkpoint[
        "model_state_dict"
    ]

    state_dict_source = (
        "checkpoint['model_state_dict']"
    )

elif all(
    torch.is_tensor(value)
    for value in checkpoint.values()
):
    state_dict = checkpoint
    state_dict_source = (
        "checkpoint_root"
    )

else:
    raise RuntimeError(
        "No recognizable state dictionary was "
        "found in the UltraVPR checkpoint.\n"
        f"Container keys: {container_keys}"
    )

tensor_items = [
    (str(key), value)
    for key, value in state_dict.items()
    if torch.is_tensor(value)
]

if not tensor_items:
    raise RuntimeError(
        "UltraVPR state dictionary contains no tensors."
    )

total_tensor_values = sum(
    tensor.numel()
    for _, tensor in tensor_items
)

prefix_counts = {}

for key, _ in tensor_items:

    clean_key = key

    if clean_key.startswith(
        "module."
    ):
        clean_key = clean_key[
            len("module.") :
        ]

    prefix = clean_key.split(
        "."
    )[0]

    prefix_counts[prefix] = (
        prefix_counts.get(
            prefix,
            0,
        )
        + 1
    )

# ------------------------------------------------------------
# 11. Save path and audit evidence
# ------------------------------------------------------------

CHECKPOINT_CONFIG_PATH.write_text(
    str(CHECKPOINT_PATH),
    encoding="utf-8",
)

audit_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "UltraVPR",

    "repository": (
        "cbbhuxx/UltraVPR"
    ),

    "official_share_url": (
        PUBLIC_SHARE_URL
    ),

    "official_share_token": (
        SHARE_TOKEN
    ),

    "official_share_inventory": [
        {
            "path": item["path"],
            "is_directory": (
                item["is_directory"]
            ),
            "size_bytes": (
                item["size_bytes"]
            ),
        }
        for item in inventory
    ],

    "released_model_files": [
        {
            "path": item["path"],
            "size_bytes": (
                item["size_bytes"]
            ),
        }
        for item in model_files
    ],

    "selected_share_path": (
        selected_share_path
    ),

    "selection_rule": (
        "EXACT_MODEL_BEST_PTH_TAR_"
        "PREFERRED_E2RESNET50_C8_SE2GEM32"
    ),

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_size_bytes": int(
        checkpoint_size_bytes
    ),

    "checkpoint_size_mb": float(
        checkpoint_size_mb
    ),

    "checkpoint_sha256": (
        checkpoint_sha256
    ),

    "checkpoint_object_type": str(
        type(checkpoint)
    ),

    "checkpoint_container_keys": (
        container_keys
    ),

    "state_dict_source": (
        state_dict_source
    ),

    "state_dict_tensor_entries": int(
        len(tensor_items)
    ),

    "total_tensor_values": int(
        total_tensor_values
    ),

    "parameter_prefix_counts": (
        prefix_counts
    ),

    "safe_loading_error": (
        safe_loading_error
    ),

    "checkpoint_origin": (
        "OFFICIAL_ULTRAVPR_TSINGHUA_CLOUD_SHARE"
    ),

    "planned_evaluation": (
        "ZERO_SHOT_TRANSFER_TO_"
        "PROJECT_DEFINED_SUES200_SPLIT"
    ),

    "provenance_label": (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "strict_model_loading_validated": False,

    "api_attempt_log": (
        api_attempt_log
    ),
}

AUDIT_JSON.write_text(
    json.dumps(
        audit_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 12. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("✅ OFFICIAL ULTRAVPR CHECKPOINT DOWNLOADED AND INSPECTED")
print("=" * 78)

print("\nOfficial shared file:")
print(selected_share_path)

print("\nSaved checkpoint:")
print(CHECKPOINT_PATH)

print("\nCheckpoint size:")
print(
    f"{checkpoint_size_mb:.2f} MB"
)

print("\nCheckpoint SHA-256:")
print(checkpoint_sha256)

print("\nCheckpoint container keys:")
print(container_keys)

print("\nState dictionary source:")
print(state_dict_source)

print("\nTensor entries:")
print(len(tensor_items))

print("\nTotal tensor values:")
print(
    f"{total_tensor_values:,}"
)

print("\nParameter prefix counts:")

for prefix, count in sorted(
    prefix_counts.items()
):
    print(
        f"- {prefix}: {count}"
    )

print("\nFirst 40 parameter keys:")

for key, tensor in tensor_items[:40]:
    print(
        f"- {key}: {tuple(tensor.shape)}"
    )

print("\nCheckpoint path configuration:")
print(CHECKPOINT_CONFIG_PATH)

print("\nAudit JSON:")
print(AUDIT_JSON)

print("\nNext action:")
print(
    "Reconstruct the official UltraVPR architecture, "
    "resolve its input preprocessing and validate "
    "checkpoint loading."
)

Public share landing status:
200

Public share final URL:
https://cloud.tsinghua.edu.cn/d/ea66ca1bbb334132bcd1/
Scanning /: 2 entries
Scanning /checkpoints: 3 entries

MODEL FILES FOUND IN OFFICIAL ULTRAVPR SHARE
- /checkpoints/checkpoint.pth.tar (111.69 MB)
- /checkpoints/model_best.pth.tar (111.69 MB)

Selected official checkpoint:
/checkpoints/model_best.pth.tar

Download status:
200

Download final URL:
https://cloud.tsinghua.edu.cn/seafhttp/files/4a9994e5-3481-4b32-8ed5-f224e880f663/model_best.pth.tar

Download content type:
application/octet-stream

✅ Checkpoint downloaded:
/content/drive/MyDrive/mobilegeo_project/checkpoints/ultravpr/model_best.pth.tar

Checkpoint object type:
<class 'dict'>

✅ OFFICIAL ULTRAVPR CHECKPOINT DOWNLOADED AND INSPECTED

Official shared file:
/checkpoints/model_best.pth.tar

Saved checkpoint:
/content/drive/MyDrive/mobilegeo_project/checkpoints/ultravpr/model_best.pth.tar

Checkpoint size:
111.69 MB

Checkpoint SHA-256:
7fb2f95721ac3bdbbfe86b2b217b34a

In [37]:
# ============================================================
# PHASE 3 — CELL 11
# Extract exact UltraVPR architecture and preprocessing evidence
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import re

import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

ULTRAVPR_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "UltraVPR"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "ultravpr_audit"
)

CHECKPOINT_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "ultravpr_checkpoint_path.txt"
)

DEFAULT_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "ultravpr"
    / "model_best.pth.tar"
)

EVAL_VPAIR_PATH = (
    ULTRAVPR_REPO
    / "eval_VPAir.py"
)

EVAL_UAV_PATH = (
    ULTRAVPR_REPO
    / "eval_UAV-VisLoc.py"
)

EXPORT_PATH = (
    ULTRAVPR_REPO
    / "export_ultravpr.py"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for required_path in [
    ULTRAVPR_REPO,
    EVAL_VPAIR_PATH,
    EVAL_UAV_PATH,
    EXPORT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

if CHECKPOINT_CONFIG_PATH.is_file():
    CHECKPOINT_PATH = Path(
        CHECKPOINT_CONFIG_PATH
        .read_text(
            encoding="utf-8"
        )
        .strip()
    )
else:
    CHECKPOINT_PATH = (
        DEFAULT_CHECKPOINT_PATH
    )

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        "UltraVPR checkpoint is missing.\n"
        f"Expected path: {CHECKPOINT_PATH}\n"
        "Run Phase 3 Cell 10 first."
    )

print("UltraVPR repository:")
print(ULTRAVPR_REPO)

print("\nCheckpoint:")
print(CHECKPOINT_PATH)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def get_source_segment(
    source_text,
    node,
):
    lines = source_text.splitlines()

    start = max(
        int(node.lineno) - 1,
        0,
    )

    end = int(
        getattr(
            node,
            "end_lineno",
            node.lineno,
        )
    )

    return "\n".join(
        lines[start:end]
    )


def extract_functions_and_classes(
    file_path,
):
    source_text = file_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    tree = ast.parse(
        source_text,
        filename=str(file_path),
    )

    records = []

    for node in ast.walk(tree):

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ):
            records.append({
                "name": node.name,
                "node_type": (
                    type(node).__name__
                ),
                "line_start": int(
                    node.lineno
                ),
                "line_end": int(
                    getattr(
                        node,
                        "end_lineno",
                        node.lineno,
                    )
                ),
                "source": get_source_segment(
                    source_text,
                    node,
                ),
            })

    return source_text, records


def print_numbered_window(
    source_text,
    center_line,
    radius=10,
):
    lines = source_text.splitlines()

    start = max(
        center_line - radius - 1,
        0,
    )

    end = min(
        center_line + radius,
        len(lines),
    )

    for index in range(
        start,
        end,
    ):
        print(
            f"{index + 1:04d}: "
            f"{lines[index]}"
        )


# ------------------------------------------------------------
# 3. Read important repository scripts
# ------------------------------------------------------------

script_paths = {
    "eval_VPAir.py": (
        EVAL_VPAIR_PATH
    ),
    "eval_UAV-VisLoc.py": (
        EVAL_UAV_PATH
    ),
    "export_ultravpr.py": (
        EXPORT_PATH
    ),
}

script_sources = {}
definition_records = []

for script_name, script_path in (
    script_paths.items()
):
    source_text, definitions = (
        extract_functions_and_classes(
            script_path
        )
    )

    script_sources[
        script_name
    ] = source_text

    for record in definitions:
        definition_records.append({
            "script": script_name,
            **record,
        })

definitions_df = pd.DataFrame(
    definition_records
)

# ------------------------------------------------------------
# 4. Locate model-loading functions
# ------------------------------------------------------------

model_function_names = {
    "load_model",
    "get_model",
    "build_model",
    "create_model",
    "load_checkpoint",
    "convert_backbone",
    "prepare_model",
}

model_definition_mask = (
    definitions_df["name"]
    .str.lower()
    .isin(model_function_names)
)

model_definitions = (
    definitions_df[
        model_definition_mask
    ]
    .copy()
)

print("\n" + "=" * 78)
print("MODEL-CONSTRUCTION FUNCTIONS")
print("=" * 78)

if model_definitions.empty:
    print(
        "No standard model-construction "
        "function name was found."
    )
else:
    for _, row in (
        model_definitions.iterrows()
    ):
        print(
            f"\n--- {row['script']} :: "
            f"{row['name']} "
            f"(lines {row['line_start']}-"
            f"{row['line_end']}) ---"
        )

        print(
            row["source"]
        )

# ------------------------------------------------------------
# 5. Search source lines for architecture evidence
# ------------------------------------------------------------

architecture_patterns = {
    "model_constructor": (
        r"e2resnet"
        r"|resnet50"
        r"|se2gem"
        r"|GeoLocalizationNet"
        r"|network\s*="
        r"|model\s*="
    ),

    "aggregation": (
        r"gem"
        r"|se2gem"
        r"|aggregation"
        r"|pool"
        r"|descriptor"
    ),

    "checkpoint_loading": (
        r"torch\.load"
        r"|load_state_dict"
        r"|state_dict"
    ),

    "preprocessing": (
        r"Normalize"
        r"|Resize"
        r"|CenterCrop"
        r"|ToTensor"
        r"|transform"
        r"|interpolation"
    ),

    "input_shape": (
        r"224"
        r"|256"
        r"|320"
        r"|384"
        r"|480"
        r"|512"
        r"|640"
        r"|input_size"
        r"|image_size"
        r"|resize"
    ),

    "descriptor_output": (
        r"features"
        r"|descriptors"
        r"|embeddings"
        r"|output"
        r"|model\("
    ),
}

source_evidence = []

for script_name, source_text in (
    script_sources.items()
):
    lines = source_text.splitlines()

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        for category, pattern in (
            architecture_patterns.items()
        ):
            if re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            ):
                source_evidence.append({
                    "script": script_name,
                    "category": category,
                    "line_number": int(
                        line_number
                    ),
                    "line_text": line.strip(),
                })

source_evidence_df = pd.DataFrame(
    source_evidence
)

# ------------------------------------------------------------
# 6. Print focused source evidence
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("ARCHITECTURE AND CHECKPOINT EVIDENCE")
print("=" * 78)

focused_categories = [
    "model_constructor",
    "aggregation",
    "checkpoint_loading",
]

focused_evidence = (
    source_evidence_df[
        source_evidence_df[
            "category"
        ].isin(
            focused_categories
        )
    ]
    .drop_duplicates(
        subset=[
            "script",
            "line_number",
            "line_text",
        ]
    )
    .sort_values(
        [
            "script",
            "line_number",
        ]
    )
)

for _, row in (
    focused_evidence
    .head(100)
    .iterrows()
):
    print(
        f"{row['script']}:"
        f"{row['line_number']} — "
        f"{row['line_text']}"
    )

# ------------------------------------------------------------
# 7. Print preprocessing evidence
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PREPROCESSING EVIDENCE")
print("=" * 78)

preprocessing_evidence = (
    source_evidence_df[
        source_evidence_df[
            "category"
        ].isin(
            [
                "preprocessing",
                "input_shape",
            ]
        )
    ]
    .drop_duplicates(
        subset=[
            "script",
            "line_number",
            "line_text",
        ]
    )
    .sort_values(
        [
            "script",
            "line_number",
        ]
    )
)

for _, row in (
    preprocessing_evidence
    .head(120)
    .iterrows()
):
    print(
        f"{row['script']}:"
        f"{row['line_number']} — "
        f"{row['line_text']}"
    )

# ------------------------------------------------------------
# 8. Inspect all repository Python imports
# ------------------------------------------------------------

import_records = []

for file_path in sorted(
    ULTRAVPR_REPO.rglob(
        "*.py"
    )
):
    if ".git" in file_path.parts:
        continue

    relative_path = file_path.relative_to(
        ULTRAVPR_REPO
    )

    source_text = file_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    try:
        tree = ast.parse(
            source_text,
            filename=str(file_path),
        )
    except SyntaxError:
        continue

    for node in ast.walk(tree):

        if isinstance(
            node,
            ast.Import,
        ):
            for alias in node.names:
                import_records.append({
                    "file": str(
                        relative_path
                    ),
                    "import_type": "import",
                    "module": alias.name,
                    "name": alias.name,
                })

        elif isinstance(
            node,
            ast.ImportFrom,
        ):
            module_name = (
                node.module
                if node.module
                else ""
            )

            for alias in node.names:
                import_records.append({
                    "file": str(
                        relative_path
                    ),
                    "import_type": "from",
                    "module": module_name,
                    "name": alias.name,
                })

imports_df = (
    pd.DataFrame(
        import_records
    )
    .drop_duplicates()
)

model_related_imports = (
    imports_df[
        imports_df[
            "file"
        ].isin(
            [
                "eval_VPAir.py",
                "eval_UAV-VisLoc.py",
                "export_ultravpr.py",
            ]
        )
    ]
    .sort_values(
        [
            "file",
            "module",
            "name",
        ]
    )
)

print("\n" + "=" * 78)
print("MODEL-SCRIPT IMPORTS")
print("=" * 78)

print(
    model_related_imports.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 9. Inspect checkpoint structure
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

except Exception:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

if not isinstance(
    checkpoint,
    dict,
):
    raise RuntimeError(
        "Checkpoint is not a dictionary."
    )

container_keys = [
    str(key)
    for key in checkpoint.keys()
]

if (
    "state_dict" in checkpoint
    and isinstance(
        checkpoint["state_dict"],
        dict,
    )
):
    state_dict = checkpoint[
        "state_dict"
    ]

    state_dict_source = (
        "checkpoint['state_dict']"
    )

elif (
    "model_state_dict" in checkpoint
    and isinstance(
        checkpoint[
            "model_state_dict"
        ],
        dict,
    )
):
    state_dict = checkpoint[
        "model_state_dict"
    ]

    state_dict_source = (
        "checkpoint['model_state_dict']"
    )

elif all(
    torch.is_tensor(value)
    for value in checkpoint.values()
):
    state_dict = checkpoint
    state_dict_source = (
        "checkpoint_root"
    )

else:
    raise RuntimeError(
        "No recognizable state dictionary found.\n"
        f"Container keys: {container_keys}"
    )

tensor_items = [
    (
        str(key),
        tensor,
    )
    for key, tensor
    in state_dict.items()
    if torch.is_tensor(tensor)
]

print("\n" + "=" * 78)
print("CHECKPOINT STRUCTURE")
print("=" * 78)

print("\nCheckpoint SHA-256:")
print(
    sha256_file(
        CHECKPOINT_PATH
    )
)

print("\nContainer keys:")
print(container_keys)

print("\nState dictionary source:")
print(state_dict_source)

print("\nTensor entries:")
print(len(tensor_items))

print("\nFirst 100 tensor keys:")

for key, tensor in (
    tensor_items[:100]
):
    print(
        f"- {key}: "
        f"{tuple(tensor.shape)}"
    )

# ------------------------------------------------------------
# 10. Derive checkpoint prefix structure
# ------------------------------------------------------------

prefix_levels = {
    "level_1": {},
    "level_2": {},
    "level_3": {},
}

for key, _ in tensor_items:

    clean_key = key

    if clean_key.startswith(
        "module."
    ):
        clean_key = clean_key[
            len("module.") :
        ]

    parts = clean_key.split(".")

    for level_name, level_count in [
        ("level_1", 1),
        ("level_2", 2),
        ("level_3", 3),
    ]:
        prefix = ".".join(
            parts[:level_count]
        )

        prefix_levels[
            level_name
        ][prefix] = (
            prefix_levels[
                level_name
            ].get(
                prefix,
                0,
            )
            + 1
        )

print("\nTop checkpoint prefixes:")

for level_name in [
    "level_1",
    "level_2",
    "level_3",
]:
    print(
        f"\n{level_name}:"
    )

    sorted_prefixes = sorted(
        prefix_levels[
            level_name
        ].items(),
        key=lambda item: (
            -item[1],
            item[0],
        ),
    )

    for prefix, count in (
        sorted_prefixes[:30]
    ):
        print(
            f"- {prefix}: {count}"
        )

# ------------------------------------------------------------
# 11. Match checkpoint prefixes with repository source
# ------------------------------------------------------------

repository_python_text = {}

for file_path in sorted(
    ULTRAVPR_REPO.rglob(
        "*.py"
    )
):
    if ".git" in file_path.parts:
        continue

    relative_path = str(
        file_path.relative_to(
            ULTRAVPR_REPO
        )
    )

    repository_python_text[
        relative_path
    ] = file_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

source_match_records = []

important_prefixes = [
    prefix
    for prefix, _
    in sorted(
        prefix_levels[
            "level_1"
        ].items(),
        key=lambda item: (
            -item[1],
            item[0],
        ),
    )
]

for prefix in important_prefixes:

    search_token = (
        prefix.split(".")[-1]
    )

    for relative_path, source_text in (
        repository_python_text.items()
    ):
        if re.search(
            rf"\b{re.escape(search_token)}\b",
            source_text,
            flags=re.IGNORECASE,
        ):
            source_match_records.append({
                "checkpoint_prefix": prefix,
                "search_token": search_token,
                "source_file": relative_path,
            })

source_matches_df = (
    pd.DataFrame(
        source_match_records
    )
    .drop_duplicates()
    if source_match_records
    else pd.DataFrame(
        columns=[
            "checkpoint_prefix",
            "search_token",
            "source_file",
        ]
    )
)

print("\n" + "=" * 78)
print("CHECKPOINT PREFIX TO SOURCE-FILE MATCHES")
print("=" * 78)

if source_matches_df.empty:
    print(
        "No direct prefix-name matches found."
    )
else:
    print(
        source_matches_df.to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# 12. Save audit evidence
# ------------------------------------------------------------

definitions_csv = (
    AUDIT_ROOT
    / "ultravpr_function_class_definitions.csv"
)

source_evidence_csv = (
    AUDIT_ROOT
    / "ultravpr_architecture_source_evidence.csv"
)

imports_csv = (
    AUDIT_ROOT
    / "ultravpr_model_script_imports.csv"
)

source_matches_csv = (
    AUDIT_ROOT
    / "ultravpr_checkpoint_source_matches.csv"
)

architecture_audit_json = (
    AUDIT_ROOT
    / "ultravpr_architecture_preprocessing_audit.json"
)

definitions_df.to_csv(
    definitions_csv,
    index=False,
)

source_evidence_df.to_csv(
    source_evidence_csv,
    index=False,
)

model_related_imports.to_csv(
    imports_csv,
    index=False,
)

source_matches_df.to_csv(
    source_matches_csv,
    index=False,
)

audit_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "UltraVPR",

    "repository_commit": (
        "e8456de13661e7d7584396c9ac3610ec020d0aaf"
    ),

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_sha256": (
        sha256_file(
            CHECKPOINT_PATH
        )
    ),

    "checkpoint_container_keys": (
        container_keys
    ),

    "state_dict_source": (
        state_dict_source
    ),

    "state_dict_tensor_entries": int(
        len(tensor_items)
    ),

    "checkpoint_prefix_levels": (
        prefix_levels
    ),

    "model_construction_definitions": [
        {
            "script": row[
                "script"
            ],
            "name": row[
                "name"
            ],
            "line_start": int(
                row[
                    "line_start"
                ]
            ),
            "line_end": int(
                row[
                    "line_end"
                ]
            ),
            "source": row[
                "source"
            ],
        }
        for _, row in (
            model_definitions.iterrows()
        )
    ],

    "sues200_pipeline_found": False,

    "planned_evaluation": (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_"
        "PROJECT_DEFINED_SUES200_SPLIT"
    ),

    "next_action": (
        "Construct the model using the exact "
        "repository function and validate checkpoint "
        "loading and descriptor output."
    ),

    "files": {
        "definitions_csv": str(
            definitions_csv
        ),
        "source_evidence_csv": str(
            source_evidence_csv
        ),
        "imports_csv": str(
            imports_csv
        ),
        "source_matches_csv": str(
            source_matches_csv
        ),
    },
}

architecture_audit_json.write_text(
    json.dumps(
        audit_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 13. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("✅ ULTRAVPR ARCHITECTURE AND PREPROCESSING AUDIT COMPLETE")
print("=" * 78)

print("\nModel-construction functions found:")
print(
    model_definitions[
        [
            "script",
            "name",
            "line_start",
            "line_end",
        ]
    ].to_string(
        index=False
    )
    if not model_definitions.empty
    else "None"
)

print("\nCheckpoint tensor entries:")
print(len(tensor_items))

print("\nArchitecture audit JSON:")
print(architecture_audit_json)

print("\nNext action:")
print(
    "Use the extracted official construction function "
    "for strict-compatible model loading and a "
    "descriptor smoke test."
)

UltraVPR repository:
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR

Checkpoint:
/content/drive/MyDrive/mobilegeo_project/checkpoints/ultravpr/model_best.pth.tar

MODEL-CONSTRUCTION FUNCTIONS

--- eval_VPAir.py :: load_model (lines 109-119) ---
def load_model(ckpt_path):
    model = ultravpr()

    if(ckpt_path != ""):
        state_dict = torch.load(ckpt_path, map_location='cpu')
        model.load_state_dict(state_dict['state_dict'], strict=False)
        # model.load_state_dict(state_dict, strict=False)
        print(f"Loaded model from {ckpt_path} Successfully!")
    model.eval()
   
    return model

--- eval_UAV-VisLoc.py :: load_model (lines 153-163) ---
def load_model(ckpt_path):
    model = ultravpr()
    
    if(ckpt_path != ""):
        state_dict = torch.load(ckpt_path, map_location='cpu')
        model.load_state_dict(state_dict['state_dict'], strict=False)
        # model.load_state_dict(state_dict, strict=False)
        print(f"Loaded model from {ckpt_pat

/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR/eval_VPAir.py:157: SyntaxWarning: invalid escape sequence '\d'
  query_label = re.findall("\d+", pred_q_path)[-1]
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR/eval_VPAir.py:170: SyntaxWarning: invalid escape sequence '\d'
  ref_label = re.findall("\d+", pred_db_paths)[-1]
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR/eval_VPAir.py:190: SyntaxWarning: invalid escape sequence '\d'
  query_label = re.findall("\d+", query_dataset.img_path_list[i])[-1]
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR/eval_VPAir.py:193: SyntaxWarning: invalid escape sequence '\d'
  ref_label = re.findall("\d+", database_dataset.img_path_list[top_k_matches[i][k]])[-1]
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVPR/dataloader/aerialvl.py:243: SyntaxWarning: invalid escape sequence '\d'
  num_th = re.findall("\d+", file)[-1]
/content/drive/MyDrive/mobilegeo_project/repositories/UltraVP


MODEL-SCRIPT IMPORTS
              file import_type                    module                   name
eval_UAV-VisLoc.py        from                       PIL                  Image
eval_UAV-VisLoc.py        from               collections            defaultdict
eval_UAV-VisLoc.py      import                       cv2                    cv2
eval_UAV-VisLoc.py      import                      glob                   glob
eval_UAV-VisLoc.py      import                      math                   math
eval_UAV-VisLoc.py        from models.aggregators.se2gem                 se2gem
eval_UAV-VisLoc.py        from models.backbones.e2resnet               E2ResNet
eval_UAV-VisLoc.py      import                     numpy                  numpy
eval_UAV-VisLoc.py      import                        os                     os
eval_UAV-VisLoc.py      import                     torch                  torch
eval_UAV-VisLoc.py      import                  torch.nn               torch.nn
eval_UAV-VisLoc.py

In [2]:
# ============================================================
# PHASE 3 — CELL 12
# Construct official UltraVPR architecture,
# load checkpoint, validate parameter coverage,
# reproduce preprocessing, and run descriptor smoke test
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import importlib
import sys
import json
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as tvf

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

ULTRAVPR_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "UltraVPR"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

AUDIT_ROOT = (
    PHASE3_ROOT
    / "ultravpr_audit"
)

CHECKPOINT_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "ultravpr_checkpoint_path.txt"
)

DEFAULT_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "ultravpr"
    / "model_best.pth.tar"
)

QUERY_MANIFEST_PATH = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
    / "sues200_test_queries.csv"
)

MODEL_AUDIT_JSON = (
    AUDIT_ROOT
    / "ultravpr_model_load_smoke_test.json"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if CHECKPOINT_CONFIG_PATH.is_file():
    CHECKPOINT_PATH = Path(
        CHECKPOINT_CONFIG_PATH.read_text(
            encoding="utf-8"
        ).strip()
    )
else:
    CHECKPOINT_PATH = (
        DEFAULT_CHECKPOINT_PATH
    )

for required_path in [
    ULTRAVPR_REPO,
    CHECKPOINT_PATH,
    QUERY_MANIFEST_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

# ------------------------------------------------------------
# 2. Compatibility aliases for older research dependencies
# ------------------------------------------------------------

if not hasattr(
    np,
    "float",
):
    np.float = float

if not hasattr(
    np,
    "int",
):
    np.int = int

if not hasattr(
    np,
    "bool",
):
    np.bool = bool

# ------------------------------------------------------------
# 3. Install only missing architecture dependencies
# ------------------------------------------------------------

def ensure_package(
    import_name,
    package_spec,
):
    try:
        module = importlib.import_module(
            import_name
        )

        print(
            f"✅ {import_name} already available"
        )

        return module

    except ImportError:
        print(
            f"Installing {package_spec}..."
        )

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_spec,
            ],
            check=True,
        )

        importlib.invalidate_caches()

        return importlib.import_module(
            import_name
        )


e2cnn_module = ensure_package(
    "e2cnn",
    "e2cnn==0.2.3",
)

mmcv_module = ensure_package(
    "mmcv",
    "mmcv==1.7.2",
)

print("\ne2cnn version:")
print(
    getattr(
        e2cnn_module,
        "__version__",
        "unknown",
    )
)

print("\nmmcv version:")
print(
    getattr(
        mmcv_module,
        "__version__",
        "unknown",
    )
)

# ------------------------------------------------------------
# 4. Import official repository architecture classes
# ------------------------------------------------------------

repo_string = str(
    ULTRAVPR_REPO
)

if repo_string not in sys.path:
    sys.path.insert(
        0,
        repo_string,
    )

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

from models.backbones.e2resnet import E2ResNet
from models.aggregators.se2gem import se2gem

# ------------------------------------------------------------
# 5. Exact architecture from official evaluator
# ------------------------------------------------------------

class UltraVPROfficialArchitecture(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = E2ResNet(
            depth=50,
            out_indices=(3,),
            with_geotensor=True,
            orientation=8,
            middle_channels=2048,
        )

        self.aggregator = se2gem(
            in_dim=256,
            out_dim=256,
        )

    def forward(
        self,
        image_tensor,
    ):
        features = self.backbone(
            image_tensor
        )

        descriptors = self.aggregator(
            features
        )

        return descriptors


ULTRAVPR_DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:")
print(ULTRAVPR_DEVICE)

ultravpr_model = (
    UltraVPROfficialArchitecture()
)

model_state_before = (
    ultravpr_model.state_dict()
)

model_parameter_count = sum(
    parameter.numel()
    for parameter
    in ultravpr_model.parameters()
)

model_state_entry_count = len(
    model_state_before
)

# ------------------------------------------------------------
# 6. Load checkpoint
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

except Exception as safe_error:
    print(
        "\nWeights-only loading failed:"
    )

    print(
        repr(safe_error)
    )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

if not isinstance(
    checkpoint,
    dict,
):
    raise RuntimeError(
        "UltraVPR checkpoint is not a dictionary."
    )

if (
    "state_dict" in checkpoint
    and isinstance(
        checkpoint["state_dict"],
        dict,
    )
):
    checkpoint_state = checkpoint[
        "state_dict"
    ]

    state_dict_source = (
        "checkpoint['state_dict']"
    )

elif (
    "model_state_dict" in checkpoint
    and isinstance(
        checkpoint["model_state_dict"],
        dict,
    )
):
    checkpoint_state = checkpoint[
        "model_state_dict"
    ]

    state_dict_source = (
        "checkpoint['model_state_dict']"
    )

elif all(
    torch.is_tensor(value)
    for value in checkpoint.values()
):
    checkpoint_state = checkpoint
    state_dict_source = (
        "checkpoint_root"
    )

else:
    raise RuntimeError(
        "No model state dictionary was found."
    )

checkpoint_state = {
    str(key): value
    for key, value
    in checkpoint_state.items()
}

# ------------------------------------------------------------
# 7. Check matching keys and tensor shapes
# ------------------------------------------------------------

model_keys = set(
    model_state_before.keys()
)

checkpoint_keys = set(
    checkpoint_state.keys()
)

shared_keys = (
    model_keys
    & checkpoint_keys
)

shape_mismatch_records = []

shape_compatible_keys = []

for key in sorted(
    shared_keys
):
    model_shape = tuple(
        model_state_before[
            key
        ].shape
    )

    checkpoint_shape = tuple(
        checkpoint_state[
            key
        ].shape
    )

    if (
        model_shape
        == checkpoint_shape
    ):
        shape_compatible_keys.append(
            key
        )

    else:
        shape_mismatch_records.append({
            "key": key,
            "model_shape": (
                model_shape
            ),
            "checkpoint_shape": (
                checkpoint_shape
            ),
        })

if shape_mismatch_records:
    print(
        "\nShape mismatches:"
    )

    for record in (
        shape_mismatch_records[:20]
    ):
        print(record)

    raise RuntimeError(
        "Checkpoint contains tensor-shape "
        "mismatches."
    )

matched_parameter_values = sum(
    model_state_before[
        key
    ].numel()
    for key in shape_compatible_keys
)

total_model_state_values = sum(
    tensor.numel()
    for tensor
    in model_state_before.values()
)

parameter_coverage_percent = (
    100.0
    * matched_parameter_values
    / total_model_state_values
)

# ------------------------------------------------------------
# 8. Official loading behavior
# ------------------------------------------------------------

load_result = (
    ultravpr_model.load_state_dict(
        checkpoint_state,
        strict=False,
    )
)

missing_keys = list(
    load_result.missing_keys
)

unexpected_keys = list(
    load_result.unexpected_keys
)

if parameter_coverage_percent < 95.0:
    raise RuntimeError(
        "Checkpoint-to-model coverage is too low: "
        f"{parameter_coverage_percent:.2f}%"
    )

ultravpr_model = (
    ultravpr_model
    .to(ULTRAVPR_DEVICE)
    .eval()
)

# ------------------------------------------------------------
# 9. Exact official evaluation preprocessing
# ------------------------------------------------------------

ULTRAVPR_INPUT_HEIGHT = 300
ULTRAVPR_INPUT_WIDTH = 400
ULTRAVPR_DESCRIPTOR_DIM = 256

ultravpr_preprocess = tvf.Compose([
    tvf.ToTensor(),

    tvf.Normalize(
        mean=[
            0.485,
            0.456,
            0.406,
        ],
        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),

    tvf.Resize(
        (
            ULTRAVPR_INPUT_HEIGHT,
            ULTRAVPR_INPUT_WIDTH,
        ),
        interpolation=(
            tvf.InterpolationMode.BICUBIC
        ),
        antialias=True,
    ),
])

# ------------------------------------------------------------
# 10. Find one real test image
# ------------------------------------------------------------

query_df = pd.read_csv(
    QUERY_MANIFEST_PATH
)

path_candidates = [
    "absolute_path",
    "image_path",
    "query_path",
    "path",
    "filepath",
    "file_path",
]

query_path_column = next(
    (
        column
        for column in path_candidates
        if column in query_df.columns
    ),
    None,
)

if query_path_column is None:
    raise KeyError(
        "No query image-path column was found.\n"
        f"Available columns: "
        f"{query_df.columns.tolist()}"
    )

smoke_image_path = Path(
    str(
        query_df.iloc[0][
            query_path_column
        ]
    )
)

if not smoke_image_path.is_file():
    raise FileNotFoundError(
        smoke_image_path
    )

with Image.open(
    smoke_image_path
) as image:
    smoke_image = image.convert(
        "RGB"
    )

smoke_tensor = (
    ultravpr_preprocess(
        smoke_image
    )
    .unsqueeze(0)
    .to(
        ULTRAVPR_DEVICE,
        non_blocking=True,
    )
)

expected_input_shape = (
    1,
    3,
    ULTRAVPR_INPUT_HEIGHT,
    ULTRAVPR_INPUT_WIDTH,
)

if tuple(
    smoke_tensor.shape
) != expected_input_shape:
    raise RuntimeError(
        "Unexpected preprocessed input shape: "
        f"{tuple(smoke_tensor.shape)}"
    )

# ------------------------------------------------------------
# 11. Descriptor smoke test
# ------------------------------------------------------------

with torch.inference_mode():

    for _ in range(3):
        _ = ultravpr_model(
            smoke_tensor
        )

    if (
        ULTRAVPR_DEVICE.type
        == "cuda"
    ):
        torch.cuda.synchronize()

    inference_times_ms = []

    raw_descriptor = None

    for _ in range(10):
        start_time = (
            time.perf_counter()
        )

        raw_descriptor = (
            ultravpr_model(
                smoke_tensor
            )
        )

        if (
            ULTRAVPR_DEVICE.type
            == "cuda"
        ):
            torch.cuda.synchronize()

        inference_times_ms.append(
            (
                time.perf_counter()
                - start_time
            )
            * 1000.0
        )

if isinstance(
    raw_descriptor,
    (
        tuple,
        list,
    ),
):
    if len(raw_descriptor) != 1:
        raise RuntimeError(
            "UltraVPR returned multiple outputs."
        )

    raw_descriptor = (
        raw_descriptor[0]
    )

if not torch.is_tensor(
    raw_descriptor
):
    raise RuntimeError(
        "UltraVPR output is not a tensor."
    )

descriptor_shape = tuple(
    raw_descriptor.shape
)

if descriptor_shape != (
    1,
    ULTRAVPR_DESCRIPTOR_DIM,
):
    raise RuntimeError(
        "Unexpected UltraVPR descriptor shape: "
        f"{descriptor_shape}"
    )

descriptor_finite = bool(
    torch.isfinite(
        raw_descriptor
    ).all().item()
)

if not descriptor_finite:
    raise RuntimeError(
        "UltraVPR descriptor contains "
        "NaN or infinity."
    )

raw_descriptor_norm = float(
    raw_descriptor
    .float()
    .norm(
        p=2,
        dim=1,
    )
    .item()
)

normalized_descriptor = (
    F.normalize(
        raw_descriptor.float(),
        p=2,
        dim=1,
    )
)

normalized_descriptor_norm = float(
    normalized_descriptor
    .norm(
        p=2,
        dim=1,
    )
    .item()
)

median_inference_ms = float(
    np.median(
        inference_times_ms
    )
)

mean_inference_ms = float(
    np.mean(
        inference_times_ms
    )
)

# ------------------------------------------------------------
# 12. Save model-load audit
# ------------------------------------------------------------

audit_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "UltraVPR",

    "repository_commit": (
        "e8456de13661e7d7584396c9ac3610ec020d0aaf"
    ),

    "architecture": {
        "backbone": (
            "E2ResNet"
        ),
        "depth": 50,
        "out_indices": [
            3
        ],
        "with_geotensor": True,
        "orientation": 8,
        "middle_channels": 2048,
        "aggregator": (
            "se2gem"
        ),
        "aggregator_in_dim": 256,
        "aggregator_out_dim": 256,
        "descriptor_dimension": 256,
    },

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "state_dict_source": (
        state_dict_source
    ),

    "official_loading_behavior": (
        "STRICT_FALSE"
    ),

    "model_state_entries": int(
        model_state_entry_count
    ),

    "checkpoint_state_entries": int(
        len(checkpoint_state)
    ),

    "shape_compatible_shared_keys": int(
        len(
            shape_compatible_keys
        )
    ),

    "parameter_coverage_percent": float(
        parameter_coverage_percent
    ),

    "missing_keys": (
        missing_keys
    ),

    "unexpected_keys": (
        unexpected_keys
    ),

    "shape_mismatches": (
        shape_mismatch_records
    ),

    "parameter_count": int(
        model_parameter_count
    ),

    "preprocessing": {
        "operation_order": [
            "ToTensor",
            "Normalize",
            "Resize",
        ],
        "mean": [
            0.485,
            0.456,
            0.406,
        ],
        "std": [
            0.229,
            0.224,
            0.225,
        ],
        "resize_height": 300,
        "resize_width": 400,
        "interpolation": (
            "BICUBIC"
        ),
    },

    "smoke_test": {
        "image_path": str(
            smoke_image_path
        ),
        "input_shape": list(
            smoke_tensor.shape
        ),
        "descriptor_shape": list(
            descriptor_shape
        ),
        "raw_descriptor_norm": (
            raw_descriptor_norm
        ),
        "normalized_descriptor_norm": (
            normalized_descriptor_norm
        ),
        "descriptor_finite": (
            descriptor_finite
        ),
        "median_batch1_inference_ms": (
            median_inference_ms
        ),
        "mean_batch1_inference_ms": (
            mean_inference_ms
        ),
        "timing_label": (
            "COLAB_BATCH1_SMOKE_TEST_"
            "NOT_DEPLOYMENT_LATENCY"
        ),
    },

    "provenance_label": (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "official_ultravpr_sues200_result": False,

    "common_benchmark_ready": True,
}

MODEL_AUDIT_JSON.write_text(
    json.dumps(
        audit_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 13. Update checkpoint audit
# ------------------------------------------------------------

checkpoint_audit_path = (
    AUDIT_ROOT
    / "ultravpr_checkpoint_audit.json"
)

if checkpoint_audit_path.is_file():

    checkpoint_audit = json.loads(
        checkpoint_audit_path.read_text(
            encoding="utf-8"
        )
    )

    checkpoint_audit[
        "strict_model_loading_validated"
    ] = True

    checkpoint_audit[
        "official_non_strict_loading_validated"
    ] = True

    checkpoint_audit[
        "parameter_coverage_percent"
    ] = parameter_coverage_percent

    checkpoint_audit[
        "descriptor_dimension"
    ] = (
        ULTRAVPR_DESCRIPTOR_DIM
    )

    checkpoint_audit[
        "model_load_audit"
    ] = str(
        MODEL_AUDIT_JSON
    )

    checkpoint_audit_path.write_text(
        json.dumps(
            checkpoint_audit,
            indent=2,
        ),
        encoding="utf-8",
    )

# ------------------------------------------------------------
# 14. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("✅ ULTRAVPR MODEL LOAD AND SMOKE TEST COMPLETE")
print("=" * 78)

print("\nArchitecture:")
print(
    "E2ResNet-50, orientation=8, "
    "se2gem 256-D"
)

print("\nOfficial checkpoint loading:")
print("strict=False")

print("\nParameter coverage:")
print(
    f"{parameter_coverage_percent:.4f}%"
)

print("\nMissing keys:")
print(len(missing_keys))

for key in missing_keys[:30]:
    print("-", key)

print("\nUnexpected keys:")
print(len(unexpected_keys))

for key in unexpected_keys[:30]:
    print("-", key)

print("\nModel parameters:")
print(
    f"{model_parameter_count:,}"
)

print("\nInput shape:")
print(
    tuple(
        smoke_tensor.shape
    )
)

print("\nDescriptor shape:")
print(descriptor_shape)

print("\nRaw descriptor norm:")
print(
    f"{raw_descriptor_norm:.6f}"
)

print("\nNormalized descriptor norm:")
print(
    f"{normalized_descriptor_norm:.6f}"
)

print("\nDescriptor finite:")
print(descriptor_finite)

print("\nMedian batch-one inference:")
print(
    f"{median_inference_ms:.3f} ms"
)

print("\nTiming interpretation:")
print(
    "COLAB_BATCH1_SMOKE_TEST_"
    "NOT_DEPLOYMENT_LATENCY"
)

print("\nAudit JSON:")
print(MODEL_AUDIT_JSON)

print("\nNext action:")
print(
    "Extract UltraVPR descriptors for the same "
    "8,000 queries and 40 gallery images."
)

Installing e2cnn==0.2.3...
Installing mmcv==1.7.2...


/usr/local/lib/python3.12/dist-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(



e2cnn version:
0.2.3

mmcv version:
1.7.2

Device:
cuda

✅ ULTRAVPR MODEL LOAD AND SMOKE TEST COMPLETE

Architecture:
E2ResNet-50, orientation=8, se2gem 256-D

Official checkpoint loading:
strict=False

Parameter coverage:
100.0000%

Missing keys:
0

Unexpected keys:
1
- upscale.centroids

Model parameters:
2,846,897

Input shape:
(1, 3, 300, 400)

Descriptor shape:
(1, 256)

Raw descriptor norm:
1.000000

Normalized descriptor norm:
1.000000

Descriptor finite:
True

Median batch-one inference:
29.442 ms

Timing interpretation:
COLAB_BATCH1_SMOKE_TEST_NOT_DEPLOYMENT_LATENCY

Audit JSON:
/content/drive/MyDrive/mobilegeo_project/results/common_benchmark/ultravpr_audit/ultravpr_model_load_smoke_test.json

Next action:
Extract UltraVPR descriptors for the same 8,000 queries and 40 gallery images.


In [3]:
# ============================================================
# PHASE 3 — CELL 13
# Extract UltraVPR descriptors for common SUES-200 split
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import sys
import time
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as tvf

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

ULTRAVPR_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "UltraVPR"
)

CHECKPOINT_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "ultravpr_checkpoint_path.txt"
)

CHECKPOINT_PATH = Path(
    CHECKPOINT_CONFIG_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

QUERY_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "sues200_test_queries.csv"
)

GALLERY_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "sues200_test_gallery.csv"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

EMBEDDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

QUERY_OUTPUT_PATH = (
    EMBEDDING_ROOT
    / "ultravpr_sues200_test_query_embeddings.npz"
)

GALLERY_OUTPUT_PATH = (
    EMBEDDING_ROOT
    / "ultravpr_sues200_test_gallery_embeddings.npz"
)

EXTRACTION_REPORT_PATH = (
    REPORT_ROOT
    / "ultravpr_descriptor_extraction_report.json"
)

for path in [
    ULTRAVPR_REPO,
    CHECKPOINT_PATH,
    QUERY_MANIFEST_PATH,
    GALLERY_MANIFEST_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 2. Runtime configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

IMAGE_HEIGHT = 300
IMAGE_WIDTH = 400
DESCRIPTOR_DIM = 256

# Conservative batch size for E2ResNet.
BATCH_SIZE = 8
NUM_WORKERS = 2

print("Device:")
print(DEVICE)

print("\nBatch size:")
print(BATCH_SIZE)

# ------------------------------------------------------------
# 3. Compatibility aliases
# ------------------------------------------------------------

if not hasattr(np, "float"):
    np.float = float

if not hasattr(np, "int"):
    np.int = int

if not hasattr(np, "bool"):
    np.bool = bool

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

# ------------------------------------------------------------
# 4. Import official UltraVPR architecture
# ------------------------------------------------------------

repo_string = str(
    ULTRAVPR_REPO
)

if repo_string not in sys.path:
    sys.path.insert(
        0,
        repo_string,
    )

from models.backbones.e2resnet import E2ResNet
from models.aggregators.se2gem import se2gem

# ------------------------------------------------------------
# 5. Build exact model
# ------------------------------------------------------------

class UltraVPRModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = E2ResNet(
            depth=50,
            out_indices=(3,),
            with_geotensor=True,
            orientation=8,
            middle_channels=2048,
        )

        self.aggregator = se2gem(
            in_dim=256,
            out_dim=256,
        )

    def forward(
        self,
        images,
    ):
        features = self.backbone(
            images
        )

        descriptors = self.aggregator(
            features
        )

        return descriptors


model = UltraVPRModel()

# ------------------------------------------------------------
# 6. Load official checkpoint
# ------------------------------------------------------------

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

except TypeError:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )

except Exception:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

if "state_dict" in checkpoint:
    state_dict = checkpoint[
        "state_dict"
    ]

elif "model_state_dict" in checkpoint:
    state_dict = checkpoint[
        "model_state_dict"
    ]

else:
    state_dict = checkpoint

load_result = model.load_state_dict(
    state_dict,
    strict=False,
)

print("\nCheckpoint loading:")
print("Successful")

print("\nMissing keys:")
print(len(load_result.missing_keys))

print("\nUnexpected keys:")
print(len(load_result.unexpected_keys))

model = (
    model
    .to(DEVICE)
    .eval()
)

# ------------------------------------------------------------
# 7. Official evaluation preprocessing
# ------------------------------------------------------------

preprocess = tvf.Compose([
    tvf.ToTensor(),

    tvf.Normalize(
        mean=[
            0.485,
            0.456,
            0.406,
        ],
        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),

    tvf.Resize(
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
        interpolation=(
            tvf.InterpolationMode.BICUBIC
        ),
        antialias=True,
    ),
])

# ------------------------------------------------------------
# 8. Load manifests
# ------------------------------------------------------------

query_df = pd.read_csv(
    QUERY_MANIFEST_PATH,
    dtype=str,
)

gallery_df = pd.read_csv(
    GALLERY_MANIFEST_PATH,
    dtype=str,
)

def find_column(
    dataframe,
    candidates,
    description,
):
    for column in candidates:
        if column in dataframe.columns:
            return column

    raise KeyError(
        f"Could not find {description}.\n"
        f"Columns: {dataframe.columns.tolist()}"
    )


query_path_column = find_column(
    query_df,
    [
        "absolute_path",
        "query_path",
        "image_path",
        "path",
        "filepath",
        "file_path",
    ],
    "query path column",
)

gallery_path_column = find_column(
    gallery_df,
    [
        "absolute_path",
        "gallery_path",
        "image_path",
        "path",
        "filepath",
        "file_path",
    ],
    "gallery path column",
)

query_frame_column = find_column(
    query_df,
    [
        "frame_id",
        "query_id",
        "image_id",
        "id",
    ],
    "query frame ID",
)

query_location_column = find_column(
    query_df,
    [
        "location_id",
        "query_location_id",
        "class_id",
    ],
    "query location ID",
)

query_height_column = find_column(
    query_df,
    [
        "nominal_height",
        "height",
        "altitude",
    ],
    "query nominal height",
)

positive_tile_column = find_column(
    query_df,
    [
        "positive_tile_id",
        "tile_id",
        "gallery_tile_id",
    ],
    "positive tile ID",
)

gallery_tile_column = find_column(
    gallery_df,
    [
        "tile_id",
        "gallery_tile_id",
        "image_id",
        "id",
    ],
    "gallery tile ID",
)

gallery_location_column = find_column(
    gallery_df,
    [
        "location_id",
        "gallery_location_id",
        "class_id",
    ],
    "gallery location ID",
)

if len(query_df) != 8000:
    raise RuntimeError(
        f"Expected 8,000 queries, found {len(query_df)}."
    )

if len(gallery_df) != 40:
    raise RuntimeError(
        f"Expected 40 gallery images, found {len(gallery_df)}."
    )

print("\nQuery images:")
print(len(query_df))

print("\nGallery images:")
print(len(gallery_df))

# ------------------------------------------------------------
# 9. Dataset
# ------------------------------------------------------------

class UltraVPRImageDataset(Dataset):

    def __init__(
        self,
        dataframe,
        path_column,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.path_column = (
            path_column
        )

    def __len__(self):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index,
    ):
        image_path = Path(
            str(
                self.dataframe.iloc[
                    index
                ][
                    self.path_column
                ]
            )
        )

        if not image_path.is_file():
            raise FileNotFoundError(
                image_path
            )

        with Image.open(
            image_path
        ) as image:

            image = image.convert(
                "RGB"
            )

            image_tensor = preprocess(
                image
            )

        return {
            "image": image_tensor,
            "index": index,
        }

# ------------------------------------------------------------
# 10. Descriptor extraction
# ------------------------------------------------------------

def extract_descriptors(
    dataframe,
    path_column,
    label,
):

    dataset = UltraVPRImageDataset(
        dataframe=dataframe,
        path_column=path_column,
    )

    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
    )

    descriptor_batches = []
    output_indices = []

    inference_seconds = 0.0

    end_to_end_start = (
        time.perf_counter()
    )

    with torch.inference_mode():

        for batch_number, batch in enumerate(
            dataloader,
            start=1,
        ):

            images = batch[
                "image"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            indices = batch[
                "index"
            ].numpy()

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            inference_start = (
                time.perf_counter()
            )

            raw_descriptors = model(
                images
            )

            if isinstance(
                raw_descriptors,
                (
                    tuple,
                    list,
                ),
            ):
                if len(raw_descriptors) != 1:
                    raise RuntimeError(
                        "Unexpected multiple model outputs."
                    )

                raw_descriptors = (
                    raw_descriptors[0]
                )

            descriptors = F.normalize(
                raw_descriptors.float(),
                p=2,
                dim=1,
            )

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()

            inference_seconds += (
                time.perf_counter()
                - inference_start
            )

            descriptor_batches.append(
                descriptors
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            output_indices.extend(
                indices.tolist()
            )

            processed = min(
                batch_number
                * BATCH_SIZE,
                len(dataset),
            )

            if (
                batch_number == 1
                or batch_number % 100 == 0
                or processed == len(dataset)
            ):
                print(
                    f"{label}: "
                    f"{processed}/{len(dataset)}"
                )

    end_to_end_seconds = (
        time.perf_counter()
        - end_to_end_start
    )

    descriptors = np.concatenate(
        descriptor_batches,
        axis=0,
    )

    output_indices = np.asarray(
        output_indices,
        dtype=np.int64,
    )

    # Restore manifest order defensively.
    order = np.argsort(
        output_indices
    )

    descriptors = descriptors[
        order
    ]

    output_indices = (
        output_indices[
            order
        ]
    )

    expected_indices = np.arange(
        len(dataframe),
        dtype=np.int64,
    )

    if not np.array_equal(
        output_indices,
        expected_indices,
    ):
        raise RuntimeError(
            f"{label} ordering mismatch."
        )

    if descriptors.shape != (
        len(dataframe),
        DESCRIPTOR_DIM,
    ):
        raise RuntimeError(
            f"Unexpected {label} descriptor shape: "
            f"{descriptors.shape}"
        )

    if not np.isfinite(
        descriptors
    ).all():
        raise RuntimeError(
            f"{label} descriptors contain "
            "NaN or infinity."
        )

    descriptor_norms = (
        np.linalg.norm(
            descriptors,
            axis=1,
        )
    )

    if not np.allclose(
        descriptor_norms,
        1.0,
        atol=1e-4,
    ):
        raise RuntimeError(
            f"{label} descriptors are not "
            "L2-normalized."
        )

    timing = {
        "image_count": int(
            len(dataframe)
        ),

        "batch_size": int(
            BATCH_SIZE
        ),

        "inference_seconds": float(
            inference_seconds
        ),

        "inference_ms_per_image": float(
            1000.0
            * inference_seconds
            / len(dataframe)
        ),

        "end_to_end_seconds": float(
            end_to_end_seconds
        ),

        "end_to_end_ms_per_image": float(
            1000.0
            * end_to_end_seconds
            / len(dataframe)
        ),
    }

    return (
        descriptors,
        timing,
    )

# ------------------------------------------------------------
# 11. CUDA warm-up
# ------------------------------------------------------------

if DEVICE.type == "cuda":

    warmup = torch.zeros(
        (
            1,
            3,
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    with torch.inference_mode():
        for _ in range(3):
            _ = model(
                warmup
            )

    torch.cuda.synchronize()

    del warmup

# ------------------------------------------------------------
# 12. Extract gallery
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXTRACTING ULTRAVPR GALLERY DESCRIPTORS")
print("=" * 72)

gallery_descriptors, gallery_timing = (
    extract_descriptors(
        dataframe=gallery_df,
        path_column=gallery_path_column,
        label="Gallery",
    )
)

# ------------------------------------------------------------
# 13. Extract queries
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXTRACTING ULTRAVPR QUERY DESCRIPTORS")
print("=" * 72)

query_descriptors, query_timing = (
    extract_descriptors(
        dataframe=query_df,
        path_column=query_path_column,
        label="Queries",
    )
)

# ------------------------------------------------------------
# 14. Save pickle-free NPZ files
# ------------------------------------------------------------

np.savez_compressed(
    GALLERY_OUTPUT_PATH,

    descriptors=(
        gallery_descriptors
    ),

    tile_ids=np.asarray(
        gallery_df[
            gallery_tile_column
        ].astype(str),
        dtype=np.str_,
    ),

    location_ids=np.asarray(
        gallery_df[
            gallery_location_column
        ].astype(str),
        dtype=np.str_,
    ),

    absolute_paths=np.asarray(
        gallery_df[
            gallery_path_column
        ].astype(str),
        dtype=np.str_,
    ),
)

np.savez_compressed(
    QUERY_OUTPUT_PATH,

    descriptors=(
        query_descriptors
    ),

    frame_ids=np.asarray(
        query_df[
            query_frame_column
        ].astype(str),
        dtype=np.str_,
    ),

    location_ids=np.asarray(
        query_df[
            query_location_column
        ].astype(str),
        dtype=np.str_,
    ),

    nominal_heights=np.asarray(
        query_df[
            query_height_column
        ].astype(str),
        dtype=np.str_,
    ),

    positive_tile_ids=np.asarray(
        query_df[
            positive_tile_column
        ].astype(str),
        dtype=np.str_,
    ),

    absolute_paths=np.asarray(
        query_df[
            query_path_column
        ].astype(str),
        dtype=np.str_,
    ),
)

# ------------------------------------------------------------
# 15. Reload with allow_pickle=False
# ------------------------------------------------------------

gallery_validation = np.load(
    GALLERY_OUTPUT_PATH,
    allow_pickle=False,
)

query_validation = np.load(
    QUERY_OUTPUT_PATH,
    allow_pickle=False,
)

if gallery_validation[
    "descriptors"
].shape != (
    40,
    DESCRIPTOR_DIM,
):
    raise RuntimeError(
        "Saved gallery descriptor shape invalid."
    )

if query_validation[
    "descriptors"
].shape != (
    8000,
    DESCRIPTOR_DIM,
):
    raise RuntimeError(
        "Saved query descriptor shape invalid."
    )

gallery_validation.close()
query_validation.close()

# ------------------------------------------------------------
# 16. Save extraction report
# ------------------------------------------------------------

report = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "UltraVPR",

    "repository_commit": (
        "e8456de13661e7d7584396c9ac3610ec020d0aaf"
    ),

    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),

    "checkpoint_loading": (
        "OFFICIAL_STRICT_FALSE"
    ),

    "architecture": (
        "E2ResNet50_orientation8_se2gem"
    ),

    "descriptor_dimension": (
        DESCRIPTOR_DIM
    ),

    "input_size": [
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
    ],

    "descriptor_normalization": (
        "L2"
    ),

    "preprocessing": {
        "order": [
            "ToTensor",
            "ImageNet_Normalize",
            "Resize",
        ],

        "mean": [
            0.485,
            0.456,
            0.406,
        ],

        "std": [
            0.229,
            0.224,
            0.225,
        ],

        "interpolation": (
            "BICUBIC"
        ),
    },

    "provenance_label": (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "official_ultravpr_sues200_result": False,

    "benchmark_protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "gallery": {
        "count": 40,
        "descriptor_shape": list(
            gallery_descriptors.shape
        ),
        "embedding_file": str(
            GALLERY_OUTPUT_PATH
        ),
        "timing": (
            gallery_timing
        ),
    },

    "queries": {
        "count": 8000,
        "descriptor_shape": list(
            query_descriptors.shape
        ),
        "embedding_file": str(
            QUERY_OUTPUT_PATH
        ),
        "timing": (
            query_timing
        ),
    },

    "timing_interpretation": (
        "BATCHED_COLAB_RUNTIME_MEASUREMENT_"
        "NOT_JETSON_DEPLOYMENT_LATENCY"
    ),
}

EXTRACTION_REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 17. Final output
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("✅ ULTRAVPR DESCRIPTOR EXTRACTION COMPLETE")
print("=" * 78)

print("\nGallery descriptor shape:")
print(
    gallery_descriptors.shape
)

print("\nQuery descriptor shape:")
print(
    query_descriptors.shape
)

print("\nGallery inference:")
print(
    f"{gallery_timing['inference_ms_per_image']:.3f} "
    "ms/image"
)

print("\nGallery end-to-end:")
print(
    f"{gallery_timing['end_to_end_ms_per_image']:.3f} "
    "ms/image"
)

print("\nQuery inference:")
print(
    f"{query_timing['inference_ms_per_image']:.3f} "
    "ms/image"
)

print("\nQuery end-to-end:")
print(
    f"{query_timing['end_to_end_ms_per_image']:.3f} "
    "ms/image"
)

print("\nGallery embeddings:")
print(
    GALLERY_OUTPUT_PATH
)

print("\nQuery embeddings:")
print(
    QUERY_OUTPUT_PATH
)

print("\nPickle-free loading:")
print("Successful")

print("\nExtraction report:")
print(
    EXTRACTION_REPORT_PATH
)

print("\nNext action:")
print(
    "Evaluate UltraVPR on the same 8,000-query "
    "and 40-gallery common retrieval protocol."
)

Device:
cuda

Batch size:
8

Checkpoint loading:
Successful

Missing keys:
0

Unexpected keys:
1

Query images:
8000

Gallery images:
40

EXTRACTING ULTRAVPR GALLERY DESCRIPTORS
Gallery: 8/40
Gallery: 40/40

EXTRACTING ULTRAVPR QUERY DESCRIPTORS
Queries: 8/8000
Queries: 800/8000
Queries: 1600/8000
Queries: 2400/8000
Queries: 3200/8000
Queries: 4000/8000
Queries: 4800/8000
Queries: 5600/8000
Queries: 6400/8000
Queries: 7200/8000
Queries: 8000/8000

✅ ULTRAVPR DESCRIPTOR EXTRACTION COMPLETE

Gallery descriptor shape:
(40, 256)

Query descriptor shape:
(8000, 256)

Gallery inference:
13.752 ms/image

Gallery end-to-end:
361.729 ms/image

Query inference:
12.392 ms/image

Query end-to-end:
154.637 ms/image

Gallery embeddings:
/content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/ultravpr_sues200_test_gallery_embeddings.npz

Query embeddings:
/content/drive/MyDrive/mobilegeo_project/results/common_benchmark/embeddings/ultravpr_sues200_test_query_embeddings.npz

Pickl

In [4]:
# ============================================================
# PHASE 3 — CELL 14
# Evaluate UltraVPR on the common SUES-200 retrieval protocol
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time
import json

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

RANKING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

GALLERY_EMBEDDING_PATH = (
    EMBEDDING_ROOT
    / "ultravpr_sues200_test_gallery_embeddings.npz"
)

QUERY_EMBEDDING_PATH = (
    EMBEDDING_ROOT
    / "ultravpr_sues200_test_query_embeddings.npz"
)

METRICS_CSV = (
    REPORT_ROOT
    / "ultravpr_location_disjoint_metrics.csv"
)

METRICS_JSON = (
    REPORT_ROOT
    / "ultravpr_location_disjoint_metrics.json"
)

QUERY_RESULTS_CSV = (
    RANKING_ROOT
    / "ultravpr_query_results.csv"
)

COMPLETE_RANKINGS_CSV = (
    RANKING_ROOT
    / "ultravpr_complete_rankings.csv"
)

PER_LOCATION_METRICS_CSV = (
    REPORT_ROOT
    / "ultravpr_per_location_metrics.csv"
)

for path in [
    GALLERY_EMBEDDING_PATH,
    QUERY_EMBEDDING_PATH,
]:
    if not path.is_file():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 2. Load embeddings
# ------------------------------------------------------------

gallery_data = np.load(
    GALLERY_EMBEDDING_PATH,
    allow_pickle=False,
)

query_data = np.load(
    QUERY_EMBEDDING_PATH,
    allow_pickle=False,
)

gallery_descriptors = (
    gallery_data["descriptors"]
    .astype(
        np.float32,
        copy=False,
    )
)

query_descriptors = (
    query_data["descriptors"]
    .astype(
        np.float32,
        copy=False,
    )
)

gallery_tile_ids = (
    gallery_data["tile_ids"]
    .astype(str)
)

gallery_location_ids = (
    gallery_data["location_ids"]
    .astype(str)
)

gallery_paths = (
    gallery_data["absolute_paths"]
    .astype(str)
)

query_frame_ids = (
    query_data["frame_ids"]
    .astype(str)
)

query_location_ids = (
    query_data["location_ids"]
    .astype(str)
)

query_nominal_heights = (
    query_data["nominal_heights"]
    .astype(str)
)

query_positive_tile_ids = (
    query_data["positive_tile_ids"]
    .astype(str)
)

query_paths = (
    query_data["absolute_paths"]
    .astype(str)
)

gallery_data.close()
query_data.close()

print("Gallery descriptors:")
print(gallery_descriptors.shape)

print("\nQuery descriptors:")
print(query_descriptors.shape)

# ------------------------------------------------------------
# 3. Validate
# ------------------------------------------------------------

if gallery_descriptors.shape != (
    40,
    256,
):
    raise RuntimeError(
        f"Unexpected gallery shape: "
        f"{gallery_descriptors.shape}"
    )

if query_descriptors.shape != (
    8000,
    256,
):
    raise RuntimeError(
        f"Unexpected query shape: "
        f"{query_descriptors.shape}"
    )

if len(
    np.unique(
        gallery_tile_ids
    )
) != 40:
    raise RuntimeError(
        "Gallery tile IDs are not unique."
    )

if not np.isfinite(
    gallery_descriptors
).all():
    raise RuntimeError(
        "Gallery descriptors contain NaN/Inf."
    )

if not np.isfinite(
    query_descriptors
).all():
    raise RuntimeError(
        "Query descriptors contain NaN/Inf."
    )

# Defensive L2 normalization.
gallery_descriptors /= np.clip(
    np.linalg.norm(
        gallery_descriptors,
        axis=1,
        keepdims=True,
    ),
    1e-12,
    None,
)

query_descriptors /= np.clip(
    np.linalg.norm(
        query_descriptors,
        axis=1,
        keepdims=True,
    ),
    1e-12,
    None,
)

# ------------------------------------------------------------
# 4. Resolve positives
# ------------------------------------------------------------

tile_to_gallery_index = {
    tile_id: index
    for index, tile_id
    in enumerate(
        gallery_tile_ids
    )
}

missing_positive_tiles = sorted(
    set(
        query_positive_tile_ids
    )
    - set(
        gallery_tile_ids
    )
)

if missing_positive_tiles:
    raise RuntimeError(
        "Missing positive tiles:\n"
        f"{missing_positive_tiles[:20]}"
    )

positive_gallery_indices = np.asarray(
    [
        tile_to_gallery_index[
            tile_id
        ]
        for tile_id
        in query_positive_tile_ids
    ],
    dtype=np.int64,
)

# ------------------------------------------------------------
# 5. Exact cosine retrieval
# ------------------------------------------------------------

search_start = time.perf_counter()

similarity_matrix = (
    query_descriptors
    @ gallery_descriptors.T
)

ranking_indices = np.argsort(
    -similarity_matrix,
    axis=1,
    kind="stable",
)

ranked_scores = np.take_along_axis(
    similarity_matrix,
    ranking_indices,
    axis=1,
)

search_seconds = (
    time.perf_counter()
    - search_start
)

search_ms_per_query = (
    search_seconds
    * 1000.0
    / len(
        query_descriptors
    )
)

# ------------------------------------------------------------
# 6. Positive ranks
# ------------------------------------------------------------

positive_matches = (
    ranking_indices
    == positive_gallery_indices[
        :,
        None,
    ]
)

if not positive_matches.any(
    axis=1
).all():
    raise RuntimeError(
        "A query has no positive match."
    )

positive_ranks = (
    np.argmax(
        positive_matches,
        axis=1,
    )
    + 1
)

top1_indices = (
    ranking_indices[:, 0]
)

top2_indices = (
    ranking_indices[:, 1]
)

top1_scores = (
    ranked_scores[:, 0]
)

top2_scores = (
    ranked_scores[:, 1]
)

top1_top2_margins = (
    top1_scores
    - top2_scores
)

top1_tile_ids = (
    gallery_tile_ids[
        top1_indices
    ]
)

top2_tile_ids = (
    gallery_tile_ids[
        top2_indices
    ]
)

top1_location_ids = (
    gallery_location_ids[
        top1_indices
    ]
)

top2_location_ids = (
    gallery_location_ids[
        top2_indices
    ]
)

top1_correct = (
    top1_indices
    == positive_gallery_indices
)

# ------------------------------------------------------------
# 7. Metric helper
# ------------------------------------------------------------

def calculate_metrics(
    ranks,
    margins,
):
    ranks = np.asarray(
        ranks,
        dtype=np.int64,
    )

    margins = np.asarray(
        margins,
        dtype=np.float64,
    )

    return {
        "query_count": int(
            len(ranks)
        ),

        "recall_at_1_percent": float(
            100.0
            * np.mean(
                ranks <= 1
            )
        ),

        "recall_at_5_percent": float(
            100.0
            * np.mean(
                ranks <= 5
            )
        ),

        "recall_at_10_percent": float(
            100.0
            * np.mean(
                ranks <= 10
            )
        ),

        "mean_average_precision_percent": float(
            100.0
            * np.mean(
                1.0 / ranks
            )
        ),

        "mean_positive_rank": float(
            np.mean(
                ranks
            )
        ),

        "median_positive_rank": float(
            np.median(
                ranks
            )
        ),

        "p95_positive_rank": float(
            np.percentile(
                ranks,
                95,
            )
        ),

        "maximum_positive_rank": int(
            np.max(
                ranks
            )
        ),

        "mean_top1_top2_margin": float(
            np.mean(
                margins
            )
        ),

        "median_top1_top2_margin": float(
            np.median(
                margins
            )
        ),
    }

# ------------------------------------------------------------
# 8. Overall + per-height metrics
# ------------------------------------------------------------

metric_records = []

overall_metrics = calculate_metrics(
    positive_ranks,
    top1_top2_margins,
)

metric_records.append({
    "group": "overall",
    "nominal_height": "all",
    **overall_metrics,
})

unique_heights = sorted(
    np.unique(
        query_nominal_heights
    ),
    key=lambda value: int(
        value
    ),
)

for height in unique_heights:

    mask = (
        query_nominal_heights
        == height
    )

    height_metrics = (
        calculate_metrics(
            positive_ranks[
                mask
            ],
            top1_top2_margins[
                mask
            ],
        )
    )

    metric_records.append({
        "group": (
            f"height_{height}"
        ),
        "nominal_height": (
            height
        ),
        **height_metrics,
    })

metrics_df = pd.DataFrame(
    metric_records
)

# ------------------------------------------------------------
# 9. Per-query result table
# ------------------------------------------------------------

query_results_df = pd.DataFrame({
    "frame_id": (
        query_frame_ids
    ),

    "query_location_id": (
        query_location_ids
    ),

    "nominal_height": (
        query_nominal_heights
    ),

    "positive_tile_id": (
        query_positive_tile_ids
    ),

    "positive_rank": (
        positive_ranks
    ),

    "top1_tile_id": (
        top1_tile_ids
    ),

    "top1_location_id": (
        top1_location_ids
    ),

    "top1_score": (
        top1_scores
    ),

    "top2_tile_id": (
        top2_tile_ids
    ),

    "top2_location_id": (
        top2_location_ids
    ),

    "top2_score": (
        top2_scores
    ),

    "top1_top2_margin": (
        top1_top2_margins
    ),

    "top1_correct": (
        top1_correct
    ),

    "absolute_path": (
        query_paths
    ),
})

# ------------------------------------------------------------
# 10. Full 320,000-row rankings
# ------------------------------------------------------------

query_count = len(
    query_frame_ids
)

gallery_count = len(
    gallery_tile_ids
)

flat_gallery_indices = (
    ranking_indices.reshape(
        -1
    )
)

complete_rankings_df = (
    pd.DataFrame({
        "frame_id": np.repeat(
            query_frame_ids,
            gallery_count,
        ),

        "query_location_id": np.repeat(
            query_location_ids,
            gallery_count,
        ),

        "nominal_height": np.repeat(
            query_nominal_heights,
            gallery_count,
        ),

        "positive_tile_id": np.repeat(
            query_positive_tile_ids,
            gallery_count,
        ),

        "rank": np.tile(
            np.arange(
                1,
                gallery_count + 1,
                dtype=np.int16,
            ),
            query_count,
        ),

        "retrieved_tile_id": (
            gallery_tile_ids[
                flat_gallery_indices
            ]
        ),

        "retrieved_location_id": (
            gallery_location_ids[
                flat_gallery_indices
            ]
        ),

        "similarity": (
            ranked_scores.reshape(
                -1
            )
        ),
    })
)

complete_rankings_df[
    "is_positive"
] = (
    complete_rankings_df[
        "retrieved_tile_id"
    ]
    == complete_rankings_df[
        "positive_tile_id"
    ]
)

if len(
    complete_rankings_df
) != 320000:
    raise RuntimeError(
        "Incorrect ranking row count."
    )

# ------------------------------------------------------------
# 11. Per-location metrics
# ------------------------------------------------------------

per_location_records = []

for location_id in sorted(
    np.unique(
        query_location_ids
    )
):
    mask = (
        query_location_ids
        == location_id
    )

    location_metrics = (
        calculate_metrics(
            positive_ranks[
                mask
            ],
            top1_top2_margins[
                mask
            ],
        )
    )

    per_location_records.append({
        "location_id": (
            location_id
        ),
        **location_metrics,
    })

per_location_df = pd.DataFrame(
    per_location_records
)

# ------------------------------------------------------------
# 12. Save CSVs
# ------------------------------------------------------------

metrics_df.to_csv(
    METRICS_CSV,
    index=False,
)

query_results_df.to_csv(
    QUERY_RESULTS_CSV,
    index=False,
)

complete_rankings_df.to_csv(
    COMPLETE_RANKINGS_CSV,
    index=False,
)

per_location_df.to_csv(
    PER_LOCATION_METRICS_CSV,
    index=False,
)

# ------------------------------------------------------------
# 13. Save JSON
# ------------------------------------------------------------

evaluation_record = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": "UltraVPR",

    "repository_commit": (
        "e8456de13661e7d7584396c9ac3610ec020d0aaf"
    ),

    "architecture": (
        "E2ResNet50_orientation8_se2gem"
    ),

    "descriptor_dimension": 256,

    "checkpoint_provenance": (
        "OFFICIAL_ULTRAVPR_CHECKPOINT"
    ),

    "evaluation_provenance_label": (
        "OFFICIAL_ULTRAVPR_CHECKPOINT_"
        "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
    ),

    "official_ultravpr_sues200_result": (
        False
    ),

    "benchmark_protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "query_count": int(
        query_count
    ),

    "gallery_count": int(
        gallery_count
    ),

    "similarity": (
        "COSINE_SIMILARITY_ON_"
        "L2_NORMALIZED_DESCRIPTORS"
    ),

    "ranking": (
        "EXACT_FULL_GALLERY_SORT"
    ),

    "positive_definition": (
        "ONE_MATCHING_SATELLITE_TILE_PER_QUERY"
    ),

    "overall_metrics": (
        overall_metrics
    ),

    "search_timing": {
        "total_seconds": float(
            search_seconds
        ),

        "milliseconds_per_query": float(
            search_ms_per_query
        ),

        "scope": (
            "SIMILARITY_MATRIX_AND_FULL_SORT_ONLY"
        ),

        "descriptor_extraction_excluded": (
            True
        ),
    },

    "files": {
        "metrics_csv": str(
            METRICS_CSV
        ),

        "query_results_csv": str(
            QUERY_RESULTS_CSV
        ),

        "complete_rankings_csv": str(
            COMPLETE_RANKINGS_CSV
        ),

        "per_location_metrics_csv": str(
            PER_LOCATION_METRICS_CSV
        ),
    },

    "reporting_restrictions": [
        (
            "Do not describe this as an official "
            "UltraVPR SUES-200 result."
        ),
        (
            "This is zero-shot transfer to the "
            "project-defined SUES-200 split."
        ),
        (
            "Do not report retrieval as verified UAV pose."
        ),
    ],
}

METRICS_JSON.write_text(
    json.dumps(
        evaluation_record,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 14. Print results
# ------------------------------------------------------------

display_df = metrics_df[
    [
        "group",
        "nominal_height",
        "query_count",
        "recall_at_1_percent",
        "recall_at_5_percent",
        "recall_at_10_percent",
        "mean_average_precision_percent",
        "mean_positive_rank",
    ]
].copy()

print("\n" + "=" * 78)
print("✅ ULTRAVPR COMMON BENCHMARK EVALUATION COMPLETE")
print("=" * 78)

print()

print(
    display_df.to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda value: f"{value:.2f}"
            ),

            "recall_at_5_percent": (
                lambda value: f"{value:.2f}"
            ),

            "recall_at_10_percent": (
                lambda value: f"{value:.2f}"
            ),

            "mean_average_precision_percent": (
                lambda value: f"{value:.2f}"
            ),

            "mean_positive_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nSearch time:")
print(
    f"{search_seconds:.6f} seconds"
)

print("\nSearch time per query:")
print(
    f"{search_ms_per_query:.6f} ms"
)

print("\nTop-1 correct queries:")
print(
    int(
        top1_correct.sum()
    )
)

print("\nTop-1 failures:")
print(
    int(
        (~top1_correct).sum()
    )
)

print("\nMaximum positive rank:")
print(
    int(
        positive_ranks.max()
    )
)

print("\nComplete ranking rows:")
print(
    len(
        complete_rankings_df
    )
)

print("\nMetrics CSV:")
print(METRICS_CSV)

print("\nMetrics JSON:")
print(METRICS_JSON)

print("\nQuery results:")
print(QUERY_RESULTS_CSV)

print("\nComplete rankings:")
print(COMPLETE_RANKINGS_CSV)

print("\nPer-location metrics:")
print(PER_LOCATION_METRICS_CSV)

print("\nNext action:")
print(
    "Build the final three-method common comparison."
)

Gallery descriptors:
(40, 256)

Query descriptors:
(8000, 256)

✅ ULTRAVPR COMMON BENCHMARK EVALUATION COMPLETE

     group nominal_height  query_count recall_at_1_percent recall_at_5_percent recall_at_10_percent mean_average_precision_percent mean_positive_rank
   overall            all         8000               81.94               95.23                98.47                          87.83              1.667
height_150            150         2000               56.35               83.00                94.15                          68.22              3.127
height_200            200         2000               83.15               98.50                99.75                          89.59              1.367
height_250            250         2000               93.85               99.50               100.00                          96.48              1.105
height_300            300         2000               94.40               99.90               100.00                          97.03       

In [5]:
# ============================================================
# PHASE 3 — CELL 15
# Final three-method common benchmark comparison:
# Local reimplementation vs Sample4Geo vs UltraVPR
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

AUDIT_ROOT = (
    PHASE3_ROOT
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. Input metric files
# ------------------------------------------------------------

LOCAL_METRICS_PATH = (
    PHASE2_ROOT
    / "corrected_location_disjoint_metrics.csv"
)

SAMPLE4GEO_METRICS_PATH = (
    REPORT_ROOT
    / "sample4geo_location_disjoint_metrics.csv"
)

ULTRAVPR_METRICS_PATH = (
    REPORT_ROOT
    / "ultravpr_location_disjoint_metrics.csv"
)

LOCAL_QUERY_RESULTS_PATH = (
    PHASE2_ROOT
    / "corrected_query_results.csv"
)

SAMPLE4GEO_QUERY_RESULTS_PATH = (
    RANKING_ROOT
    / "sample4geo_query_results.csv"
)

ULTRAVPR_QUERY_RESULTS_PATH = (
    RANKING_ROOT
    / "ultravpr_query_results.csv"
)

required_paths = [
    LOCAL_METRICS_PATH,
    SAMPLE4GEO_METRICS_PATH,
    ULTRAVPR_METRICS_PATH,
    LOCAL_QUERY_RESULTS_PATH,
    SAMPLE4GEO_QUERY_RESULTS_PATH,
    ULTRAVPR_QUERY_RESULTS_PATH,
]

for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )

# ------------------------------------------------------------
# 3. Output files
# ------------------------------------------------------------

FINAL_METRICS_CSV = (
    REPORT_ROOT
    / "final_three_method_common_metrics.csv"
)

FINAL_HEIGHT_CSV = (
    REPORT_ROOT
    / "final_three_method_height_metrics.csv"
)

FINAL_PAIRED_CSV = (
    RANKING_ROOT
    / "final_three_method_paired_query_results.csv"
)

FINAL_SUMMARY_JSON = (
    REPORT_ROOT
    / "final_three_method_comparison.json"
)

FINAL_REPORT_MD = (
    REPORT_ROOT
    / "final_three_method_comparison_report.md"
)

# ------------------------------------------------------------
# 4. Method metadata
# ------------------------------------------------------------

METHODS = {
    "Local_Reimplementation": {
        "display_name": (
            "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
        ),
        "provenance": (
            "LOCAL_REIMPLEMENTATION"
        ),
        "training_context": (
            "PROJECT_LOCAL_CHECKPOINT"
        ),
        "descriptor_dimension": 2048,
        "metrics_path": (
            LOCAL_METRICS_PATH
        ),
        "query_results_path": (
            LOCAL_QUERY_RESULTS_PATH
        ),
    },

    "Sample4Geo": {
        "display_name": (
            "Sample4Geo"
        ),
        "provenance": (
            "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
            "ZERO_SHOT_TRANSFER_FROM_UNIVERSITY1652"
        ),
        "training_context": (
            "University-1652"
        ),
        "descriptor_dimension": 1024,
        "metrics_path": (
            SAMPLE4GEO_METRICS_PATH
        ),
        "query_results_path": (
            SAMPLE4GEO_QUERY_RESULTS_PATH
        ),
    },

    "UltraVPR": {
        "display_name": (
            "UltraVPR"
        ),
        "provenance": (
            "OFFICIAL_ULTRAVPR_CHECKPOINT_"
            "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
        ),
        "training_context": (
            "OFFICIAL_ULTRAVPR_RELEASE_CHECKPOINT"
        ),
        "descriptor_dimension": 256,
        "metrics_path": (
            ULTRAVPR_METRICS_PATH
        ),
        "query_results_path": (
            ULTRAVPR_QUERY_RESULTS_PATH
        ),
    },
}

# ------------------------------------------------------------
# 5. Load metric tables
# ------------------------------------------------------------

metric_tables = {}

for method_key, metadata in METHODS.items():

    dataframe = pd.read_csv(
        metadata[
            "metrics_path"
        ]
    )

    required_columns = {
        "group",
        "nominal_height",
        "query_count",
        "recall_at_1_percent",
        "recall_at_5_percent",
        "recall_at_10_percent",
        "mean_average_precision_percent",
        "mean_positive_rank",
    }

    missing = (
        required_columns
        - set(
            dataframe.columns
        )
    )

    if missing:
        raise KeyError(
            f"{method_key} metric columns missing: "
            f"{sorted(missing)}"
        )

    metric_tables[
        method_key
    ] = dataframe

# ------------------------------------------------------------
# 6. Build final overall comparison
# ------------------------------------------------------------

overall_records = []

for method_key, metadata in METHODS.items():

    dataframe = (
        metric_tables[
            method_key
        ]
    )

    overall = (
        dataframe[
            dataframe["group"]
            == "overall"
        ]
        .iloc[0]
    )

    overall_records.append({
        "method_key": (
            method_key
        ),

        "method": (
            metadata[
                "display_name"
            ]
        ),

        "provenance": (
            metadata[
                "provenance"
            ]
        ),

        "training_context": (
            metadata[
                "training_context"
            ]
        ),

        "descriptor_dimension": (
            metadata[
                "descriptor_dimension"
            ]
        ),

        "query_count": int(
            overall[
                "query_count"
            ]
        ),

        "recall_at_1_percent": float(
            overall[
                "recall_at_1_percent"
            ]
        ),

        "recall_at_5_percent": float(
            overall[
                "recall_at_5_percent"
            ]
        ),

        "recall_at_10_percent": float(
            overall[
                "recall_at_10_percent"
            ]
        ),

        "mean_average_precision_percent": float(
            overall[
                "mean_average_precision_percent"
            ]
        ),

        "mean_positive_rank": float(
            overall[
                "mean_positive_rank"
            ]
        ),
    })

overall_df = pd.DataFrame(
    overall_records
)

overall_df[
    "rank_by_recall_at_1"
] = (
    overall_df[
        "recall_at_1_percent"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

overall_df[
    "rank_by_map"
] = (
    overall_df[
        "mean_average_precision_percent"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

overall_df = (
    overall_df
    .sort_values(
        [
            "recall_at_1_percent",
            "mean_average_precision_percent",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

overall_df.to_csv(
    FINAL_METRICS_CSV,
    index=False,
)

# ------------------------------------------------------------
# 7. Build per-height comparison
# ------------------------------------------------------------

height_records = []

for method_key, metadata in METHODS.items():

    dataframe = (
        metric_tables[
            method_key
        ]
    )

    height_rows = (
        dataframe[
            dataframe["group"]
            != "overall"
        ]
        .copy()
    )

    for _, row in (
        height_rows.iterrows()
    ):

        nominal_height = str(
            row[
                "nominal_height"
            ]
        ).replace(
            ".0",
            "",
        )

        height_records.append({
            "method_key": (
                method_key
            ),

            "method": (
                metadata[
                    "display_name"
                ]
            ),

            "nominal_height": (
                nominal_height
            ),

            "query_count": int(
                row[
                    "query_count"
                ]
            ),

            "recall_at_1_percent": float(
                row[
                    "recall_at_1_percent"
                ]
            ),

            "recall_at_5_percent": float(
                row[
                    "recall_at_5_percent"
                ]
            ),

            "recall_at_10_percent": float(
                row[
                    "recall_at_10_percent"
                ]
            ),

            "mean_average_precision_percent": float(
                row[
                    "mean_average_precision_percent"
                ]
            ),

            "mean_positive_rank": float(
                row[
                    "mean_positive_rank"
                ]
            ),
        })

height_df = pd.DataFrame(
    height_records
)

height_df = (
    height_df
    .sort_values(
        [
            "nominal_height",
            "recall_at_1_percent",
        ],
        key=lambda series: (
            series.astype(int)
            if series.name
            == "nominal_height"
            else series
        ),
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

height_df.to_csv(
    FINAL_HEIGHT_CSV,
    index=False,
)

# ------------------------------------------------------------
# 8. Load all per-query results
# ------------------------------------------------------------

query_tables = {}

required_query_columns = {
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
    "positive_rank",
    "top1_tile_id",
    "top1_location_id",
    "top1_correct",
}

def parse_bool(
    series,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    result = normalized.map(
        mapping
    )

    if result.isna().any():
        raise ValueError(
            "Unsupported boolean values: "
            f"{sorted(normalized[result.isna()].unique())}"
        )

    return result.astype(bool)


for method_key, metadata in METHODS.items():

    dataframe = pd.read_csv(
        metadata[
            "query_results_path"
        ],
        dtype=str,
    )

    missing = (
        required_query_columns
        - set(
            dataframe.columns
        )
    )

    if missing:
        raise KeyError(
            f"{method_key} query columns missing: "
            f"{sorted(missing)}"
        )

    if len(dataframe) != 8000:
        raise RuntimeError(
            f"{method_key}: expected 8,000 queries, "
            f"found {len(dataframe)}."
        )

    dataframe[
        "frame_id"
    ] = dataframe[
        "frame_id"
    ].astype(str)

    dataframe[
        "query_location_id"
    ] = (
        dataframe[
            "query_location_id"
        ]
        .astype(str)
        .str.zfill(4)
    )

    dataframe[
        "top1_location_id"
    ] = (
        dataframe[
            "top1_location_id"
        ]
        .astype(str)
        .str.zfill(4)
    )

    dataframe[
        "nominal_height"
    ] = (
        dataframe[
            "nominal_height"
        ]
        .astype(str)
        .str.replace(
            ".0",
            "",
            regex=False,
        )
    )

    dataframe[
        "top1_correct"
    ] = parse_bool(
        dataframe[
            "top1_correct"
        ]
    )

    dataframe[
        "positive_rank"
    ] = pd.to_numeric(
        dataframe[
            "positive_rank"
        ],
        errors="raise",
    ).astype(int)

    query_tables[
        method_key
    ] = dataframe

# ------------------------------------------------------------
# 9. Verify identical query identities
# ------------------------------------------------------------

identity_columns = [
    "frame_id",
    "query_location_id",
    "nominal_height",
    "positive_tile_id",
]

reference_identity = (
    query_tables[
        "Local_Reimplementation"
    ][
        identity_columns
    ]
    .sort_values(
        "frame_id"
    )
    .reset_index(drop=True)
)

for method_key in [
    "Sample4Geo",
    "UltraVPR",
]:

    current_identity = (
        query_tables[
            method_key
        ][
            identity_columns
        ]
        .sort_values(
            "frame_id"
        )
        .reset_index(drop=True)
    )

    if not reference_identity.equals(
        current_identity
    ):
        raise RuntimeError(
            f"Query identity mismatch for {method_key}."
        )

print(
    "✅ All three methods use the same "
    "8,000 query identities."
)

# ------------------------------------------------------------
# 10. Build paired query table
# ------------------------------------------------------------

base = (
    query_tables[
        "Local_Reimplementation"
    ][
        identity_columns
    ]
    .copy()
)

for method_key, prefix in [
    (
        "Local_Reimplementation",
        "local",
    ),
    (
        "Sample4Geo",
        "sample4geo",
    ),
    (
        "UltraVPR",
        "ultravpr",
    ),
]:

    method_df = (
        query_tables[
            method_key
        ][
            [
                "frame_id",
                "positive_rank",
                "top1_tile_id",
                "top1_location_id",
                "top1_correct",
            ]
        ]
        .copy()
        .rename(
            columns={
                "positive_rank": (
                    f"{prefix}_positive_rank"
                ),
                "top1_tile_id": (
                    f"{prefix}_top1_tile_id"
                ),
                "top1_location_id": (
                    f"{prefix}_top1_location_id"
                ),
                "top1_correct": (
                    f"{prefix}_top1_correct"
                ),
            }
        )
    )

    base = base.merge(
        method_df,
        on="frame_id",
        how="inner",
        validate="one_to_one",
    )

paired_df = base

if len(paired_df) != 8000:
    raise RuntimeError(
        "Three-method paired table does not "
        "contain 8,000 rows."
    )

# ------------------------------------------------------------
# 11. Three-method outcome categories
# ------------------------------------------------------------

local_correct = (
    paired_df[
        "local_top1_correct"
    ].astype(bool)
)

sample_correct = (
    paired_df[
        "sample4geo_top1_correct"
    ].astype(bool)
)

ultra_correct = (
    paired_df[
        "ultravpr_top1_correct"
    ].astype(bool)
)

paired_df[
    "correct_method_count"
] = (
    local_correct.astype(int)
    + sample_correct.astype(int)
    + ultra_correct.astype(int)
)

paired_df[
    "all_three_correct"
] = (
    local_correct
    & sample_correct
    & ultra_correct
)

paired_df[
    "all_three_wrong"
] = (
    ~local_correct
    & ~sample_correct
    & ~ultra_correct
)

paired_df[
    "local_only_correct"
] = (
    local_correct
    & ~sample_correct
    & ~ultra_correct
)

paired_df[
    "sample4geo_only_correct"
] = (
    ~local_correct
    & sample_correct
    & ~ultra_correct
)

paired_df[
    "ultravpr_only_correct"
] = (
    ~local_correct
    & ~sample_correct
    & ultra_correct
)

paired_df[
    "all_same_top1_prediction"
] = (
    (
        paired_df[
            "local_top1_tile_id"
        ]
        == paired_df[
            "sample4geo_top1_tile_id"
        ]
    )
    &
    (
        paired_df[
            "local_top1_tile_id"
        ]
        == paired_df[
            "ultravpr_top1_tile_id"
        ]
    )
)

paired_df[
    "best_positive_rank"
] = (
    paired_df[
        [
            "local_positive_rank",
            "sample4geo_positive_rank",
            "ultravpr_positive_rank",
        ]
    ]
    .min(
        axis=1
    )
)

paired_df[
    "worst_positive_rank"
] = (
    paired_df[
        [
            "local_positive_rank",
            "sample4geo_positive_rank",
            "ultravpr_positive_rank",
        ]
    ]
    .max(
        axis=1
    )
)

paired_df.to_csv(
    FINAL_PAIRED_CSV,
    index=False,
)

# ------------------------------------------------------------
# 12. Pairwise R@1 differences
# ------------------------------------------------------------

overall_lookup = {
    row[
        "method_key"
    ]: row
    for _, row
    in overall_df.iterrows()
}

local_r1 = float(
    overall_lookup[
        "Local_Reimplementation"
    ][
        "recall_at_1_percent"
    ]
)

sample_r1 = float(
    overall_lookup[
        "Sample4Geo"
    ][
        "recall_at_1_percent"
    ]
)

ultra_r1 = float(
    overall_lookup[
        "UltraVPR"
    ][
        "recall_at_1_percent"
    ]
)

local_map = float(
    overall_lookup[
        "Local_Reimplementation"
    ][
        "mean_average_precision_percent"
    ]
)

sample_map = float(
    overall_lookup[
        "Sample4Geo"
    ][
        "mean_average_precision_percent"
    ]
)

ultra_map = float(
    overall_lookup[
        "UltraVPR"
    ][
        "mean_average_precision_percent"
    ]
)

pairwise_differences = {
    "local_minus_sample4geo_r1_pp": (
        local_r1
        - sample_r1
    ),

    "local_minus_ultravpr_r1_pp": (
        local_r1
        - ultra_r1
    ),

    "sample4geo_minus_ultravpr_r1_pp": (
        sample_r1
        - ultra_r1
    ),

    "local_minus_sample4geo_map_pp": (
        local_map
        - sample_map
    ),

    "local_minus_ultravpr_map_pp": (
        local_map
        - ultra_map
    ),

    "sample4geo_minus_ultravpr_map_pp": (
        sample_map
        - ultra_map
    ),
}

# ------------------------------------------------------------
# 13. MobileGeo comparability status
# ------------------------------------------------------------

mobilegeo_status = {
    "release_provenance": (
        "PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION"
    ),

    "published_descriptor_dimension": (
        768
    ),

    "published_assets_available": [
        "second_D2S.mat",
        "second_S2D.mat",
    ],

    "direct_common_benchmark_entry": (
        False
    ),

    "reason": (
        "No verified official MobileGeo checkpoint "
        "and complete raw-image-to-descriptor pipeline "
        "were released for generating descriptors on "
        "the exact project-defined 8,000-query / "
        "40-gallery SUES-200 test split."
    ),

    "reporting_rule": (
        "Published MobileGeo MAT results must remain "
        "in a separate provenance table and must not "
        "be ranked directly against the three methods "
        "evaluated on the common project split."
    ),
}

# ------------------------------------------------------------
# 14. Summary statistics
# ------------------------------------------------------------

three_method_summary = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "phase": (
        "PHASE_3_COMMON_RETRIEVAL_BENCHMARK"
    ),

    "protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "query_count": 8000,
    "gallery_count": 40,
    "test_locations": 40,

    "methods_in_direct_common_table": [
        (
            "AdvancedEdgeGeoLPN_"
            "LOCAL_REIMPLEMENTATION"
        ),
        "Sample4Geo",
        "UltraVPR",
    ],

    "overall_results": (
        overall_df.to_dict(
            orient="records"
        )
    ),

    "pairwise_differences": (
        pairwise_differences
    ),

    "paired_query_summary": {
        "all_three_correct": int(
            paired_df[
                "all_three_correct"
            ].sum()
        ),

        "all_three_wrong": int(
            paired_df[
                "all_three_wrong"
            ].sum()
        ),

        "local_only_correct": int(
            paired_df[
                "local_only_correct"
            ].sum()
        ),

        "sample4geo_only_correct": int(
            paired_df[
                "sample4geo_only_correct"
            ].sum()
        ),

        "ultravpr_only_correct": int(
            paired_df[
                "ultravpr_only_correct"
            ].sum()
        ),

        "all_same_top1_prediction": int(
            paired_df[
                "all_same_top1_prediction"
            ].sum()
        ),

        "at_least_one_correct": int(
            (
                paired_df[
                    "correct_method_count"
                ] >= 1
            ).sum()
        ),
    },

    "mobilegeo_comparability": (
        mobilegeo_status
    ),

    "interpretation_restrictions": [
        (
            "All three direct-comparison methods "
            "use identical queries, gallery and evaluator."
        ),
        (
            "Their checkpoint training regimes differ, "
            "so accuracy differences do not isolate architecture."
        ),
        (
            "Sample4Geo is zero-shot transfer from University-1652."
        ),
        (
            "UltraVPR is zero-shot transfer to this "
            "project-defined SUES-200 split."
        ),
        (
            "The local model is not official MobileGeo."
        ),
        (
            "MobileGeo published MAT evaluation is not "
            "directly comparable to this common benchmark."
        ),
        (
            "Retrieval results are not verified UAV pose estimates."
        ),
    ],

    "files": {
        "overall_metrics_csv": str(
            FINAL_METRICS_CSV
        ),

        "height_metrics_csv": str(
            FINAL_HEIGHT_CSV
        ),

        "paired_query_results_csv": str(
            FINAL_PAIRED_CSV
        ),

        "report_md": str(
            FINAL_REPORT_MD
        ),
    },
}

FINAL_SUMMARY_JSON.write_text(
    json.dumps(
        three_method_summary,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 15. Markdown report
# ------------------------------------------------------------

overall_lines = []

for _, row in overall_df.iterrows():

    overall_lines.append(
        "| "
        f"{row['method']} | "
        f"{int(row['descriptor_dimension'])} | "
        f"{row['recall_at_1_percent']:.2f}% | "
        f"{row['recall_at_5_percent']:.2f}% | "
        f"{row['recall_at_10_percent']:.2f}% | "
        f"{row['mean_average_precision_percent']:.2f}% | "
        f"{row['mean_positive_rank']:.3f} |"
    )

overall_table = "\n".join(
    overall_lines
)

height_pivot = (
    height_df.pivot(
        index="nominal_height",
        columns="method_key",
        values="recall_at_1_percent",
    )
    .reset_index()
)

height_lines = []

for _, row in height_pivot.iterrows():

    height_lines.append(
        "| "
        f"{row['nominal_height']} | "
        f"{row['Local_Reimplementation']:.2f}% | "
        f"{row['Sample4Geo']:.2f}% | "
        f"{row['UltraVPR']:.2f}% |"
    )

height_table = "\n".join(
    height_lines
)

best_method = (
    overall_df.iloc[0][
        "method"
    ]
)

best_r1 = float(
    overall_df.iloc[0][
        "recall_at_1_percent"
    ]
)

report_text = f"""# Phase 3 — Common Retrieval Benchmark

## Protocol

- Test queries: 8,000
- Gallery images: 40
- Held-out test locations: 40
- Nominal dataset height labels: 150, 200, 250, 300
- Retrieval: exact cosine similarity
- Descriptor normalization: L2
- Benchmark: `PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20`

## Direct common benchmark

| Method | Descriptor dim | R@1 | R@5 | R@10 | mAP | Mean positive rank |
|---|---:|---:|---:|---:|---:|---:|
{overall_table}

Best Recall@1 on this project-defined protocol:
**{best_method} — {best_r1:.2f}%**

## Recall@1 by nominal height

| Height | Local reimplementation | Sample4Geo | UltraVPR |
|---:|---:|---:|---:|
{height_table}

The height values are dataset folder labels and are not verified
as AGL or MSL altitude.

## Pairwise differences

- Local minus Sample4Geo R@1:
  {pairwise_differences['local_minus_sample4geo_r1_pp']:.2f} percentage points
- Local minus UltraVPR R@1:
  {pairwise_differences['local_minus_ultravpr_r1_pp']:.2f} percentage points
- Sample4Geo minus UltraVPR R@1:
  {pairwise_differences['sample4geo_minus_ultravpr_r1_pp']:.2f} percentage points

## Three-method paired query analysis

- All three correct:
  {int(paired_df['all_three_correct'].sum())}
- All three wrong:
  {int(paired_df['all_three_wrong'].sum())}
- Local only correct:
  {int(paired_df['local_only_correct'].sum())}
- Sample4Geo only correct:
  {int(paired_df['sample4geo_only_correct'].sum())}
- UltraVPR only correct:
  {int(paired_df['ultravpr_only_correct'].sum())}
- At least one method correct:
  {int((paired_df['correct_method_count'] >= 1).sum())}
- All three produced the same Top-1 prediction:
  {int(paired_df['all_same_top1_prediction'].sum())}

## Provenance

### AdvancedEdgeGeoLPN local reimplementation

`LOCAL_REIMPLEMENTATION`

This method must not be described as official MobileGeo.

### Sample4Geo

Official Sample4Geo University-1652 checkpoint evaluated as zero-shot
transfer on the project-defined SUES-200 test split.

This is not an official Sample4Geo SUES-200 result.

### UltraVPR

Official UltraVPR checkpoint evaluated as zero-shot transfer on the
project-defined SUES-200 test split.

This is not an official UltraVPR SUES-200 result.

## MobileGeo release status

MobileGeo is not included in the direct common accuracy ranking.

The released MobileGeo artifacts support published precomputed-feature
evaluation with 768-D MAT descriptors, but a verified official checkpoint
and complete raw-image-to-descriptor pipeline were not available for
generating descriptors on the exact same 8,000-query / 40-gallery split.

Therefore the MobileGeo published MAT results must remain a separate
provenance result and must not be ranked directly against the three
methods above.

## Interpretation restriction

The three directly compared methods use the same test queries, gallery
and evaluator, but their checkpoints have different training regimes.
The observed accuracy differences therefore do not isolate architecture
alone.

Image retrieval performance is not equivalent to verified geographic
UAV pose estimation.
"""

FINAL_REPORT_MD.write_text(
    report_text,
    encoding="utf-8",
)

# ------------------------------------------------------------
# 16. Final display
# ------------------------------------------------------------

print("\n" + "=" * 82)
print("✅ FINAL THREE-METHOD COMMON BENCHMARK COMPLETE")
print("=" * 82)

print("\nOverall results:")

print(
    overall_df[
        [
            "method",
            "descriptor_dimension",
            "recall_at_1_percent",
            "recall_at_5_percent",
            "recall_at_10_percent",
            "mean_average_precision_percent",
            "mean_positive_rank",
        ]
    ].to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda value: f"{value:.2f}"
            ),

            "recall_at_5_percent": (
                lambda value: f"{value:.2f}"
            ),

            "recall_at_10_percent": (
                lambda value: f"{value:.2f}"
            ),

            "mean_average_precision_percent": (
                lambda value: f"{value:.2f}"
            ),

            "mean_positive_rank": (
                lambda value: f"{value:.3f}"
            ),
        },
    )
)

print("\nRecall@1 by nominal height:")

print(
    height_pivot.to_string(
        index=False,
        formatters={
            "Local_Reimplementation": (
                lambda value: f"{value:.2f}"
            ),

            "Sample4Geo": (
                lambda value: f"{value:.2f}"
            ),

            "UltraVPR": (
                lambda value: f"{value:.2f}"
            ),
        },
    )
)

print("\nPairwise R@1 differences:")

print(
    "Local - Sample4Geo:",
    f"{pairwise_differences['local_minus_sample4geo_r1_pp']:.2f}",
    "percentage points",
)

print(
    "Local - UltraVPR:",
    f"{pairwise_differences['local_minus_ultravpr_r1_pp']:.2f}",
    "percentage points",
)

print(
    "Sample4Geo - UltraVPR:",
    f"{pairwise_differences['sample4geo_minus_ultravpr_r1_pp']:.2f}",
    "percentage points",
)

print("\nThree-method paired outcomes:")

print(
    "All three correct:",
    int(
        paired_df[
            "all_three_correct"
        ].sum()
    ),
)

print(
    "All three wrong:",
    int(
        paired_df[
            "all_three_wrong"
        ].sum()
    ),
)

print(
    "Local only correct:",
    int(
        paired_df[
            "local_only_correct"
        ].sum()
    ),
)

print(
    "Sample4Geo only correct:",
    int(
        paired_df[
            "sample4geo_only_correct"
        ].sum()
    ),
)

print(
    "UltraVPR only correct:",
    int(
        paired_df[
            "ultravpr_only_correct"
        ].sum()
    ),
)

print(
    "At least one correct:",
    int(
        (
            paired_df[
                "correct_method_count"
            ] >= 1
        ).sum()
    ),
)

print("\nMobileGeo direct common-table status:")
print("NOT DIRECTLY COMPARABLE")

print("\nFinal metrics CSV:")
print(FINAL_METRICS_CSV)

print("\nPer-height CSV:")
print(FINAL_HEIGHT_CSV)

print("\nPaired query CSV:")
print(FINAL_PAIRED_CSV)

print("\nFinal comparison JSON:")
print(FINAL_SUMMARY_JSON)

print("\nFinal comparison report:")
print(FINAL_REPORT_MD)

print("\nNext action:")
print(
    "Create the final Phase 3 evidence package and ZIP."
)

✅ All three methods use the same 8,000 query identities.

✅ FINAL THREE-METHOD COMMON BENCHMARK COMPLETE

Overall results:
                                   method  descriptor_dimension recall_at_1_percent recall_at_5_percent recall_at_10_percent mean_average_precision_percent mean_positive_rank
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION                  2048               94.58               99.99               100.00                          96.93              1.082
                               Sample4Geo                  1024               85.90               98.74                99.78                          91.79              1.274
                                 UltraVPR                   256               81.94               95.22                98.47                          87.83              1.667

Recall@1 by nominal height:
nominal_height Local_Reimplementation Sample4Geo UltraVPR
           150                  93.35      78.55    56.35
           200                  

In [6]:
# ============================================================
# PHASE 3 — CELL 16
# Final validation + provenance manifest + evidence ZIP
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import zipfile

import pandas as pd

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

PHASE2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "sues200_corrected"
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / "results"
    / "common_benchmark"
)

REPORT_ROOT = (
    PHASE3_ROOT
    / "reports"
)

RANKING_ROOT = (
    PHASE3_ROOT
    / "rankings"
)

EMBEDDING_ROOT = (
    PHASE3_ROOT
    / "embeddings"
)

SAMPLE4GEO_AUDIT_ROOT = (
    PHASE3_ROOT
    / "sample4geo_audit"
)

ULTRAVPR_AUDIT_ROOT = (
    PHASE3_ROOT
    / "ultravpr_audit"
)

MANIFEST_ROOT = (
    PROJECT_ROOT
    / "manifests"
    / "sues200_corrected"
)

CONFIG_ROOT = (
    PROJECT_ROOT
    / "configs"
)

PACKAGE_ROOT = (
    PROJECT_ROOT
    / "reports"
)

PACKAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 2. Main Phase 3 result files
# ------------------------------------------------------------

FINAL_METRICS_CSV = (
    REPORT_ROOT
    / "final_three_method_common_metrics.csv"
)

FINAL_HEIGHT_CSV = (
    REPORT_ROOT
    / "final_three_method_height_metrics.csv"
)

FINAL_COMPARISON_JSON = (
    REPORT_ROOT
    / "final_three_method_comparison.json"
)

FINAL_REPORT_MD = (
    REPORT_ROOT
    / "final_three_method_comparison_report.md"
)

FINAL_PAIRED_QUERY_CSV = (
    RANKING_ROOT
    / "final_three_method_paired_query_results.csv"
)

SAMPLE_METRICS_CSV = (
    REPORT_ROOT
    / "sample4geo_location_disjoint_metrics.csv"
)

ULTRA_METRICS_CSV = (
    REPORT_ROOT
    / "ultravpr_location_disjoint_metrics.csv"
)

LOCAL_METRICS_CSV = (
    PHASE2_ROOT
    / "corrected_location_disjoint_metrics.csv"
)

required_final_files = [
    FINAL_METRICS_CSV,
    FINAL_HEIGHT_CSV,
    FINAL_COMPARISON_JSON,
    FINAL_REPORT_MD,
    FINAL_PAIRED_QUERY_CSV,
    SAMPLE_METRICS_CSV,
    ULTRA_METRICS_CSV,
    LOCAL_METRICS_CSV,
]

for path in required_final_files:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required final file missing:\n{path}"
        )

# ------------------------------------------------------------
# 3. Validate final metrics
# ------------------------------------------------------------

final_metrics_df = pd.read_csv(
    FINAL_METRICS_CSV
)

required_methods = {
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION",
    "Sample4Geo",
    "UltraVPR",
}

found_methods = set(
    final_metrics_df[
        "method"
    ].astype(str)
)

if found_methods != required_methods:
    raise RuntimeError(
        "Final metric table does not contain "
        "exactly the expected three methods.\n"
        f"Found: {sorted(found_methods)}"
    )

if len(final_metrics_df) != 3:
    raise RuntimeError(
        "Expected exactly three rows in final "
        "overall metric table."
    )

required_metric_columns = [
    "recall_at_1_percent",
    "recall_at_5_percent",
    "recall_at_10_percent",
    "mean_average_precision_percent",
    "mean_positive_rank",
]

for column in required_metric_columns:

    if column not in final_metrics_df.columns:
        raise KeyError(
            f"Missing final metric column: {column}"
        )

    values = pd.to_numeric(
        final_metrics_df[column],
        errors="raise",
    )

    if values.isna().any():
        raise RuntimeError(
            f"NaN values found in {column}"
        )

# ------------------------------------------------------------
# 4. Validate exact headline metrics
# ------------------------------------------------------------

metric_lookup = (
    final_metrics_df
    .set_index("method")
)

local_r1 = float(
    metric_lookup.loc[
        "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION",
        "recall_at_1_percent",
    ]
)

sample_r1 = float(
    metric_lookup.loc[
        "Sample4Geo",
        "recall_at_1_percent",
    ]
)

ultra_r1 = float(
    metric_lookup.loc[
        "UltraVPR",
        "recall_at_1_percent",
    ]
)

# Known completed results for the first two methods.
if abs(
    local_r1 - 94.575
) > 0.02:
    raise RuntimeError(
        f"Unexpected local R@1: {local_r1}"
    )

if abs(
    sample_r1 - 85.9
) > 0.02:
    raise RuntimeError(
        f"Unexpected Sample4Geo R@1: {sample_r1}"
    )

# UltraVPR result is whatever Cell 14 produced,
# but it must be a valid percentage.
if not (
    0.0 <= ultra_r1 <= 100.0
):
    raise RuntimeError(
        f"Invalid UltraVPR R@1: {ultra_r1}"
    )

# ------------------------------------------------------------
# 5. Validate paired query table
# ------------------------------------------------------------

paired_df = pd.read_csv(
    FINAL_PAIRED_QUERY_CSV
)

if len(paired_df) != 8000:
    raise RuntimeError(
        "Final paired query table must contain "
        f"8,000 rows, found {len(paired_df)}."
    )

if not paired_df[
    "frame_id"
].astype(str).is_unique:
    raise RuntimeError(
        "Paired frame IDs are not unique."
    )

# ------------------------------------------------------------
# 6. Validate height table
# ------------------------------------------------------------

height_df = pd.read_csv(
    FINAL_HEIGHT_CSV
)

expected_heights = {
    "150",
    "200",
    "250",
    "300",
}

height_values = set(
    height_df[
        "nominal_height"
    ]
    .astype(str)
    .str.replace(
        ".0",
        "",
        regex=False,
    )
)

if height_values != expected_heights:
    raise RuntimeError(
        "Unexpected nominal-height labels:\n"
        f"{sorted(height_values)}"
    )

# 3 methods x 4 heights.
if len(height_df) != 12:
    raise RuntimeError(
        "Expected 12 per-height rows, "
        f"found {len(height_df)}."
    )

# ------------------------------------------------------------
# 7. SHA-256 helper
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()

# ------------------------------------------------------------
# 8. Record major checkpoint provenance
# ------------------------------------------------------------

LOCAL_CHECKPOINT = Path(
    "/content/drive/MyDrive/mobilegeo_project/"
    "baseline_v1/checkpoint/"
    "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION.pth"
)

SAMPLE_CHECKPOINT_CONFIG = (
    CONFIG_ROOT
    / "sample4geo_checkpoint_path.txt"
)

ULTRA_CHECKPOINT_CONFIG = (
    CONFIG_ROOT
    / "ultravpr_checkpoint_path.txt"
)

checkpoint_records = []

# Local checkpoint.
if LOCAL_CHECKPOINT.is_file():

    checkpoint_records.append({
        "method": (
            "AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION"
        ),
        "path": str(
            LOCAL_CHECKPOINT
        ),
        "sha256": sha256_file(
            LOCAL_CHECKPOINT
        ),
        "provenance": (
            "LOCAL_REIMPLEMENTATION"
        ),
    })

# Sample4Geo checkpoint.
if SAMPLE_CHECKPOINT_CONFIG.is_file():

    sample_checkpoint = Path(
        SAMPLE_CHECKPOINT_CONFIG
        .read_text(
            encoding="utf-8"
        )
        .strip()
    )

    if sample_checkpoint.is_file():

        checkpoint_records.append({
            "method": "Sample4Geo",
            "path": str(
                sample_checkpoint
            ),
            "sha256": sha256_file(
                sample_checkpoint
            ),
            "provenance": (
                "OFFICIAL_SAMPLE4GEO_CHECKPOINT"
            ),
        })

# UltraVPR checkpoint.
if ULTRA_CHECKPOINT_CONFIG.is_file():

    ultra_checkpoint = Path(
        ULTRA_CHECKPOINT_CONFIG
        .read_text(
            encoding="utf-8"
        )
        .strip()
    )

    if ultra_checkpoint.is_file():

        checkpoint_records.append({
            "method": "UltraVPR",
            "path": str(
                ultra_checkpoint
            ),
            "sha256": sha256_file(
                ultra_checkpoint
            ),
            "provenance": (
                "OFFICIAL_ULTRAVPR_CHECKPOINT"
            ),
        })

# ------------------------------------------------------------
# 9. Embedding file inventory
# ------------------------------------------------------------

embedding_records = []

if EMBEDDING_ROOT.is_dir():

    for file_path in sorted(
        EMBEDDING_ROOT.glob(
            "*.npz"
        )
    ):

        embedding_records.append({
            "filename": (
                file_path.name
            ),
            "path": str(
                file_path
            ),
            "size_bytes": int(
                file_path.stat().st_size
            ),
            "sha256": (
                sha256_file(
                    file_path
                )
            ),
        })

# Phase 2 local embeddings are outside Phase 3.
local_embedding_root = (
    PHASE2_ROOT
    / "embeddings"
)

if local_embedding_root.is_dir():

    for file_path in sorted(
        local_embedding_root.glob(
            "*.npz"
        )
    ):

        embedding_records.append({
            "filename": (
                file_path.name
            ),
            "path": str(
                file_path
            ),
            "size_bytes": int(
                file_path.stat().st_size
            ),
            "sha256": (
                sha256_file(
                    file_path
                )
            ),
        })

# ------------------------------------------------------------
# 10. Collect evidence files for ZIP
# ------------------------------------------------------------

evidence_files = []

def add_file(
    file_path,
    category,
):
    file_path = Path(
        file_path
    )

    if file_path.is_file():

        evidence_files.append({
            "path": file_path,
            "category": category,
        })


def add_directory_files(
    directory,
    category,
    allowed_suffixes=None,
):
    directory = Path(
        directory
    )

    if not directory.is_dir():
        return

    for file_path in sorted(
        directory.rglob("*")
    ):

        if not file_path.is_file():
            continue

        if (
            allowed_suffixes is not None
            and file_path.suffix.lower()
            not in allowed_suffixes
        ):
            continue

        add_file(
            file_path,
            category,
        )

# Final comparison.
for path in [
    FINAL_METRICS_CSV,
    FINAL_HEIGHT_CSV,
    FINAL_COMPARISON_JSON,
    FINAL_REPORT_MD,
]:
    add_file(
        path,
        "final_comparison",
    )

# Compact paired result.
add_file(
    FINAL_PAIRED_QUERY_CSV,
    "paired_query_results",
)

# Method-level reports.
add_directory_files(
    REPORT_ROOT,
    "phase3_reports",
    allowed_suffixes={
        ".csv",
        ".json",
        ".md",
    },
)

# Sample4Geo audit evidence.
add_directory_files(
    SAMPLE4GEO_AUDIT_ROOT,
    "sample4geo_audit",
    allowed_suffixes={
        ".csv",
        ".json",
        ".txt",
    },
)

# UltraVPR audit evidence.
add_directory_files(
    ULTRAVPR_AUDIT_ROOT,
    "ultravpr_audit",
    allowed_suffixes={
        ".csv",
        ".json",
        ".txt",
    },
)

# Corrected SUES-200 manifests.
add_directory_files(
    MANIFEST_ROOT,
    "benchmark_manifests",
    allowed_suffixes={
        ".csv",
        ".json",
        ".txt",
    },
)

# Relevant configuration files.
for path in [
    CONFIG_ROOT
    / "sues200_resolved_paths.txt",

    CONFIG_ROOT
    / "sample4geo_checkpoint_path.txt",

    CONFIG_ROOT
    / "sample4geo_expected_checkpoint_path.txt",

    CONFIG_ROOT
    / "ultravpr_checkpoint_path.txt",
]:
    add_file(
        path,
        "configuration",
    )

# Phase 2 benchmark evidence needed to trace local results.
for path in [
    PHASE2_ROOT
    / "corrected_location_disjoint_metrics.csv",

    PHASE2_ROOT
    / "corrected_location_disjoint_metrics.json",

    PHASE2_ROOT
    / "failure_analysis.json",

    PHASE2_ROOT
    / "phase2_summary.json",

    PHASE2_ROOT
    / "phase2_corrected_sues200_report.md",
]:
    add_file(
        path,
        "phase2_local_baseline_evidence",
    )

# Remove duplicate physical paths.
unique_evidence = {}

for record in evidence_files:

    resolved_path = str(
        record["path"].resolve()
    )

    unique_evidence[
        resolved_path
    ] = record

evidence_files = list(
    unique_evidence.values()
)

# ------------------------------------------------------------
# 11. Create evidence inventory with hashes
# ------------------------------------------------------------

evidence_inventory = []

for record in sorted(
    evidence_files,
    key=lambda item: (
        str(
            item["path"]
        )
    ),
):

    file_path = (
        record["path"]
    )

    evidence_inventory.append({
        "category": (
            record["category"]
        ),
        "source_path": str(
            file_path
        ),
        "size_bytes": int(
            file_path.stat().st_size
        ),
        "sha256": sha256_file(
            file_path
        ),
    })

# ------------------------------------------------------------
# 12. Phase 3 completion manifest
# ------------------------------------------------------------

completion_manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "project": (
        "UAV_GEO_LOCALIZATION"
    ),

    "phase": (
        "PHASE_3_COMMON_RETRIEVAL_BENCHMARK"
    ),

    "phase_complete": True,

    "benchmark": (
        "COMMON_SUES200_LOCATION_DISJOINT_BENCHMARK"
    ),

    "protocol": (
        "PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20"
    ),

    "test_query_count": 8000,

    "test_gallery_count": 40,

    "test_location_count": 40,

    "nominal_height_labels": [
        150,
        200,
        250,
        300,
    ],

    "height_label_interpretation": (
        "NOMINAL_DATASET_FOLDER_LABEL_"
        "NOT_VERIFIED_AS_AGL_OR_MSL"
    ),

    "direct_common_benchmark_methods": [
        {
            "method": (
                "AdvancedEdgeGeoLPN_"
                "LOCAL_REIMPLEMENTATION"
            ),
            "provenance": (
                "LOCAL_REIMPLEMENTATION"
            ),
            "descriptor_dimension": 2048,
            "recall_at_1_percent": (
                local_r1
            ),
        },
        {
            "method": (
                "Sample4Geo"
            ),
            "provenance": (
                "OFFICIAL_SAMPLE4GEO_CHECKPOINT_"
                "ZERO_SHOT_TRANSFER_FROM_UNIVERSITY1652"
            ),
            "descriptor_dimension": 1024,
            "recall_at_1_percent": (
                sample_r1
            ),
        },
        {
            "method": (
                "UltraVPR"
            ),
            "provenance": (
                "OFFICIAL_ULTRAVPR_CHECKPOINT_"
                "ZERO_SHOT_TRANSFER_TO_PROJECT_SUES200_SPLIT"
            ),
            "descriptor_dimension": 256,
            "recall_at_1_percent": (
                ultra_r1
            ),
        },
    ],

    "mobilegeo_release_status": {
        "direct_common_table_entry": False,

        "provenance": (
            "PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION"
        ),

        "released_descriptor_dimension": 768,

        "reason": (
            "No verified official MobileGeo checkpoint "
            "and complete raw-image-to-descriptor pipeline "
            "were available for generating descriptors "
            "on this exact project-defined test split."
        ),
    },

    "checkpoint_records": (
        checkpoint_records
    ),

    "embedding_records": (
        embedding_records
    ),

    "embedding_files_included_in_zip": (
        False
    ),

    "embedding_exclusion_reason": (
        "Embedding NPZ files are retained separately "
        "in Google Drive to avoid duplicating large "
        "binary artifacts in the evidence ZIP."
    ),

    "full_320k_ranking_files_included_in_zip": (
        False
    ),

    "ranking_exclusion_reason": (
        "Large full-ranking CSV files are retained "
        "in the project results directory; compact "
        "metrics, audits and paired-query evidence "
        "are packaged instead."
    ),

    "reporting_restrictions": [
        (
            "Do not call the local reimplementation "
            "an official MobileGeo implementation."
        ),
        (
            "Do not describe Sample4Geo zero-shot "
            "results as official SUES-200 results."
        ),
        (
            "Do not describe UltraVPR zero-shot "
            "results as official SUES-200 results."
        ),
        (
            "Do not directly rank MobileGeo published "
            "MAT metrics against the project-defined "
            "common benchmark."
        ),
        (
            "Do not interpret image retrieval as "
            "verified geographic UAV pose."
        ),
        (
            "Accuracy differences do not isolate "
            "architecture because training regimes differ."
        ),
    ],

    "next_phase": (
        "PHASE_4_GEOREFERENCED_PROJECT_BENCHMARK"
    ),
}

COMPLETION_MANIFEST_JSON = (
    REPORT_ROOT
    / "phase3_completion_manifest.json"
)

COMPLETION_MANIFEST_JSON.write_text(
    json.dumps(
        completion_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

# Add manifest itself after creation.
add_file(
    COMPLETION_MANIFEST_JSON,
    "phase3_manifest",
)

# ------------------------------------------------------------
# 13. Save evidence-file checksum inventory
# ------------------------------------------------------------

checksum_records = []

# Rebuild unique list including manifest.
unique_paths = {}

for record in evidence_files:

    file_path = (
        Path(
            record["path"]
        )
    )

    unique_paths[
        str(
            file_path.resolve()
        )
    ] = {
        "path": file_path,
        "category": (
            record["category"]
        ),
    }

unique_paths[
    str(
        COMPLETION_MANIFEST_JSON.resolve()
    )
] = {
    "path": (
        COMPLETION_MANIFEST_JSON
    ),
    "category": (
        "phase3_manifest"
    ),
}

for record in sorted(
    unique_paths.values(),
    key=lambda item: (
        str(
            item["path"]
        )
    ),
):

    file_path = (
        record["path"]
    )

    checksum_records.append({
        "category": (
            record["category"]
        ),
        "source_path": str(
            file_path
        ),
        "size_bytes": int(
            file_path.stat().st_size
        ),
        "sha256": (
            sha256_file(
                file_path
            )
        ),
    })

CHECKSUM_INVENTORY_CSV = (
    REPORT_ROOT
    / "phase3_evidence_checksums.csv"
)

pd.DataFrame(
    checksum_records
).to_csv(
    CHECKSUM_INVENTORY_CSV,
    index=False,
)

# ------------------------------------------------------------
# 14. Create ZIP package
# ------------------------------------------------------------

ZIP_PATH = (
    PACKAGE_ROOT
    / "mobilegeo_phase3_common_benchmark.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

zip_members = set()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:

    # Add all evidence records.
    for record in checksum_records:

        source_path = Path(
            record[
                "source_path"
            ]
        )

        category = (
            record[
                "category"
            ]
        )

        # Preserve useful relative structure.
        try:
            relative_to_project = (
                source_path.relative_to(
                    PROJECT_ROOT
                )
            )

            archive_name = (
                Path("project_evidence")
                / relative_to_project
            )

        except ValueError:

            archive_name = (
                Path("external_evidence")
                / category
                / source_path.name
            )

        archive_name_string = str(
            archive_name
        )

        if (
            archive_name_string
            in zip_members
        ):
            continue

        archive.write(
            source_path,
            arcname=(
                archive_name_string
            ),
        )

        zip_members.add(
            archive_name_string
        )

    # Add checksum inventory.
    checksum_archive_name = (
        "project_evidence/"
        "results/common_benchmark/reports/"
        "phase3_evidence_checksums.csv"
    )

    if (
        checksum_archive_name
        not in zip_members
    ):
        archive.write(
            CHECKSUM_INVENTORY_CSV,
            arcname=(
                checksum_archive_name
            ),
        )

        zip_members.add(
            checksum_archive_name
        )

# ------------------------------------------------------------
# 15. Validate ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as archive:

    corrupt_member = (
        archive.testzip()
    )

    archived_names = (
        archive.namelist()
    )

if corrupt_member is not None:
    raise RuntimeError(
        "Corrupt ZIP member detected:\n"
        f"{corrupt_member}"
    )

if len(
    archived_names
) == 0:
    raise RuntimeError(
        "Evidence ZIP is empty."
    )

# ------------------------------------------------------------
# 16. ZIP checksum
# ------------------------------------------------------------

ZIP_SHA256 = sha256_file(
    ZIP_PATH
)

ZIP_CHECKSUM_PATH = (
    PACKAGE_ROOT
    / "mobilegeo_phase3_common_benchmark.zip.sha256"
)

ZIP_CHECKSUM_PATH.write_text(
    (
        f"{ZIP_SHA256}  "
        f"{ZIP_PATH.name}\n"
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 17. Final human-readable phase summary
# ------------------------------------------------------------

best_row = (
    final_metrics_df
    .sort_values(
        "recall_at_1_percent",
        ascending=False,
    )
    .iloc[0]
)

FINAL_PHASE_SUMMARY = (
    REPORT_ROOT
    / "phase3_final_summary.txt"
)

summary_text = f"""
PHASE 3 COMMON RETRIEVAL BENCHMARK — COMPLETE

Protocol:
PROJECT_DEFINED_LOCATION_DISJOINT_60_20_20

Queries:
8,000

Gallery images:
40

Test locations:
40

Direct common benchmark:

AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION
Recall@1: {local_r1:.2f}%

Sample4Geo
Recall@1: {sample_r1:.2f}%

UltraVPR
Recall@1: {ultra_r1:.2f}%

Best method on this project-defined benchmark:
{best_row['method']}
Recall@1: {float(best_row['recall_at_1_percent']):.2f}%

MobileGeo:
NOT INCLUDED IN DIRECT COMMON RANKING.
Published MobileGeo precomputed MAT features remain
a separate provenance result.

Important:
- The local model is not official MobileGeo.
- Sample4Geo and UltraVPR are zero-shot transfers.
- Training regimes differ.
- Retrieval accuracy is not verified UAV pose.

Next notebook:
04_georeferenced_project_benchmark.ipynb
""".strip()

FINAL_PHASE_SUMMARY.write_text(
    summary_text,
    encoding="utf-8",
)

# ------------------------------------------------------------
# 18. Final output
# ------------------------------------------------------------

print("\n" + "=" * 84)
print("✅ PHASE 3 COMMON RETRIEVAL BENCHMARK COMPLETE")
print("=" * 84)

print("\nFinal direct common benchmark:")

print(
    final_metrics_df[
        [
            "method",
            "descriptor_dimension",
            "recall_at_1_percent",
            "recall_at_5_percent",
            "recall_at_10_percent",
            "mean_average_precision_percent",
        ]
    ].to_string(
        index=False,
        formatters={
            "recall_at_1_percent": (
                lambda x: f"{x:.2f}"
            ),
            "recall_at_5_percent": (
                lambda x: f"{x:.2f}"
            ),
            "recall_at_10_percent": (
                lambda x: f"{x:.2f}"
            ),
            "mean_average_precision_percent": (
                lambda x: f"{x:.2f}"
            ),
        },
    )
)

print("\nBest method by Recall@1:")
print(
    best_row[
        "method"
    ]
)

print(
    f"{float(best_row['recall_at_1_percent']):.2f}%"
)

print("\nMobileGeo common-table status:")
print(
    "NOT DIRECTLY COMPARABLE — "
    "published precomputed-feature evaluation only"
)

print("\nCheckpoint records:")
for record in checkpoint_records:

    print(
        "-",
        record["method"],
    )

    print(
        "  SHA-256:",
        record["sha256"],
    )

print("\nEmbedding artifacts retained separately:")
print(
    len(
        embedding_records
    )
)

print("\nEvidence files packaged:")
print(
    len(
        archived_names
    )
)

print("\nCompletion manifest:")
print(
    COMPLETION_MANIFEST_JSON
)

print("\nChecksum inventory:")
print(
    CHECKSUM_INVENTORY_CSV
)

print("\nFinal summary:")
print(
    FINAL_PHASE_SUMMARY
)

print("\nEvidence ZIP:")
print(
    ZIP_PATH
)

print("\nZIP SHA-256:")
print(
    ZIP_SHA256
)

print("\nZIP checksum file:")
print(
    ZIP_CHECKSUM_PATH
)

print("\n" + "=" * 84)
print("✅ NOTEBOOK 03 COMPLETE")
print("=" * 84)

print("\nNext notebook:")
print(
    PROJECT_ROOT
    / "notebooks"
    / "04_georeferenced_project_benchmark.ipynb"
)


✅ PHASE 3 COMMON RETRIEVAL BENCHMARK COMPLETE

Final direct common benchmark:
                                   method  descriptor_dimension recall_at_1_percent recall_at_5_percent recall_at_10_percent mean_average_precision_percent
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION                  2048               94.58               99.99               100.00                          96.93
                               Sample4Geo                  1024               85.90               98.74                99.78                          91.79
                                 UltraVPR                   256               81.94               95.22                98.47                          87.83

Best method by Recall@1:
AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION
94.58%

MobileGeo common-table status:
NOT DIRECTLY COMPARABLE — published precomputed-feature evaluation only

Checkpoint records:
- AdvancedEdgeGeoLPN_LOCAL_REIMPLEMENTATION
  SHA-256: adc64177582638690b841c1be390b7d05dfb27f4d2

## Work cells

Add the phase-specific implementation below this cell.
